## 1. Dataset Loading and Inspection

In this section, we load the competition dataset and inspect the structure, columns, row counts, missing values, and basic characteristics of the three sources and ground truth.

In [10]:
import pandas as pd

base = r"E:\Amazon-ML-Challenge-2026\data\student_resource"

train_source1 = pd.read_csv(f"{base}\\dataset\\train\\train_source1.tsv", sep="\t")
train_source2 = pd.read_csv(f"{base}\\dataset\\train\\train_source2.tsv", sep="\t")
train_source3 = pd.read_csv(f"{base}\\dataset\\train\\train_source3.tsv", sep="\t")
train_ground_truth = pd.read_csv(f"{base}\\dataset\\train\\train_ground_truth.tsv", sep="\t")

test_source1 = pd.read_csv(f"{base}\\dataset\\test\\test_source1.tsv", sep="\t")
test_source2 = pd.read_csv(f"{base}\\dataset\\test\\test_source2.tsv", sep="\t")
test_source3 = pd.read_csv(f"{base}\\dataset\\test\\test_source3.tsv", sep="\t")

print("Loaded OK")

Loaded OK


In [11]:
for name, df in [
    ("train_source1", train_source1),
    ("train_source2", train_source2),
    ("train_source3", train_source3),
    ("train_ground_truth", train_ground_truth),
    ("test_source1", test_source1),
    ("test_source2", test_source2),
    ("test_source3", test_source3),
]:
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("missing values:\n", df.isna().sum())


=== train_source1 ===
shape: (2206821, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

=== train_source2 ===
shape: (5034616, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

=== train_source3 ===
shape: (5285603, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

=== train_ground_truth ===
shape: (2206821, 2)
columns: ['source1_entity_id', 'matched_entity_ids']
missing values:
 source1_entity_id          0
matched_entity_ids    123247
dtype: int64

=== test_source1 ===
shape: (1732544, 4)
columns: ['entity_id', 'business_

## 2. Ground Truth and Dataset Distribution

Before designing the matching system, we inspect the distribution of
matches and countries in the training data.

In [12]:
train_source1.head(5)

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [13]:
train_source2.head(5)

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


In [14]:
train_ground_truth.head(10)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [15]:
# Confirm the missing ground truth rows really are "no match" (empty string vs NaN)
print(train_ground_truth['matched_entity_ids'].isna().sum())
print((train_ground_truth['matched_entity_ids'] == '').sum())

# How many matches does a typical Source 1 entity have? (only for non-missing rows)
match_counts = train_ground_truth['matched_entity_ids'].dropna().apply(lambda x: len(x.split(',')))
print(match_counts.describe())

# Country distribution across sources
print(train_source1['country'].value_counts())
print(train_source2['country'].value_counts())

123247
0
count    2.083574e+06
mean     3.665992e+00
std      1.526294e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: matched_entity_ids, dtype: float64
country
US       1323633
India     883188
Name: count, dtype: int64
country
US       3016817
India    2017799
Name: count, dtype: int64


In [16]:
import re

def has_devanagari(text):
    if pd.isna(text):
        return False
    return bool(re.search(r'[\u0900-\u097F]', text))

for name, df in [
    ("train_source1", train_source1),
    ("train_source2", train_source2),
    ("train_source3", train_source3)
]:
    n = df['business_name'].apply(has_devanagari).sum()
    print(f"{name}: {n} / {len(df)} names contain Devanagari script ({n/len(df)*100:.2f}%)")

train_source1: 0 / 2206821 names contain Devanagari script (0.00%)
train_source2: 269424 / 5034616 names contain Devanagari script (5.35%)
train_source3: 158003 / 5285603 names contain Devanagari script (2.99%)


## 3. Train-Validation Split

The training data contains Source 1 records together with their ground-truth
matches from Source 2 and Source 3.

To evaluate our matching approach before applying it to the unseen test data,
we split the Source 1 entities into two groups:

- **80% training/development set** — used to develop the baseline approach.
- **20% validation set** — used to evaluate the approach on unseen Source 1 entities.

The split is random and uses a fixed random seed (`random_state=42`) so that
the same validation set can be reproduced across experiments.

Source 2 and Source 3 are not split at this stage because they serve as the
candidate databases against which Source 1 entities are matched.

In [17]:
from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    train_source1['entity_id'],
    test_size=0.20,
    random_state=42
)

print("Training Source 1 entities:", len(train_ids))
print("Validation Source 1 entities:", len(val_ids))

# Sanity check: no overlap, and counts add up
assert len(set(train_ids) & set(val_ids)) == 0
assert len(train_ids) + len(val_ids) == len(train_source1)
print("Split check passed.")

Training Source 1 entities: 1765456
Validation Source 1 entities: 441365
Split check passed.


In [18]:
train_source1_split = train_source1[
    train_source1['entity_id'].isin(train_ids)
].copy().reset_index(drop=True)

val_source1 = train_source1[
    train_source1['entity_id'].isin(val_ids)
].copy().reset_index(drop=True)

train_ground_truth_split = train_ground_truth[
    train_ground_truth['source1_entity_id'].isin(train_ids)
].copy().reset_index(drop=True)

val_ground_truth = train_ground_truth[
    train_ground_truth['source1_entity_id'].isin(val_ids)
].copy().reset_index(drop=True)

print("Train S1:", train_source1_split.shape)
print("Validation S1:", val_source1.shape)
print("Train ground truth:", train_ground_truth_split.shape)
print("Validation ground truth:", val_ground_truth.shape)

Train S1: (1765456, 4)
Validation S1: (441365, 4)
Train ground truth: (1765456, 2)
Validation ground truth: (441365, 2)


## 4. Baseline Matching Approach

We now develop a simple entity-resolution baseline using the training data
and evaluate its predictions on the held-out validation set.

The baseline will first focus on high-confidence matching rules and
candidate generation before introducing more complex matching methods.

### 4.1 Validation Evaluation

The competition evaluates predictions using macro-averaged F₀.₅ across
Source 1 entities. Since precision is weighted more heavily than recall,
the evaluation function measures the overlap between the predicted and
ground-truth match sets for each Source 1 entity.

This function will be used to evaluate our baseline on the held-out
validation set before generating test predictions.

In [19]:
def parse_ids(cell):
    """Convert a comma-separated ID string (or NaN) into a set of IDs."""
    if pd.isna(cell) or cell == '':
        return set()
    return set(cell.split(','))


def f_beta_score(true_set, pred_set, beta=0.5):
    """Compute F_beta for one entity given its true and predicted match sets."""
    # Singleton case: no true matches
    if len(true_set) == 0:
        return 1.0 if len(pred_set) == 0 else 0.0

    # We have true matches but predicted nothing
    if len(pred_set) == 0:
        return 0.0

    tp = len(true_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(true_set)

    if precision == 0 and recall == 0:
        return 0.0

    beta_sq = beta ** 2
    return (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)


def evaluate_predictions(ground_truth_df, predictions_df):
    """
    ground_truth_df: columns [source1_entity_id, matched_entity_ids]
    predictions_df:  columns [source1_entity_id, matched_entity_ids]
    Returns macro-averaged F_0.5 across all entities in ground_truth_df.
    """
    gt = ground_truth_df.set_index('source1_entity_id')['matched_entity_ids']
    pred = predictions_df.set_index('source1_entity_id')['matched_entity_ids']

    scores = []
    for entity_id in gt.index:
        true_set = parse_ids(gt.loc[entity_id])
        # If we didn't predict anything for this entity at all, treat as empty
        pred_cell = pred.loc[entity_id] if entity_id in pred.index else None
        pred_set = parse_ids(pred_cell)
        scores.append(f_beta_score(true_set, pred_set))

    return sum(scores) / len(scores)

In [20]:
import pandas as pd

test_gt = pd.DataFrame({
    'source1_entity_id': ['S1-00001'],
    'matched_entity_ids': ['S2-00047,S3-00812']
})
test_pred = pd.DataFrame({
    'source1_entity_id': ['S1-00001'],
    'matched_entity_ids': ['S2-00047,S2-00193,S3-00812']
})

score = evaluate_predictions(test_gt, test_pred)
print(score)  # PDF says this should be 0.714

0.7142857142857142


## Evaluation Function — Verification

Before using the evaluation function on our real validation data, we tested it
with a small manually constructed example.

The ground truth contained two correct matches, while the prediction contained
those two matches plus one incorrect match. The resulting F_0.5 score was
approximately 0.7143, confirming that the evaluation function penalizes the
extra false match while giving credit for the two correct matches.

This is only a test of the evaluation code. No model has been trained and no
real validation score has been calculated yet.

### 4.2 Name Normalization for Blocking

Before generating candidate pairs, we first normalize business names so that
records representing the same business are less likely to be separated by
simple formatting differences.

The baseline normalization performs four operations:

- Converts the business name to lowercase.
- Replaces punctuation with spaces.
- Removes selected common legal suffixes such as `Inc`, `Ltd`, `LLC`, and
  `Corporation`.
- Collapses whitespace and reconstructs the normalized name.

This normalized representation will later be used to construct a blocking key.
The purpose of this first experiment is to establish a simple baseline before
testing more sophisticated blocking and matching strategies.

In [21]:
import re

def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    
    # Strip punctuation (& . , - etc.), keep letters/digits/spaces
    name = re.sub(r'[^\w\s]', ' ', name)
    
    # Remove common legal-suffix noise words
    stopwords = {'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
                 'corp', 'corporation', 'llc', 'llp', 'co', 'company'}
    tokens = [t for t in name.split() if t not in stopwords]
    
    # Collapse whitespace and return
    return ' '.join(tokens).strip()

In [22]:
sample_names = (
    train_source1['business_name'].head(10).tolist()
    + train_source2['business_name'].head(10).tolist()
    + train_source3['business_name'].head(10).tolist()
)

for n in sample_names:
    print(repr(n), "->", repr(normalize_name(n)))

"Orelee's Barbershop" -> 'orelee s barbershop'
'Prime Money' -> 'prime money'
'B+ Retail Inc' -> 'b retail'
'Christ Chapel' -> 'christ chapel'
'Prabhav Business Center' -> 'prabhav business center'
'Custom Wealth Services LLC' -> 'custom wealth services'
'Consulting Nyasa Nursing Private Limited' -> 'consulting nyasa nursing'
'Nexus Anchor Rain' -> 'nexus anchor rain'
'Moore Bitwise Inc' -> 'moore bitwise'
'Dermatology Green Medicine' -> 'dermatology green medicine'
'राम मार्केटिंग प्राइवेट लिमिटेड' -> 'र म म र क ट ग प र इव ट ल म ट ड'
'-- Holloway Peak Inc Seafood' -> 'holloway peak seafood'
'आदित्य प्रॉपर्टीज एलएलपी' -> 'आद त य प र पर ट ज एलएलप'
'Summit Inc' -> 'summit'
'Delta Tetlecommunication Inc' -> 'delta tetlecommunication'
'Lee and Lawson' -> 'lee and lawson'
'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.shivshakti.com' -> 'shivshakti vidyalaya vidyalaya overseas www shivshakti com'
'Shree Infracon Private Ltd' -> 'shree infracon'
'Chavira Platinum Chimera LLC' -> 

### Normalization Sanity Check — Baseline

The normalization function was tested on a small sample of real business names
from all three sources.

For Latin-script names, the baseline behaves as expected in many cases. It
removes capitalization differences, punctuation, and common legal suffixes
while preserving the main business name.

However, the sample also revealed an important issue with non-Latin text.
Devanagari business names are being corrupted during punctuation removal. For
example, a Hindi name such as `राम मार्केटिंग प्राइवेट लिमिटेड` is transformed
into separated characters rather than preserving the original words.

This happens because the current regular expression does not correctly preserve
the Unicode combining characters used in Devanagari text.

Therefore, this version of the normalization function is not suitable as the
final baseline. We will create a new normalization experiment that handles
Unicode text more safely while preserving the useful normalization steps
identified here.

### 4.3 Experiment — Unicode-Safe Name Normalization

The previous normalization experiment showed that punctuation and legal-suffix
normalization works for many Latin-script names but breaks Devanagari text:
the `\w` regex character class does not recognize Devanagari vowel marks
(matras) as word characters, so they were stripped and words were shredded
into individual base letters.

In this experiment, we modify only the punctuation-handling step. Instead of
matching on `\w`, we remove characters using their Unicode general category
(punctuation categories only), which preserves letters and combining marks
across scripts while still stripping punctuation and normalizing whitespace.

The revised function is tested on the same sample as before so the two
normalization approaches can be compared directly.

In [23]:
import unicodedata

def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    
    # Remove only true punctuation characters (Unicode category starting with 'P'),
    # keep letters, combining marks (needed for Devanagari), digits, and spaces
    name = ''.join(' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch for ch in name)
    
    stopwords = {'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
                 'corp', 'corporation', 'llc', 'llp', 'co', 'company'}
    tokens = [t for t in name.split() if t not in stopwords]
    
    return ' '.join(tokens).strip()

In [24]:
sample_names = (
    train_source1['business_name'].head(10).tolist()
    + train_source2['business_name'].head(10).tolist()
    + train_source3['business_name'].head(10).tolist()
)

for n in sample_names:
    print(repr(n), "->", repr(normalize_name(n)))

"Orelee's Barbershop" -> 'orelee s barbershop'
'Prime Money' -> 'prime money'
'B+ Retail Inc' -> 'b retail'
'Christ Chapel' -> 'christ chapel'
'Prabhav Business Center' -> 'prabhav business center'
'Custom Wealth Services LLC' -> 'custom wealth services'
'Consulting Nyasa Nursing Private Limited' -> 'consulting nyasa nursing'
'Nexus Anchor Rain' -> 'nexus anchor rain'
'Moore Bitwise Inc' -> 'moore bitwise'
'Dermatology Green Medicine' -> 'dermatology green medicine'
'राम मार्केटिंग प्राइवेट लिमिटेड' -> 'राम मार्केटिंग प्राइवेट लिमिटेड'
'-- Holloway Peak Inc Seafood' -> 'holloway peak seafood'
'आदित्य प्रॉपर्टीज एलएलपी' -> 'आदित्य प्रॉपर्टीज एलएलपी'
'Summit Inc' -> 'summit'
'Delta Tetlecommunication Inc' -> 'delta tetlecommunication'
'Lee and Lawson' -> 'lee and lawson'
'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.shivshakti.com' -> 'shivshakti vidyalaya vidyalaya overseas www shivshakti com'
'Shree Infracon Private Ltd' -> 'shree infracon'
'Chavira Platinum Chimera LLC' -

### 4.4 Blocking Key and Candidate Generation

After normalizing business names, we construct a simple blocking key to reduce
the number of Source 2 and Source 3 records that must be considered for each
Source 1 entity.

For this first blocking experiment, the key consists of the country and the
first token of the normalized business name. Using the entire normalized name
would be more restrictive and could discard true matches when names contain
typos or other variations. Using only the first token produces larger but more
forgiving candidate groups, while the country helps limit unrelated records.

We build lookup indexes for Source 2 and Source 3 so that candidates can be
retrieved efficiently without comparing every Source 1 entity against every
record in the other sources.

The blocking stage is evaluated on the held-out validation set using:
- candidate recall: the fraction of known true matches that survive blocking;
- average candidates per Source 1 entity: the size of the resulting candidate
  sets.

Candidate recall represents the recall ceiling of the downstream matching
stage: a true match that is removed during blocking cannot be recovered later.

In [25]:
def first_token(normalized_name):
    tokens = normalized_name.split()
    return tokens[0] if tokens else ""

def build_block_key(df):
    """Adds norm_name and block_key columns to a dataframe with business_name/country."""
    df = df.copy()
    df['norm_name'] = df['business_name'].apply(normalize_name)
    df['block_key'] = df['country'] + '||' + df['norm_name'].apply(first_token)
    return df

In [26]:
train_source2_keyed = build_block_key(train_source2)
train_source3_keyed = build_block_key(train_source3)
val_source1_keyed = build_block_key(val_source1)

print("Done building keys.")

Done building keys.


In [27]:
def build_index(df_keyed):
    """Returns dict: block_key -> list of entity_ids"""
    return df_keyed.groupby('block_key')['entity_id'].apply(list).to_dict()

index_source2 = build_index(train_source2_keyed)
index_source3 = build_index(train_source3_keyed)

print("Number of distinct keys in Source 2 index:", len(index_source2))
print("Number of distinct keys in Source 3 index:", len(index_source3))

Number of distinct keys in Source 2 index: 363912
Number of distinct keys in Source 3 index: 412262


In [28]:
def get_candidates(block_key):
    c2 = index_source2.get(block_key, [])
    c3 = index_source3.get(block_key, [])
    return set(c2) | set(c3)

val_gt_lookup = val_ground_truth.set_index('source1_entity_id')['matched_entity_ids']

recalls = []
candidate_counts = []

for _, row in val_source1_keyed.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts.append(len(candidate_set))
    
    if len(true_set) == 0:
        # No true matches — recall undefined here, skip from recall average
        continue
    
    found = len(true_set & candidate_set)
    recalls.append(found / len(true_set))

print("Average candidates per entity:", sum(candidate_counts) / len(candidate_counts))
print("Candidate recall (macro-average, over entities with true matches):", sum(recalls) / len(recalls))
print("Entities with true matches evaluated:", len(recalls))

Average candidates per entity: 5680.185884698605
Candidate recall (macro-average, over entities with true matches): 0.7563608887127291
Entities with true matches evaluated: 416780


### Baseline Blocking Results

The first blocking experiment used the country and the first token of the
normalized business name as the blocking key.

On the 20% validation split, the Source 2 index contained 363,912 distinct
blocking keys and the Source 3 index contained 412,262 distinct keys.

The resulting candidate sets contained an average of 5,680.19 candidates per
Source 1 entity. Candidate recall was 75.64% across validation entities with at
least one known match.

This means that approximately 24% of true matches were excluded during
blocking and therefore could not be recovered by any downstream matching
model.

The baseline demonstrates that simple blocking can reduce the search space,
but the candidate sets are still large and the recall ceiling is not high
enough for the final pipeline. The next experiment will therefore investigate
a blocking strategy that improves recall without producing unnecessarily large
candidate sets.

In [29]:
# 1. Are a few keys responsible for most of the bloat? (bucket size distribution)
import numpy as np

bucket_sizes_s2 = [len(v) for v in index_source2.values()]
bucket_sizes_s3 = [len(v) for v in index_source3.values()]

print("Source 2 bucket sizes — mean:", np.mean(bucket_sizes_s2), "median:", np.median(bucket_sizes_s2), "max:", np.max(bucket_sizes_s2))
print("Source 3 bucket sizes — mean:", np.mean(bucket_sizes_s3), "median:", np.median(bucket_sizes_s3), "max:", np.max(bucket_sizes_s3))

Source 2 bucket sizes — mean: 13.83470729187276 median: 1.0 max: 40489
Source 3 bucket sizes — mean: 12.820980347448952 median: 1.0 max: 50862


In [30]:
# Inspect the largest blocking buckets

largest_s2 = sorted(
    index_source2.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]

largest_s3 = sorted(
    index_source3.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]

print("Largest Source 2 buckets:")
for key, ids in largest_s2:
    print(key, "->", len(ids))

print("\nLargest Source 3 buckets:")
for key, ids in largest_s3:
    print(key, "->", len(ids))

Largest Source 2 buckets:
US||the -> 40489
US||pediatric -> 30497
India||m -> 25796
India||shri -> 25702
India||dr -> 25673
India||sri -> 25658
India||smt -> 25474
India||mr -> 25449
India||new -> 14299
US||family -> 13231
US||a -> 11974
India||india -> 10655
US||eye -> 10532
US||ear -> 10393
India||mumbai -> 10384
US||blue -> 10379
US||physical -> 10338
US||dental -> 10315
US||foot -> 10272
US||primary -> 10221

Largest Source 3 buckets:
US||the -> 50862
US||pediatric -> 30944
India||m -> 29422
India||mr -> 29388
India||sri -> 29359
India||smt -> 29274
India||shri -> 29199
India||dr -> 29189
India||new -> 15783
US||family -> 13187
US||a -> 11972
India||india -> 10909
India||shree -> 10679
US||ear -> 10652
US||primary -> 10522
US||dental -> 10514
US||eye -> 10463
US||physical -> 10371
US||foot -> 10299
US||blue -> 10235


### Blocking Bucket Analysis

The first-token blocking strategy produced highly uneven bucket sizes. Although
the median bucket size was only 1 for both Source 2 and Source 3, a small number
of blocking keys produced extremely large buckets.

The largest buckets were associated with common or generic first tokens such as
`the`, `m`, `mr`, `dr`, `smt`, `sri`, and `shri`. For example, the largest Source
2 bucket contained 40,489 records for the key `US||the`, while the largest
Source 3 bucket contained 50,862 records for the same key.

This shows that the high average candidate count is driven disproportionately
by a relatively small number of very large buckets. It also suggests that
using the first token alone does not provide sufficiently selective blocking
for records beginning with common prefixes or generic terms.

We will next examine the true matches that were missed by this blocking
strategy before deciding how to modify the blocking key.

In [31]:
# 2. For entities where we LOST recall (missed a true match), was the true match's first word actually different?
# Sample a handful of failure cases to eyeball
sample_checked = 0
for _, row in val_source1_keyed.iterrows():
    entity_id = row['entity_id']
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    if len(true_set) == 0:
        continue
    candidate_set = get_candidates(row['block_key'])
    missed = true_set - candidate_set
    if missed:
        print("Source1:", row['business_name'], "| norm:", row['norm_name'])
        # look up one missed match's actual name
        missed_id = next(iter(missed))
        if missed_id.startswith('S2-'):
            match_row = train_source2[train_source2['entity_id'] == missed_id]
        else:
            match_row = train_source3[train_source3['entity_id'] == missed_id]
        print("  Missed match:", match_row['business_name'].values)
        sample_checked += 1
        if sample_checked >= 8:
            break

Source1: Prime Money | norm: prime money
  Missed match: ['primemoney.com']
Source1: Dream Construction Limited | norm: dream construction
  Missed match: ['డ్రీమ్ కన్\u200cస్ట్రక్షన్ లిమిటెడ్']
Source1: Rajani Enterprises of Kannur | norm: rajani enterprises of kannur
  Missed match: ['M/s Rajani Enterprises of']
Source1: Schaefer, Michael and Silver Associates | norm: schaefer michael and silver associates
  Missed match: ['Silver Michael and Schaefer, Ass0ciates']
Source1: Aggie E. Nagle, L.C.S.W. | norm: aggie e nagle l c s w
  Missed match: ['aggieenagle.com']
Source1: Pacific Learning Laboratories LLC | norm: pacific learning laboratories
  Missed match: ['... The Pacific Learning Laboratories [LLC] [LLC]']
Source1: Mayur Hospital of Gondal | norm: mayur hospital of gondal
  Missed match: ['M/s Mayur Hospital of Gondal Corporation']
Source1: Orchid Renewable Ltd | norm: orchid renewable
  Missed match: ['Smt BRIXUMBRABELO']


### Baseline Failure Analysis

The missed-match examples show that the first-token blocking strategy fails for
several different reasons. Some records contain leading noise prefixes such as
`M/s`, while others use alternative representations such as domain names or
different scripts. These cases indicate that improving blocking will require
more than simply changing the first-token rule.

We first address the clearly observed prefix and honorific noise because these
tokens also contributed to the largest blocking buckets. Cross-script and
other name-variation cases will be investigated separately in subsequent
experiments.

### 4.5 Experiment — Noise-Prefix Normalization

In [32]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    name = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in name
    )
    
    stopwords = {
        'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
        'corp', 'corporation', 'llc', 'llp', 'co', 'company',
        'dr', 'mr', 'mrs', 'ms', 'smt', 'sri', 'shri', 'prof',
        # leading noise prefixes seen in real data
        'm', 's',  # "m/s" becomes "m s" after punctuation stripping
    }
    tokens = [t for t in name.split() if t not in stopwords]
    
    return ' '.join(tokens).strip()

In [33]:
train_source2_keyed_v2 = build_block_key(train_source2)
train_source3_keyed_v2 = build_block_key(train_source3)
val_source1_keyed_v2 = build_block_key(val_source1)

print("Done building v2 keys.")

Done building v2 keys.


In [34]:
index_source2_v2 = build_index(train_source2_keyed_v2)
index_source3_v2 = build_index(train_source3_keyed_v2)

print("Number of distinct keys in Source 2 index (v2):", len(index_source2_v2))
print("Number of distinct keys in Source 3 index (v2):", len(index_source3_v2))

Number of distinct keys in Source 2 index (v2): 372168
Number of distinct keys in Source 3 index (v2): 420329


In [35]:
bucket_sizes_s2_v2 = [len(v) for v in index_source2_v2.values()]
bucket_sizes_s3_v2 = [len(v) for v in index_source3_v2.values()]

print("Source 2 (v2) bucket sizes — mean:", np.mean(bucket_sizes_s2_v2), "median:", np.median(bucket_sizes_s2_v2), "max:", np.max(bucket_sizes_s2_v2))
print("Source 3 (v2) bucket sizes — mean:", np.mean(bucket_sizes_s3_v2), "median:", np.median(bucket_sizes_s3_v2), "max:", np.max(bucket_sizes_s3_v2))

Source 2 (v2) bucket sizes — mean: 13.527804647363556 median: 1.0 max: 40489
Source 3 (v2) bucket sizes — mean: 12.574918694641578 median: 1.0 max: 50863


In [36]:
def get_candidates_v2(block_key):
    c2 = index_source2_v2.get(block_key, [])
    c3 = index_source3_v2.get(block_key, [])
    return set(c2) | set(c3)

recalls_v2 = []
candidate_counts_v2 = []

for _, row in val_source1_keyed_v2.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates_v2(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts_v2.append(len(candidate_set))
    
    if len(true_set) == 0:
        continue
    
    found = len(true_set & candidate_set)
    recalls_v2.append(found / len(true_set))

print("Average candidates per entity (v2):", sum(candidate_counts_v2) / len(candidate_counts_v2))
print("Candidate recall (v2):", sum(recalls_v2) / len(recalls_v2))
print("Entities with true matches evaluated:", len(recalls_v2))

Average candidates per entity (v2): 5846.941461148936
Candidate recall (v2): 0.784331800797802
Entities with true matches evaluated: 416780


## Experiment 4.5 Results — Noise-Prefix Normalization

The second normalization experiment removed common honorifics and leading noise prefixes observed in the training data, including `dr`, `mr`, `mrs`, `ms`, `smt`, `sri`, `shri`, `prof`, and the `m/s` representation after punctuation normalization.

This modification increased candidate recall from 75.64% to 78.43%, an improvement of approximately 2.80 percentage points.

The average candidate count increased slightly from 5,680.19 to 5,846.94 candidates per Source 1 entity. The maximum bucket sizes remained almost unchanged: the largest Source 2 bucket contained 40,489 records in both versions, while the largest Source 3 bucket changed from 50,862 to 50,863 records.

This indicates that removing noise prefixes improved the ability to retrieve true matches, but did not substantially reduce the largest blocking buckets. The increase in average candidate count suggests that normalization caused some records to be grouped into the same blocking keys.

Overall, this normalization change improved candidate recall and will therefore be retained for subsequent experiments.

## 4.6 Experiment — Generic Token Normalization

The previous experiment removed common honorifics and noise prefixes such as `dr`, `mr`, `smt`, `sri`, `shri`, and `m/s`. However, the blocking analysis also showed that the token `the` produced very large buckets, particularly in Source 2 and Source 3.

In this experiment, `the` was added to the normalization stopword list. The goal was to determine whether removing this common generic token could improve blocking selectivity without losing true matches.

In [37]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    name = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in name
    )
    
    stopwords = {
        'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
        'corp', 'corporation', 'llc', 'llp', 'co', 'company',
        'dr', 'mr', 'mrs', 'ms', 'smt', 'sri', 'shri', 'prof',
        'm', 's',
        'the'
    }
    tokens = [t for t in name.split() if t not in stopwords]
    
    return ' '.join(tokens).strip()

In [38]:
train_source2_keyed_v3 = build_block_key(train_source2)
train_source3_keyed_v3 = build_block_key(train_source3)
val_source1_keyed_v3 = build_block_key(val_source1)

In [39]:
index_source2_v3 = build_index(train_source2_keyed_v3)
index_source3_v3 = build_index(train_source3_keyed_v3)

print("Number of distinct keys in Source 2 index (v3):", len(index_source2_v3))
print("Number of distinct keys in Source 3 index (v3):", len(index_source3_v3))

Number of distinct keys in Source 2 index (v3): 372673
Number of distinct keys in Source 3 index (v3): 420923


In [40]:
bucket_sizes_s2_v3 = [len(v) for v in index_source2_v3.values()]
bucket_sizes_s3_v3 = [len(v) for v in index_source3_v3.values()]

print("Source 2 (v3) bucket sizes — mean:", np.mean(bucket_sizes_s2_v3), "median:", np.median(bucket_sizes_s2_v3), "max:", np.max(bucket_sizes_s2_v3))
print("Source 3 (v3) bucket sizes — mean:", np.mean(bucket_sizes_s3_v3), "median:", np.median(bucket_sizes_s3_v3), "max:", np.max(bucket_sizes_s3_v3))

Source 2 (v3) bucket sizes — mean: 13.509473452597854 median: 1.0 max: 30875
Source 3 (v3) bucket sizes — mean: 12.557173164688079 median: 1.0 max: 31445


In [41]:
def get_candidates_v3(block_key):
    c2 = index_source2_v3.get(block_key, [])
    c3 = index_source3_v3.get(block_key, [])
    return set(c2) | set(c3)

recalls_v3 = []
candidate_counts_v3 = []

for _, row in val_source1_keyed_v3.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates_v3(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts_v3.append(len(candidate_set))
    
    if len(true_set) == 0:
        continue
    
    found = len(true_set & candidate_set)
    recalls_v3.append(found / len(true_set))

print("Average candidates per entity (v3):", sum(candidate_counts_v3) / len(candidate_counts_v3))
print("Candidate recall (v3):", sum(recalls_v3) / len(recalls_v3))
print("Entities with true matches evaluated:", len(recalls_v3))

Average candidates per entity (v3): 5830.759423606312
Candidate recall (v3): 0.7926473105808727
Entities with true matches evaluated: 416780


## Experiment 4.6 Results — Generic Token Normalization

Removing the generic token `the` improved candidate recall from 78.43% to 79.26%, an improvement of approximately 0.83 percentage points.

At the same time, the average number of candidates per Source 1 entity decreased slightly from 5,846.94 to 5,830.76. This means the modification improved recall while also slightly reducing the overall candidate volume.

The maximum blocking bucket sizes were also substantially smaller than the original baseline, with the largest Source 2 bucket containing 30,875 records and the largest Source 3 bucket containing 31,445 records.

Overall, removing `the` produced a small but positive improvement in both candidate recall and candidate efficiency. This version will therefore be retained for the matching stage.

## 5. Matching Stage — Computational Feasibility Test

Before running similarity scoring over the full validation set, we first measure the actual runtime on a small sample of 50 validation entities.

The blocking stage has already produced the candidate pairs. The next step is to calculate name and address similarity for each candidate pair. Because the dataset contains millions of records, this timing test helps estimate the computational cost of scoring the full validation set before committing to a long-running computation.

This is a feasibility test rather than a leaderboard experiment. The results will be used to decide how the matching stage should be implemented efficiently.

In [42]:
from rapidfuzz import fuzz

def pair_similarity(name1, addr1, name2, addr2):
    """Returns a combined similarity score (0-100) for one candidate pair."""
    name_sim = fuzz.token_sort_ratio(str(name1), str(name2)) if pd.notna(name1) and pd.notna(name2) else 0
    
    if pd.isna(addr1) or pd.isna(addr2):
        addr_sim = 0
    else:
        addr_sim = fuzz.token_sort_ratio(str(addr1), str(addr2))
    
    return 0.6 * name_sim + 0.4 * addr_sim

In [43]:
source2_lookup = train_source2_keyed_v3.set_index('entity_id')[['business_name', 'business_address']].to_dict('index')
source3_lookup = train_source3_keyed_v3.set_index('entity_id')[['business_name', 'business_address']].to_dict('index')

print("Lookups built.")

Lookups built.


In [44]:
import time

sample_val = val_source1_keyed_v3.head(50)

start = time.time()

sample_results = []
for _, row in sample_val.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    name1 = row['business_name']
    addr1 = row['business_address']
    
    candidate_ids = list(get_candidates_v3(block_key))
    
    for cand_id in candidate_ids:
        lookup = source2_lookup if cand_id.startswith('S2-') else source3_lookup
        cand_data = lookup.get(cand_id)
        if cand_data is None:
            continue
        
        name2 = cand_data['business_name']
        addr2 = cand_data['business_address']
        
        score = pair_similarity(name1, addr1, name2, addr2)
        sample_results.append((entity_id, cand_id, score))

elapsed = time.time() - start
print(f"Time for 50 entities: {elapsed:.2f} seconds")
print(f"Estimated total pairs scored: {len(sample_results)}")
print(f"Estimated time for all {len(val_source1_keyed_v3)} entities: {elapsed / 50 * len(val_source1_keyed_v3) / 60:.1f} minutes")

Time for 50 entities: 3.54 seconds
Estimated total pairs scored: 337398
Estimated time for all 441365 entities: 520.7 minutes


### 5.3 Batched RapidFuzz Scoring — Timing Test

The previous implementation improved candidate lookup by using dictionary-based lookups, but scoring all candidate pairs individually was still estimated to require several hours.

This experiment tests batched RapidFuzz scoring using `process.cdist()`. Instead of calling the similarity function separately for every candidate pair, candidate names and addresses are collected into arrays and scored in batches.

The purpose is to determine whether batched similarity computation can substantially reduce the runtime while producing the same type of similarity scores.

In [45]:
from rapidfuzz import process, fuzz
import numpy as np

def score_entity_candidates(name1, addr1, candidate_ids):
    """Score one Source 1 entity against all its candidates, batched."""
    cand_names = []
    cand_addrs = []
    valid_ids = []
    
    for cand_id in candidate_ids:
        lookup = source2_lookup if cand_id.startswith('S2-') else source3_lookup
        cand_data = lookup.get(cand_id)
        if cand_data is None:
            continue
        valid_ids.append(cand_id)
        cand_names.append(str(cand_data['business_name']) if pd.notna(cand_data['business_name']) else "")
        cand_addrs.append(str(cand_data['business_address']) if pd.notna(cand_data['business_address']) else "")
    
    if not valid_ids:
        return []
    
    name1_str = str(name1) if pd.notna(name1) else ""
    addr1_str = str(addr1) if pd.notna(addr1) else ""
    
    name_scores = process.cdist([name1_str], cand_names, scorer=fuzz.token_sort_ratio)[0]
    addr_scores = process.cdist([addr1_str], cand_addrs, scorer=fuzz.token_sort_ratio)[0]
    
    combined = 0.6 * name_scores + 0.4 * addr_scores
    return list(zip(valid_ids, combined))

In [46]:
import time

sample_val = val_source1_keyed_v3.head(50)

start = time.time()

sample_results = []
for _, row in sample_val.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    name1 = row['business_name']
    addr1 = row['business_address']
    
    candidate_ids = list(get_candidates_v3(block_key))
    scored = score_entity_candidates(name1, addr1, candidate_ids)
    
    for cand_id, score in scored:
        sample_results.append((entity_id, cand_id, score))

elapsed = time.time() - start
print(f"Time for 50 entities: {elapsed:.2f} seconds")
print(f"Estimated total pairs scored: {len(sample_results)}")
print(f"Estimated time for all {len(val_source1_keyed_v3)} entities: {elapsed / 50 * len(val_source1_keyed_v3) / 60:.1f} minutes")

Time for 50 entities: 2.64 seconds
Estimated total pairs scored: 337398
Estimated time for all 441365 entities: 388.9 minutes


### 5.4 Token-Overlap Pre-filter

The previous timing experiments showed that even with optimized dictionary lookups and batched RapidFuzz scoring, scoring every candidate produced by V3 blocking would take several hours.

To reduce the number of pairs reaching the more expensive fuzzy-matching stage, this experiment introduces a cheap token-overlap pre-filter.

For each Source 1 entity, the normalized business name is compared with the normalized names of its V3 candidates using Jaccard similarity. Candidates are ranked by their token overlap, and only the top K candidates are retained for subsequent RapidFuzz scoring.

The purpose of this experiment is to measure whether this inexpensive filtering step can substantially reduce the candidate set while retaining most of the true matches already recovered by V3 blocking.

Before applying the filter to the full validation set, its recall will be measured on a smaller validation sample.

#### Step 1 — Define the Jaccard Pre-filter

The pre-filter calculates Jaccard similarity between the sets of normalized name tokens of the Source 1 entity and each candidate. Candidates with the highest token overlap are retained.

In [47]:
def jaccard_prefilter(name1, candidate_ids, keyed_lookup_dict, top_k=50):
    """
    name1: normalized Source 1 name (string)
    candidate_ids: list of candidate entity_ids
    keyed_lookup_dict: entity_id -> {'norm_name': ..., 'business_address': ..., 'business_name': ...}
    Returns top_k candidate_ids ranked by Jaccard token overlap.
    """
    tokens1 = set(name1.split())
    
    scored = []
    for cand_id in candidate_ids:
        cand_data = keyed_lookup_dict.get(cand_id)
        if cand_data is None:
            continue
        tokens2 = set(cand_data['norm_name'].split())
        
        union_size = len(tokens1 | tokens2)
        if union_size == 0:
            jaccard = 0
        else:
            jaccard = len(tokens1 & tokens2) / union_size
        
        scored.append((cand_id, jaccard))
    
    scored.sort(key=lambda x: x[1], reverse=True)
    return [cand_id for cand_id, _ in scored[:top_k]]

#### Step 2 — Build Candidate Lookups

The existing candidate lookups contain the original business name and address. For the Jaccard pre-filter, the normalized name is also required, so new lookup dictionaries containing `norm_name` are created.

In [48]:
source2_lookup_full = train_source2_keyed_v3.set_index('entity_id')[['business_name', 'business_address', 'norm_name']].to_dict('index')
source3_lookup_full = train_source3_keyed_v3.set_index('entity_id')[['business_name', 'business_address', 'norm_name']].to_dict('index')

print("Full lookups built.")

Full lookups built.


#### Step 3 — Measure Pre-filter Recall

Before applying the pre-filter to the full validation set, it is evaluated on a sample of 500 validation entities.

For each entity with known true matches, we check how many of its true matches remain within the top-50 candidates selected by the Jaccard pre-filter.

This measures whether the pre-filter is safe enough to place before RapidFuzz scoring. A large loss in recall would indicate that the top-50 cutoff is too aggressive and should be increased or replaced.

In [49]:
sample_val = val_source1_keyed_v3.head(500)  # bigger sample since this step is cheap

prefilter_recalls = []

for _, row in sample_val.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    norm_name1 = row['norm_name']
    
    candidate_ids = list(get_candidates_v3(block_key))
    
    def lookup_fn(cid):
        return source2_lookup_full.get(cid) if cid.startswith('S2-') else source3_lookup_full.get(cid)
    
    combined_lookup = {cid: lookup_fn(cid) for cid in candidate_ids}
    
    filtered_ids = jaccard_prefilter(norm_name1, candidate_ids, combined_lookup, top_k=50)
    
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    if len(true_set) == 0:
        continue
    
    found = len(true_set & set(filtered_ids))
    prefilter_recalls.append(found / len(true_set))

print("Pre-filter recall (top-50, sample of 500):", sum(prefilter_recalls) / len(prefilter_recalls))
print("Entities with true matches in sample:", len(prefilter_recalls))

Pre-filter recall (top-50, sample of 500): 0.62677384344051
Entities with true matches in sample: 468


### Initial Result

The top-50 Jaccard pre-filter retained 62.79% of the true matches on a sample of 500 validation entities with true matches.

This indicates that a top-50 cutoff is too aggressive for the current blocking strategy, so it will not be used as the final pre-filter cutoff. Additional K values will be evaluated to determine whether a larger candidate set can provide a better recall-efficiency trade-off.

#### Testing Multiple Top-K Cutoffs

The initial top-50 pre-filter retained only 62.79% of true matches in the 500-entity sample, indicating that the cutoff was too aggressive.

To find a better balance between computational efficiency and recall, the same validation sample is now evaluated using several larger candidate cutoffs: 50, 100, 200, 300, and 500.

The goal is to identify how recall changes as more candidates are retained before RapidFuzz scoring.

In [50]:
for k in [50, 100, 200, 300, 500]:
    prefilter_recalls = []
    
    for _, row in sample_val.iterrows():
        entity_id = row['entity_id']
        block_key = row['block_key']
        norm_name1 = row['norm_name']
        
        candidate_ids = list(get_candidates_v3(block_key))
        
        def lookup_fn(cid):
            return source2_lookup_full.get(cid) if cid.startswith('S2-') else source3_lookup_full.get(cid)
        
        combined_lookup = {cid: lookup_fn(cid) for cid in candidate_ids}
        
        filtered_ids = jaccard_prefilter(norm_name1, candidate_ids, combined_lookup, top_k=k)
        
        true_set = parse_ids(val_gt_lookup.get(entity_id))
        if len(true_set) == 0:
            continue
        
        found = len(true_set & set(filtered_ids))
        prefilter_recalls.append(found / len(true_set))
    
    print(f"K={k}: recall = {sum(prefilter_recalls) / len(prefilter_recalls):.4f}")
    

K=50: recall = 0.6268
K=100: recall = 0.6742
K=200: recall = 0.7141
K=300: recall = 0.7241
K=500: recall = 0.7344


In [51]:
import os
print("CPU cores available:", os.cpu_count())


CPU cores available: 4


### 4.7 Experiment — Address-Augmented Blocking

The V3 blocking strategy uses country and the first normalized business-name token as the blocking key. Although V3 improved candidate recall to 79.26%, the resulting candidate set remains large, averaging approximately 5,831 candidates per Source 1 entity.

A new blocking strategy is therefore being investigated that adds a coarse address signal to the existing name-based key. The initial hypothesis is that combining the first normalized name token with an address token may split large blocking buckets into smaller groups and reduce the number of candidate pairs.

Before rebuilding the full Source 2 and Source 3 indexes, the proposed address token is inspected on a small sample to determine whether it provides a useful locality signal or is dominated by house numbers and other non-informative values.

In [52]:
def extract_city_token(address):
    """Very cheap: take the first alphabetic token from the address as a coarse locality signal."""
    if pd.isna(address):
        return ""
    addr_norm = normalize_name(address)  # reuse existing normalization
    tokens = addr_norm.split()
    return tokens[0] if tokens else ""

def build_block_key_v4(df):
    df = df.copy()
    df['norm_name'] = df['business_name'].apply(normalize_name)
    df['addr_token'] = df['business_address'].apply(extract_city_token)
    df['block_key'] = df['country'] + '||' + df['norm_name'].apply(first_token) + '||' + df['addr_token']
    return df

In [53]:
sample_check = train_source1.head(15).copy()
sample_check['norm_name'] = sample_check['business_name'].apply(normalize_name)
sample_check['addr_token'] = sample_check['business_address'].apply(extract_city_token)
print(sample_check[['business_name', 'business_address', 'norm_name', 'addr_token']].to_string())

                               business_name                                                                                                           business_address                   norm_name addr_token
0                        Orelee's Barbershop                                                                                     1795 Westchester Drive, High Point, NC           orelee barbershop       1795
1                                Prime Money                                                                                            17560 Ellis Road, Tahlequah, OK                 prime money      17560
2                              B+ Retail Inc                                                                                        1712 Montebello Avenue, Phoenix, AZ                    b retail       1712
3                              Christ Chapel                                                                          2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD     

#### Initial Inspection

The sample shows that the first normalized address token is frequently a house number, unit indicator, or other structural component rather than a reliable locality signal. Examples include `1795`, `17560`, `1712`, `797`, `2505`, `1111`, `unit`, and `h`.

Because this token does not consistently represent a useful geographic or address component, it is not suitable as the second blocking key in its current form. The full V4 indexes will therefore not be built using this feature.

#### V4 Candidate — Last Address Segment

The previous address-based blocking attempt used the first address token, but inspection showed that it was frequently a house number, unit indicator, or other structural component rather than a useful locality signal.

As an alternative, this experiment investigates the last comma-separated segment of the address. Business addresses often place geographic information such as a city, state, or region toward the end of the address.

The last segment is therefore tested as a coarse address signal. This is only an initial inspection; the full blocking indexes will not be rebuilt unless the sample suggests that the signal is useful.

In [54]:
def extract_last_segment(address):
    """Take the last comma-separated segment of the address as a coarse locality signal."""
    if pd.isna(address):
        return ""
    segments = address.split(',')
    last = segments[-1].strip()
    return normalize_name(last)  # reuse existing normalization for consistency

sample_check = train_source1.head(15).copy()
sample_check['norm_name'] = sample_check['business_name'].apply(normalize_name)
sample_check['addr_last_seg'] = sample_check['business_address'].apply(extract_last_segment)
print(sample_check[['business_name', 'business_address', 'norm_name', 'addr_last_seg']].to_string())

                               business_name                                                                                                           business_address                   norm_name        addr_last_seg
0                        Orelee's Barbershop                                                                                     1795 Westchester Drive, High Point, NC           orelee barbershop                   nc
1                                Prime Money                                                                                            17560 Ellis Road, Tahlequah, OK                 prime money                   ok
2                              B+ Retail Inc                                                                                        1712 Montebello Avenue, Phoenix, AZ                    b retail                   az
3                              Christ Chapel                                                                          2100 Cameron D

#### V4 Address Signal Inspection — Result

The sample shows that the last comma-separated address segment frequently contains geographic information such as a state, province, or region. Examples include `nc`, `ok`, `az`, `west bengal`, `maharashtra`, `tn`, `in`, and `telangana`.

Some addresses use a different ordering and therefore produce a street or locality value instead. However, the signal appears substantially more informative than the first address token, which was frequently a house number or unit indicator.

The last address segment will therefore be tested as an additional blocking signal. The full indexes will be evaluated for candidate recall and candidate volume before deciding whether to retain this strategy.

#### V4 — Address-Augmented Blocking: Bucket Size Feasibility Test

The V3 blocking strategy uses country and the first normalized business-name token. To reduce the large candidate volume, this experiment adds the last comma-separated address segment as a second blocking signal.

Before rebuilding the full Source 2 and Source 3 indexes, the new blocking key is tested on a 200,000-record Source 2 sample. The purpose is to determine whether the additional address signal substantially reduces bucket sizes while remaining computationally practical.

If the resulting buckets are sufficiently smaller, the strategy will then be evaluated on the validation set for candidate recall before being considered for the final pipeline.

In [55]:
def build_block_key_v5(df):
    df = df.copy()
    df['norm_name'] = df['business_name'].apply(normalize_name)
    df['addr_last_seg'] = df['business_address'].apply(extract_last_segment)
    df['block_key'] = df['country'] + '||' + df['norm_name'].apply(first_token) + '||' + df['addr_last_seg']
    return df

In [56]:
sample_s2 = train_source2.sample(n=200000, random_state=42)
sample_s2_keyed = build_block_key_v5(sample_s2)

bucket_sizes_sample = sample_s2_keyed.groupby('block_key').size()
print("Sample bucket sizes — mean:", bucket_sizes_sample.mean(), "median:", bucket_sizes_sample.median(), "max:", bucket_sizes_sample.max())
print("Number of distinct keys:", len(bucket_sizes_sample))

Sample bucket sizes — mean: 1.6035534744994908 median: 1.0 max: 292
Number of distinct keys: 124723


In [57]:
build_block_key_v4 = build_block_key_v5  # alias for consistent naming going forward

In [58]:
train_source2_keyed_v4 = build_block_key_v5(train_source2)
train_source3_keyed_v4 = build_block_key_v5(train_source3)
val_source1_keyed_v4 = build_block_key_v5(val_source1)

print("Done building v4 keys.")

Done building v4 keys.


In [59]:
index_source2_v4 = build_index(train_source2_keyed_v4)
index_source3_v4 = build_index(train_source3_keyed_v4)

print("Number of distinct keys in Source 2 index (v4):", len(index_source2_v4))
print("Number of distinct keys in Source 3 index (v4):", len(index_source3_v4))

Number of distinct keys in Source 2 index (v4): 1540612
Number of distinct keys in Source 3 index (v4): 1799828


In [60]:
bucket_sizes_s2_v4 = [len(v) for v in index_source2_v4.values()]
bucket_sizes_s3_v4 = [len(v) for v in index_source3_v4.values()]

print(
    "Source 2 (v4) bucket sizes — mean:",
    np.mean(bucket_sizes_s2_v4),
    "median:",
    np.median(bucket_sizes_s2_v4),
    "max:",
    np.max(bucket_sizes_s2_v4)
)

print(
    "Source 3 (v4) bucket sizes — mean:",
    np.mean(bucket_sizes_s3_v4),
    "median:",
    np.median(bucket_sizes_s3_v4),
    "max:",
    np.max(bucket_sizes_s3_v4)
)

Source 2 (v4) bucket sizes — mean: 3.2679324839739015 median: 1.0 max: 7118
Source 3 (v4) bucket sizes — mean: 2.93672673166547 median: 1.0 max: 6651


In [61]:
def get_candidates_v4(block_key):
    c2 = index_source2_v4.get(block_key, [])
    c3 = index_source3_v4.get(block_key, [])
    return set(c2) | set(c3)

recalls_v4 = []
candidate_counts_v4 = []

for _, row in val_source1_keyed_v4.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates_v4(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts_v4.append(len(candidate_set))
    
    if len(true_set) == 0:
        continue
    
    found = len(true_set & candidate_set)
    recalls_v4.append(found / len(true_set))

print(
    "Average candidates per entity (v4):",
    sum(candidate_counts_v4) / len(candidate_counts_v4)
)

print(
    "Candidate recall (v4):",
    sum(recalls_v4) / len(recalls_v4)
)

print(
    "Entities with true matches evaluated:",
    len(recalls_v4)
)

Average candidates per entity (v4): 163.58522084895722
Candidate recall (v4): 0.2558794237404655
Entities with true matches evaluated: 416780


## Experiment 4.7 Results — Address-Augmented Blocking

Adding the last comma-separated address segment to the V3 blocking key reduced the average candidate count dramatically, from approximately 5,831 to 164 candidates per Source 1 entity.

However, this reduction came at a major cost to candidate recall. Candidate recall decreased from 79.26% in V3 to only 25.59% in V4.

This indicates that the additional address constraint is too restrictive for the noisy entity-resolution data. Many true matches do not share the same final address segment even when they refer to the same business.

Since a true match excluded during blocking cannot be recovered by the later matching model, preserving candidate recall is critical. Therefore, V4 will not be used for the final matching pipeline.

V3 remains the current blocking strategy because it provides substantially higher candidate recall while avoiding the overly restrictive address constraint.

### 5.1 Experiment — Sampled Validation of RapidFuzz Matching on V3 Candidates

Blocking experiments are frozen for now. V3 (first-significant-word + country blocking,
with honorific/prefix stopword cleanup) is our accepted baseline: ~79.26% candidate
recall at ~5,831 candidates per Source 1 entity.

V4 (adding last-address-segment to the blocking key) was tested and rejected: it cut
average candidates to ~164 but collapsed candidate recall to ~25.59%, because address
field order and completeness vary too much across sources to use as a hard blocking
constraint.

Running RapidFuzz matching over the full 441K-entity validation set at V3's candidate
volume is estimated at ~10 hours, which isn't practical under the current time
constraint. This experiment instead scores a random sample of 20,000 Source 1
validation entities (random_state=42), using the same V3 blocking, batched RapidFuzz
`token_sort_ratio` scoring (0.6 name / 0.4 address), and computes precision, recall,
and F_0.5 at several similarity thresholds — scoring each candidate pair once and
sweeping thresholds over the stored scores, rather than rescoring per threshold.

Goal: pick a reasonable threshold to generate a first submittable `matching_results.tsv`
and `candidate_pairs.tsv`, not to finalize the matching approach.

In [62]:
sample_val_20k = val_source1_keyed_v3.sample(n=20000, random_state=42).reset_index(drop=True)
print("Sampled entities:", len(sample_val_20k))

Sampled entities: 20000


In [63]:
""" import time

start = time.time()

entity_scores = {}  # entity_id -> (true_set, [(cand_id, score), ...])

for i, row in sample_val_20k.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    name1 = row['business_name']
    addr1 = row['business_address']

    candidate_ids = list(get_candidates_v3(block_key))
    scored = score_entity_candidates(name1, addr1, candidate_ids)  # reuses existing batched function

    true_set = parse_ids(val_gt_lookup.get(entity_id))
    entity_scores[entity_id] = (true_set, scored)

    if (i + 1) % 2000 == 0:
        elapsed = time.time() - start
        print(f"Processed {i+1}/20000 entities, {elapsed:.1f}s elapsed")

elapsed = time.time() - start
print(f"Done. Total time: {elapsed/60:.1f} minutes") """

' import time\n\nstart = time.time()\n\nentity_scores = {}  # entity_id -> (true_set, [(cand_id, score), ...])\n\nfor i, row in sample_val_20k.iterrows():\n    entity_id = row[\'entity_id\']\n    block_key = row[\'block_key\']\n    name1 = row[\'business_name\']\n    addr1 = row[\'business_address\']\n\n    candidate_ids = list(get_candidates_v3(block_key))\n    scored = score_entity_candidates(name1, addr1, candidate_ids)  # reuses existing batched function\n\n    true_set = parse_ids(val_gt_lookup.get(entity_id))\n    entity_scores[entity_id] = (true_set, scored)\n\n    if (i + 1) % 2000 == 0:\n        elapsed = time.time() - start\n        print(f"Processed {i+1}/20000 entities, {elapsed:.1f}s elapsed")\n\nelapsed = time.time() - start\nprint(f"Done. Total time: {elapsed/60:.1f} minutes") '

In [64]:
"""def evaluate_at_threshold(entity_scores, threshold):
    scores = []
    for entity_id, (true_set, scored) in entity_scores.items():
        pred_set = {cand_id for cand_id, score in scored if score >= threshold}
        scores.append(f_beta_score(true_set, pred_set))
    return sum(scores) / len(scores)

def precision_recall_at_threshold(entity_scores, threshold):
    total_tp, total_pred, total_true = 0, 0, 0
    for entity_id, (true_set, scored) in entity_scores.items():
        pred_set = {cand_id for cand_id, score in scored if score >= threshold}
        total_tp += len(true_set & pred_set)
        total_pred += len(pred_set)
        total_true += len(true_set)
    precision = total_tp / total_pred if total_pred > 0 else 0
    recall = total_tp / total_true if total_true > 0 else 0
    return precision, recall

thresholds = [50, 55, 60, 65, 70, 75, 80, 85, 90]

print(f"{'Threshold':>10} {'F0.5':>8} {'Precision':>10} {'Recall':>8}")
for t in thresholds:
    f05 = evaluate_at_threshold(entity_scores, t)
    prec, rec = precision_recall_at_threshold(entity_scores, t)
    print(f"{t:>10} {f05:>8.4f} {prec:>10.4f} {rec:>8.4f}")"""

'def evaluate_at_threshold(entity_scores, threshold):\n    scores = []\n    for entity_id, (true_set, scored) in entity_scores.items():\n        pred_set = {cand_id for cand_id, score in scored if score >= threshold}\n        scores.append(f_beta_score(true_set, pred_set))\n    return sum(scores) / len(scores)\n\ndef precision_recall_at_threshold(entity_scores, threshold):\n    total_tp, total_pred, total_true = 0, 0, 0\n    for entity_id, (true_set, scored) in entity_scores.items():\n        pred_set = {cand_id for cand_id, score in scored if score >= threshold}\n        total_tp += len(true_set & pred_set)\n        total_pred += len(pred_set)\n        total_true += len(true_set)\n    precision = total_tp / total_pred if total_pred > 0 else 0\n    recall = total_tp / total_true if total_true > 0 else 0\n    return precision, recall\n\nthresholds = [50, 55, 60, 65, 70, 75, 80, 85, 90]\n\nprint(f"{\'Threshold\':>10} {\'F0.5\':>8} {\'Precision\':>10} {\'Recall\':>8}")\nfor t in threshold

### 5.2 Optimized Threshold Sweep

The initial threshold-sweep implementation repeatedly scanned the stored
candidate scores for every threshold. Although the RapidFuzz scoring itself
completed successfully, this evaluation method caused excessive memory
pressure and very slow execution on the available hardware.

The stored `entity_scores` are retained and reused; RapidFuzz scoring is
not repeated.

This experiment uses a single pass over each entity's candidate scores.
Threshold counts are accumulated using a small difference-array approach,
allowing F_0.5, precision, and recall to be calculated for all thresholds
without repeatedly scanning the candidate list or creating large temporary
structures.

The thresholds evaluated are 50, 55, 60, 65, 70, 75, 80, 85, and 90.

In [65]:
"""import time
from bisect import bisect_right
from itertools import accumulate

thresholds = [50, 55, 60, 65, 70, 75, 80, 85, 90]
n_thresh = len(thresholds)

f05_sums = [0.0] * n_thresh
tp_totals = [0] * n_thresh
pred_totals = [0] * n_thresh
n_entities = 0
true_total = 0

start = time.time()

for entity_id, (true_set, scored) in entity_scores.items():
    n_entities += 1
    true_count = len(true_set)
    true_total += true_count

    pred_diff = [0] * (n_thresh + 1)
    tp_diff = [0] * (n_thresh + 1)

    for cand_id, score in scored:              # each candidate visited exactly once
        k = bisect_right(thresholds, score + 1)  # number of thresholds this score passes (score >= threshold)
        if k == 0:
            continue
        pred_diff[0] += 1
        pred_diff[k] -= 1
        if cand_id in true_set:
            tp_diff[0] += 1
            tp_diff[k] -= 1

    pred_counts = list(accumulate(pred_diff))[:n_thresh]
    tp_counts = list(accumulate(tp_diff))[:n_thresh]

    for j in range(n_thresh):
        pred_count = pred_counts[j]
        tp = tp_counts[j]

        if true_count == 0:
            f05 = 1.0 if pred_count == 0 else 0.0
        elif pred_count == 0:
            f05 = 0.0
        else:
            precision = tp / pred_count
            recall = tp / true_count
            f05 = 0.0 if (precision == 0 and recall == 0) else (1.25 * precision * recall) / (0.25 * precision + recall)

        f05_sums[j] += f05
        tp_totals[j] += tp
        pred_totals[j] += pred_count

    if n_entities % 5000 == 0:
        print(f"Processed {n_entities}/{len(entity_scores)} entities, {time.time()-start:.1f}s elapsed")

elapsed = time.time() - start
print(f"Done. Total time: {elapsed:.1f}s")

print(f"\n{'Threshold':>10} {'F0.5':>8} {'Precision':>10} {'Recall':>8}")
for j, t in enumerate(thresholds):
    f05_macro = f05_sums[j] / n_entities
    prec = tp_totals[j] / pred_totals[j] if pred_totals[j] > 0 else 0
    rec = tp_totals[j] / true_total if true_total > 0 else 0
    print(f"{t:>10} {f05_macro:>8.4f} {prec:>10.4f} {rec:>8.4f}")"""

'import time\nfrom bisect import bisect_right\nfrom itertools import accumulate\n\nthresholds = [50, 55, 60, 65, 70, 75, 80, 85, 90]\nn_thresh = len(thresholds)\n\nf05_sums = [0.0] * n_thresh\ntp_totals = [0] * n_thresh\npred_totals = [0] * n_thresh\nn_entities = 0\ntrue_total = 0\n\nstart = time.time()\n\nfor entity_id, (true_set, scored) in entity_scores.items():\n    n_entities += 1\n    true_count = len(true_set)\n    true_total += true_count\n\n    pred_diff = [0] * (n_thresh + 1)\n    tp_diff = [0] * (n_thresh + 1)\n\n    for cand_id, score in scored:              # each candidate visited exactly once\n        k = bisect_right(thresholds, score + 1)  # number of thresholds this score passes (score >= threshold)\n        if k == 0:\n            continue\n        pred_diff[0] += 1\n        pred_diff[k] -= 1\n        if cand_id in true_set:\n            tp_diff[0] += 1\n            tp_diff[k] -= 1\n\n    pred_counts = list(accumulate(pred_diff))[:n_thresh]\n    tp_counts = list(accu

In [66]:
"""import pickle

with open('entity_scores_20k.pkl', 'wb') as f:
    pickle.dump(entity_scores, f)

print("Saved entity_scores to disk.")"""

'import pickle\n\nwith open(\'entity_scores_20k.pkl\', \'wb\') as f:\n    pickle.dump(entity_scores, f)\n\nprint("Saved entity_scores to disk.")'

In [67]:
"""import random

random.seed(42)
sample_keys = random.sample(list(entity_scores.keys()), 3000)
entity_scores_small = {k: entity_scores[k] for k in sample_keys}

print("Small sample size:", len(entity_scores_small))"""

'import random\n\nrandom.seed(42)\nsample_keys = random.sample(list(entity_scores.keys()), 3000)\nentity_scores_small = {k: entity_scores[k] for k in sample_keys}\n\nprint("Small sample size:", len(entity_scores_small))'

In [68]:
sample_val_3k = val_source1_keyed_v3.sample(n=3000, random_state=42).reset_index(drop=True)
print("Sampled entities:", len(sample_val_3k))

Sampled entities: 3000


In [72]:
import time
import pickle
from bisect import bisect_right
from itertools import accumulate

thresholds = [70, 75, 80, 85]
n_thresh = len(thresholds)

f05_sums = [0.0] * n_thresh
tp_totals = [0] * n_thresh
pred_totals = [0] * n_thresh
n_entities = 0
true_total = 0

batch_size = 250
start = time.time()

for batch_start in range(0, len(sample_val_3k), batch_size):
    batch = sample_val_3k.iloc[batch_start:batch_start + batch_size]

    for _, row in batch.iterrows():
        entity_id = row['entity_id']
        block_key = row['block_key']
        name1 = row['business_name']
        addr1 = row['business_address']

        candidate_ids = list(get_candidates_v3(block_key))
        scored = score_entity_candidates(name1, addr1, candidate_ids)  # only this entity's pairs, ever

        true_set = parse_ids(val_gt_lookup.get(entity_id))
        true_count = len(true_set)

        n_entities += 1
        true_total += true_count

        pred_diff = [0] * (n_thresh + 1)
        tp_diff = [0] * (n_thresh + 1)

        for cand_id, score in scored:
            k = bisect_right(thresholds, score)
            if k == 0:
                continue
            pred_diff[0] += 1
            pred_diff[k] -= 1
            if cand_id in true_set:
                tp_diff[0] += 1
                tp_diff[k] -= 1
        # `scored` goes out of scope at the next loop iteration — nothing retained

        pred_counts = list(accumulate(pred_diff))[:n_thresh]
        tp_counts = list(accumulate(tp_diff))[:n_thresh]

        for j in range(n_thresh):
            pred_count = pred_counts[j]
            tp = tp_counts[j]

            if true_count == 0:
                f05 = 1.0 if pred_count == 0 else 0.0
            elif pred_count == 0:
                f05 = 0.0
            else:
                precision = tp / pred_count
                recall = tp / true_count
                f05 = 0.0 if (precision == 0 and recall == 0) else (1.25 * precision * recall) / (0.25 * precision + recall)

            f05_sums[j] += f05
            tp_totals[j] += tp
            pred_totals[j] += pred_count

    # Checkpoint compact stats (tiny — a few numbers) after each batch, for safety
    with open('threshold_stats_checkpoint.pkl', 'wb') as f:
        pickle.dump({
            'f05_sums': f05_sums, 'tp_totals': tp_totals, 'pred_totals': pred_totals,
            'n_entities': n_entities, 'true_total': true_total
        }, f)

    print(f"Processed {n_entities}/{len(sample_val_3k)} entities, {time.time()-start:.1f}s elapsed")

elapsed = time.time() - start
print(f"\nDone. Total time: {elapsed:.1f}s")

print(f"\n{'Threshold':>10} {'F0.5':>8} {'Precision':>10} {'Recall':>8}")
for j, t in enumerate(thresholds):
    f05_macro = f05_sums[j] / n_entities
    prec = tp_totals[j] / pred_totals[j] if pred_totals[j] > 0 else 0
    rec = tp_totals[j] / true_total if true_total > 0 else 0
    print(f"{t:>10} {f05_macro:>8.4f} {prec:>10.4f} {rec:>8.4f}")

Processed 250/3000 entities, 138.2s elapsed
Processed 500/3000 entities, 156.1s elapsed
Processed 750/3000 entities, 172.0s elapsed
Processed 1000/3000 entities, 185.0s elapsed
Processed 1250/3000 entities, 195.7s elapsed
Processed 1500/3000 entities, 207.4s elapsed
Processed 1750/3000 entities, 219.5s elapsed
Processed 2000/3000 entities, 232.2s elapsed
Processed 2250/3000 entities, 243.6s elapsed
Processed 2500/3000 entities, 254.3s elapsed
Processed 2750/3000 entities, 265.9s elapsed
Processed 3000/3000 entities, 276.0s elapsed

Done. Total time: 276.0s

 Threshold     F0.5  Precision   Recall
        70   0.4748     0.1799   0.4716
        75   0.4999     0.5474   0.3618
        80   0.4511     0.8259   0.2572
        85   0.3568     0.9128   0.1747


In [69]:
import os 
print(os.path.exists('entity_scores_20k.pkl'))

False


In [70]:
"""import os
size_gb= os.path.getsize('entity_scores_20k.pkl') / (1024**3)
print(f"File size:{size_gb: .2f} GB")"""

'import os\nsize_gb= os.path.getsize(\'entity_scores_20k.pkl\') / (1024**3)\nprint(f"File size:{size_gb: .2f} GB")'

### 5.3 Note — Kernel Restart Wiped entity_scores

After interrupting the threshold sweep due to memory pressure, we attempted to
save `entity_scores` to disk via pickle. This raised `NameError: entity_scores
is not defined`, revealing that the Jupyter kernel had already silently
restarted under memory pressure — losing entity_scores before it could be
persisted.

A leftover 0-byte `entity_scores_20k.pkl` file was found (from the failed save
attempt) and deleted, since it contained no usable data.

Since `entity_scores` no longer exists, we must re-run the RapidFuzz scoring
stage. This time, we checkpoint to disk incrementally (every 2,000 entities,
clearing memory after each save) so a future crash costs at most one chunk of
work, not the full run.

In [71]:
print(len(sample_val_20k))

20000


## Test-Time Feasibility Check

Before generating the final test predictions, we measure the runtime of the
current V3 blocking and RapidFuzz matching pipeline on a small sample of
unseen test entities.

This experiment is only for estimating runtime. Since the test set has no
ground truth, it does not produce an accuracy score.

In [86]:
# Build V3 blocking structures for the TEST data

test_source1_keyed_v3 = build_block_key(test_source1)
test_source2_keyed_v3 = build_block_key(test_source2)
test_source3_keyed_v3 = build_block_key(test_source3)

index_source2_test_v3 = build_index(test_source2_keyed_v3)
index_source3_test_v3 = build_index(test_source3_keyed_v3)

print("Test V3 structures ready.")
print("Test Source 1:", len(test_source1_keyed_v3))
print("Test Source 2 keys:", len(index_source2_test_v3))
print("Test Source 3 keys:", len(index_source3_test_v3))

Test V3 structures ready.
Test Source 1: 1732544
Test Source 2 keys: 343967
Test Source 3 keys: 384817


In [87]:
def get_candidates_test_v3(block_key):
    c2 = index_source2_test_v3.get(block_key, [])
    c3 = index_source3_test_v3.get(block_key, [])
    return set(c2) | set(c3)

In [90]:
# Test-time scoring setup

from rapidfuzz import process, fuzz

# Build lookups for TEST Source 2 and Source 3
source2_test_lookup = test_source2_keyed_v3.set_index(
    'entity_id'
)[['business_name', 'business_address']].to_dict('index')

source3_test_lookup = test_source3_keyed_v3.set_index(
    'entity_id'
)[['business_name', 'business_address']].to_dict('index')


# Sample 100 TEST Source 1 entities for runtime measurement
sample_test_100 = test_source1_keyed_v3.sample(
    n=100,
    random_state=42
).reset_index(drop=True)


def score_entity_candidates_test(name1, addr1, candidate_ids):
    """Score one TEST Source 1 entity against TEST Source 2/3 candidates."""

    cand_names = []
    cand_addrs = []
    valid_ids = []

    for cand_id in candidate_ids:

        lookup = (
            source2_test_lookup
            if cand_id.startswith('S2-')
            else source3_test_lookup
        )

        cand_data = lookup.get(cand_id)

        if cand_data is None:
            continue

        valid_ids.append(cand_id)

        cand_names.append(
            str(cand_data['business_name'])
            if pd.notna(cand_data['business_name'])
            else ""
        )

        cand_addrs.append(
            str(cand_data['business_address'])
            if pd.notna(cand_data['business_address'])
            else ""
        )

    if not valid_ids:
        return []

    name1_str = (
        str(name1)
        if pd.notna(name1)
        else ""
    )

    addr1_str = (
        str(addr1)
        if pd.notna(addr1)
        else ""
    )

    name_scores = process.cdist(
        [name1_str],
        cand_names,
        scorer=fuzz.token_sort_ratio
    )[0]

    addr_scores = process.cdist(
        [addr1_str],
        cand_addrs,
        scorer=fuzz.token_sort_ratio
    )[0]

    combined = 0.6 * name_scores + 0.4 * addr_scores

    return list(zip(valid_ids, combined))


print("Test-time scoring setup ready.")
print("Sample size:", len(sample_test_100))

Test-time scoring setup ready.
Sample size: 100


In [91]:
import time

start = time.time()

total_pairs = 0
candidate_counts = []

for _, row in sample_test_100.iterrows():

    candidate_ids = list(
        get_candidates_test_v3(row['block_key'])
    )

    candidate_counts.append(len(candidate_ids))

    scored = score_entity_candidates_test(
        row['business_name'],
        row['business_address'],
        candidate_ids
    )

    total_pairs += len(scored)

elapsed = time.time() - start

print(f"Time for 100 test entities: {elapsed:.2f} seconds")
print(f"Candidate pairs scored: {total_pairs}")
print(f"Average candidates/entity: {sum(candidate_counts)/len(candidate_counts):.1f}")
print(
    f"Estimated time for all {len(test_source1_keyed_v3):,} "
    f"test entities: "
    f"{elapsed / 100 * len(test_source1_keyed_v3) / 3600:.2f} hours"
)

Time for 100 test entities: 3.53 seconds
Candidate pairs scored: 445263
Average candidates/entity: 4452.6
Estimated time for all 1,732,544 test entities: 16.98 hours


In [92]:
# Benchmark: faster test-time scoring without changing the matching logic

def score_entity_candidates_test_fast(name1, addr1, candidate_ids):
    """Score test candidates using the same 0.6 name / 0.4 address RapidFuzz logic."""

    cand_names = []
    cand_addrs = []
    valid_ids = []

    for cand_id in candidate_ids:

        lookup = (
            source2_test_lookup
            if cand_id.startswith('S2-')
            else source3_test_lookup
        )

        cand_data = lookup.get(cand_id)

        if cand_data is None:
            continue

        valid_ids.append(cand_id)

        cand_names.append(
            str(cand_data['business_name'])
            if pd.notna(cand_data['business_name'])
            else ""
        )

        cand_addrs.append(
            str(cand_data['business_address'])
            if pd.notna(cand_data['business_address'])
            else ""
        )

    if not valid_ids:
        return valid_ids, np.array([])

    name1_str = str(name1) if pd.notna(name1) else ""
    addr1_str = str(addr1) if pd.notna(addr1) else ""

    name_scores = process.cdist(
        [name1_str],
        cand_names,
        scorer=fuzz.token_sort_ratio
    )[0]

    addr_scores = process.cdist(
        [addr1_str],
        cand_addrs,
        scorer=fuzz.token_sort_ratio
    )[0]

    combined = 0.6 * name_scores + 0.4 * addr_scores

    return valid_ids, combined


# Benchmark on the SAME 100 test entities
import time

start = time.time()

total_pairs_fast = 0

for _, row in sample_test_100.iterrows():

    candidate_ids = list(
        get_candidates_test_v3(row['block_key'])
    )

    valid_ids, scores = score_entity_candidates_test_fast(
        row['business_name'],
        row['business_address'],
        candidate_ids
    )

    total_pairs_fast += len(valid_ids)

elapsed_fast = time.time() - start

print(f"Fast scorer time for 100 test entities: {elapsed_fast:.2f} seconds")
print(f"Candidate pairs scored: {total_pairs_fast}")
print(
    f"Estimated time for all {len(test_source1_keyed_v3):,} "
    f"test entities: "
    f"{elapsed_fast / 100 * len(test_source1_keyed_v3) / 3600:.2f} hours"
)

Fast scorer time for 100 test entities: 3.59 seconds
Candidate pairs scored: 445263
Estimated time for all 1,732,544 test entities: 17.28 hours


In [93]:
# Benchmark Jaccard pre-filter on TEST data
# This only measures candidate reduction and runtime.

import time

# Build TEST lookup dictionaries containing normalized names
source2_test_lookup_full = test_source2_keyed_v3.set_index(
    'entity_id'
)[['business_name', 'business_address', 'norm_name']].to_dict('index')

source3_test_lookup_full = test_source3_keyed_v3.set_index(
    'entity_id'
)[['business_name', 'business_address', 'norm_name']].to_dict('index')


def jaccard_prefilter_test(name1, candidate_ids, top_k=500):

    tokens1 = set(name1.split())

    scored = []

    for cand_id in candidate_ids:

        lookup = (
            source2_test_lookup_full
            if cand_id.startswith('S2-')
            else source3_test_lookup_full
        )

        cand_data = lookup.get(cand_id)

        if cand_data is None:
            continue

        tokens2 = set(cand_data['norm_name'].split())

        union_size = len(tokens1 | tokens2)

        if union_size == 0:
            jaccard = 0
        else:
            jaccard = len(tokens1 & tokens2) / union_size

        scored.append((cand_id, jaccard))

    scored.sort(key=lambda x: x[1], reverse=True)

    return [cand_id for cand_id, _ in scored[:top_k]]


# Benchmark on the SAME 100 test entities
start = time.time()

total_original = 0
total_filtered = 0

for _, row in sample_test_100.iterrows():

    candidate_ids = list(
        get_candidates_test_v3(row['block_key'])
    )

    total_original += len(candidate_ids)

    filtered_ids = jaccard_prefilter_test(
        row['norm_name'],
        candidate_ids,
        top_k=500
    )

    total_filtered += len(filtered_ids)

elapsed_prefilter = time.time() - start

print(f"Jaccard pre-filter time for 100 entities: {elapsed_prefilter:.2f} seconds")
print(f"Original candidate pairs: {total_original}")
print(f"Filtered candidate pairs: {total_filtered}")

print(
    f"Average original candidates/entity: "
    f"{total_original / 100:.1f}"
)

print(
    f"Average filtered candidates/entity: "
    f"{total_filtered / 100:.1f}"
)

print(
    f"Candidate reduction: "
    f"{(1 - total_filtered / total_original) * 100:.2f}%"
)

Jaccard pre-filter time for 100 entities: 3.70 seconds
Original candidate pairs: 445263
Filtered candidate pairs: 33526
Average original candidates/entity: 4452.6
Average filtered candidates/entity: 335.3
Candidate reduction: 92.47%


In [94]:
# Benchmark larger Jaccard cutoffs on the same 100 TEST entities

for k in [1000, 2000]:

    start = time.time()

    total_filtered = 0

    for _, row in sample_test_100.iterrows():

        candidate_ids = list(
            get_candidates_test_v3(row['block_key'])
        )

        filtered_ids = jaccard_prefilter_test(
            row['norm_name'],
            candidate_ids,
            top_k=k
        )

        total_filtered += len(filtered_ids)

    elapsed = time.time() - start

    print(f"\nK={k}")
    print(f"Time: {elapsed:.2f} seconds")
    print(f"Filtered candidate pairs: {total_filtered}")
    print(f"Average candidates/entity: {total_filtered / 100:.1f}")
    print(
        f"Estimated RapidFuzz test time if ALL filtered candidates "
        f"are scored: "
        f"{elapsed / 100 * len(test_source1_keyed_v3) / 3600:.2f} hours "
        f"(Jaccard only)"
    )


K=1000
Time: 2.03 seconds
Filtered candidate pairs: 60987
Average candidates/entity: 609.9
Estimated RapidFuzz test time if ALL filtered candidates are scored: 9.76 hours (Jaccard only)

K=2000
Time: 1.81 seconds
Filtered candidate pairs: 107728
Average candidates/entity: 1077.3
Estimated RapidFuzz test time if ALL filtered candidates are scored: 8.69 hours (Jaccard only)


In [95]:
import time

# Precompute NaN-filled string columns ONCE (avoids repeated pd.notna/str() per candidate)
test_source2_keyed_v3['name_filled'] = test_source2_keyed_v3['business_name'].fillna('').astype(str)
test_source2_keyed_v3['addr_filled'] = test_source2_keyed_v3['business_address'].fillna('').astype(str)
test_source3_keyed_v3['name_filled'] = test_source3_keyed_v3['business_name'].fillna('').astype(str)
test_source3_keyed_v3['addr_filled'] = test_source3_keyed_v3['business_address'].fillna('').astype(str)

source2_test_lookup_fast = test_source2_keyed_v3.set_index('entity_id')[['name_filled', 'addr_filled']].to_dict('index')
source3_test_lookup_fast = test_source3_keyed_v3.set_index('entity_id')[['name_filled', 'addr_filled']].to_dict('index')

def get_cand_arrays(candidate_ids):
    names, addrs, valid_ids = [], [], []
    for cid in candidate_ids:
        lookup = source2_test_lookup_fast if cid.startswith('S2-') else source3_test_lookup_fast
        d = lookup.get(cid)
        if d is None:
            continue
        valid_ids.append(cid)
        names.append(d['name_filled'])
        addrs.append(d['addr_filled'])
    return valid_ids, names, addrs


start = time.time()
total_pairs_grouped = 0
n_groups = 0

for block_key, group in sample_test_100.groupby('block_key'):
    candidate_ids = list(get_candidates_test_v3(block_key))
    valid_ids, cand_names, cand_addrs = get_cand_arrays(candidate_ids)
    n_groups += 1

    if not valid_ids:
        continue

    query_names = group['business_name'].fillna('').astype(str).tolist()
    query_addrs = group['business_address'].fillna('').astype(str).tolist()

    name_matrix = process.cdist(query_names, cand_names, scorer=fuzz.token_sort_ratio)
    addr_matrix = process.cdist(query_addrs, cand_addrs, scorer=fuzz.token_sort_ratio)

    combined_matrix = 0.6 * name_matrix + 0.4 * addr_matrix
    total_pairs_grouped += combined_matrix.size

elapsed_grouped = time.time() - start

print(f"Grouped scorer time for 100 test entities: {elapsed_grouped:.2f} seconds")
print(f"Unique block-key groups: {n_groups} (out of 100 entities)")
print(f"Total candidate pairs scored: {total_pairs_grouped}")
print(
    f"Estimated time for all {len(test_source1_keyed_v3):,} test entities: "
    f"{elapsed_grouped / 100 * len(test_source1_keyed_v3) / 3600:.2f} hours"
)

Grouped scorer time for 100 test entities: 13.90 seconds
Unique block-key groups: 99 (out of 100 entities)
Total candidate pairs scored: 445263
Estimated time for all 1,732,544 test entities: 66.90 hours


## Test-Time Inference Optimization — Block-Key Grouping

We tested grouping Source 1 test entities by `block_key` so that candidates for the same block could be constructed once and scored in a batched RapidFuzz operation.

### Benchmark

- Test sample: 100 Source 1 entities
- Total candidate pairs: 445,263
- Unique block-key groups: 99
- Runtime: 13.90 seconds
- Estimated full-test runtime: ~66.90 hours

### Result

This approach was rejected.

The 100-entity sample contained almost no repeated block keys (99 unique groups), so candidate-set reuse provided little benefit. The grouped implementation was also slower than the original entity-by-entity implementation (~3.53 seconds per 100 entities).

The V3 blocking strategy remains unchanged and frozen. Further work will focus on optimizing candidate lookup and scoring without relying on repeated block keys.

In [97]:
import time
import numpy as np

# --- One-time setup: flat arrays, NO reset_index needed (avoids the deep-copy crash) ---
s2_names = test_source2_keyed_v3['business_name'].fillna('').astype(str).to_numpy()
s2_addrs = test_source2_keyed_v3['business_address'].fillna('').astype(str).to_numpy()
s2_ids = test_source2_keyed_v3['entity_id'].to_numpy()

s3_names = test_source3_keyed_v3['business_name'].fillna('').astype(str).to_numpy()
s3_addrs = test_source3_keyed_v3['business_address'].fillna('').astype(str).to_numpy()
s3_ids = test_source3_keyed_v3['entity_id'].to_numpy()

# .indices returns POSITIONAL integer arrays — aligned with .to_numpy() row order above
index_source2_test_int = test_source2_keyed_v3.groupby('block_key').indices
index_source3_test_int = test_source3_keyed_v3.groupby('block_key').indices

empty_idx = np.array([], dtype=int)

def get_candidate_arrays_fast(block_key):
    idx2 = index_source2_test_int.get(block_key, empty_idx)
    idx3 = index_source3_test_int.get(block_key, empty_idx)

    names = np.concatenate([s2_names[idx2], s3_names[idx3]])
    addrs = np.concatenate([s2_addrs[idx2], s3_addrs[idx3]])
    ids = np.concatenate([s2_ids[idx2], s3_ids[idx3]])
    return ids, names, addrs

print("Setup done.")

# --- Benchmark on the SAME 100 test entities ---
start = time.time()
total_pairs = 0

for _, row in sample_test_100.iterrows():
    ids, names, addrs = get_candidate_arrays_fast(row['block_key'])
    if len(ids) == 0:
        continue

    name1_str = str(row['business_name']) if pd.notna(row['business_name']) else ""
    addr1_str = str(row['business_address']) if pd.notna(row['business_address']) else ""

    name_scores = process.cdist([name1_str], names, scorer=fuzz.token_sort_ratio)[0]
    addr_scores = process.cdist([addr1_str], addrs, scorer=fuzz.token_sort_ratio)[0]
    combined = 0.6 * name_scores + 0.4 * addr_scores

    total_pairs += len(combined)

elapsed = time.time() - start

print(f"Fast array-indexed scorer time for 100 test entities: {elapsed:.2f} seconds")
print(f"Total candidate pairs scored: {total_pairs}")
print(
    f"Estimated time for all {len(test_source1_keyed_v3):,} test entities: "
    f"{elapsed / 100 * len(test_source1_keyed_v3) / 3600:.2f} hours"
)

Setup done.
Fast array-indexed scorer time for 100 test entities: 3.79 seconds
Total candidate pairs scored: 445263
Estimated time for all 1,732,544 test entities: 18.26 hours


## Test-Time Inference Optimization — Integer Array Indexing

We tested replacing entity-ID dictionary lookups with flat NumPy arrays indexed by integer row positions.

### Benchmark

- Test sample: 100 Source 1 entities
- Candidate pairs: 445,263
- Runtime: 3.79 seconds
- Estimated full-test runtime: ~18.26 hours

### Comparison

The original entity-by-entity implementation required ~3.53 seconds for the same 100 entities, while the integer-array implementation required ~3.79 seconds.

### Result

This optimization was rejected.

Using integer row positions instead of entity-ID dictionary lookups did not reduce runtime. The benchmark indicates that the main computational cost is the RapidFuzz similarity scoring rather than candidate lookup.

The V3 blocking strategy remains frozen. Future optimization should focus on reducing the number of expensive similarity comparisons or changing the scoring implementation rather than further optimizing candidate retrieval.

In [99]:
import time

NAME_CUTOFF = 58.0   # safety margin below the exact 58.33 bound
ADDR_CUTOFF = 37.0   # safety margin below the exact 37.5 bound

start = time.time()
total_pairs = 0

for _, row in sample_test_100.iterrows():
    candidate_ids = list(get_candidates_test_v3(row['block_key']))
    valid_ids, cand_names, cand_addrs = get_cand_arrays(candidate_ids)  # reuse fast lookup from earlier

    if not valid_ids:
        continue

    name1_str = str(row['business_name']) if pd.notna(row['business_name']) else ""
    addr1_str = str(row['business_address']) if pd.notna(row['business_address']) else ""

    try:
        name_scores = process.cdist([name1_str], cand_names, scorer=fuzz.token_sort_ratio,
                                     score_cutoff=NAME_CUTOFF, workers=-1)[0]
        addr_scores = process.cdist([addr1_str], cand_addrs, scorer=fuzz.token_sort_ratio,
                                     score_cutoff=ADDR_CUTOFF, workers=-1)[0]
    except TypeError:
        # Fallback if installed rapidfuzz version doesn't support one of these kwargs
        name_scores = process.cdist([name1_str], cand_names, scorer=fuzz.token_sort_ratio, score_cutoff=NAME_CUTOFF)[0]
        addr_scores = process.cdist([addr1_str], cand_addrs, scorer=fuzz.token_sort_ratio, score_cutoff=ADDR_CUTOFF)[0]

    combined = 0.6 * name_scores + 0.4 * addr_scores
    total_pairs += len(combined)

elapsed = time.time() - start

print(f"Cutoff+parallel scorer time for 100 test entities: {elapsed:.2f} seconds")
print(f"Total candidate pairs scored: {total_pairs}")
print(
    f"Estimated time for all {len(test_source1_keyed_v3):,} test entities: "
    f"{elapsed / 100 * len(test_source1_keyed_v3) / 3600:.2f} hours"
)

Cutoff+parallel scorer time for 100 test entities: 9.70 seconds
Total candidate pairs scored: 445263
Estimated time for all 1,732,544 test entities: 46.69 hours


## Test-Time Inference Optimization — RapidFuzz Cutoffs and Parallelism

We tested RapidFuzz's `score_cutoff` and native parallelism (`workers=-1`) to reduce computation while preserving the threshold-75 decision boundary.

The combined score is:

`0.6 × name_score + 0.4 × address_score`

Therefore, for a candidate to reach 75:
- `name_score` must be at least 58.33 if the address score is 100.
- `address_score` must be at least 37.5 if the name score is 100.

We therefore tested conservative cutoffs of 58.0 for name and 37.0 for address.

### Benchmark

- Test sample: 100 Source 1 entities
- Candidate pairs: 445,263
- Runtime: 9.70 seconds
- Estimated full-test runtime: ~46.69 hours

### Result

This approach was rejected.

Although the cutoffs are mathematically derived from the existing threshold and do not change the theoretical threshold-75 decision boundary, the RapidFuzz implementation with these parameters was substantially slower than the original scorer.

The original scorer remains the runtime baseline. Further optimization must reduce the number of expensive similarity comparisons rather than add computation controls to the existing full-candidate scoring stage.

In [100]:
import numpy as np

def retrieve_top_n(name1, cand_names_arr, cand_ids_arr, top_n=100, max_len_ratio=0.5):
    name1 = str(name1) if pd.notna(name1) else ""
    len1 = len(name1)
    if len1 == 0 or len(cand_names_arr) == 0:
        return cand_ids_arr[:0]

    cand_lens = np.char.str_len(cand_names_arr.astype(str))
    len_diff_ratio = np.abs(cand_lens - len1) / max(len1, 1)
    mask = len_diff_ratio <= max_len_ratio

    f_names, f_ids = cand_names_arr[mask], cand_ids_arr[mask]
    if len(f_names) == 0:
        return f_ids

    name_scores = process.cdist([name1], f_names.tolist(), scorer=fuzz.token_sort_ratio)[0]
    order = np.argsort(-name_scores)
    return f_ids[order][:top_n]

In [101]:
import time

start = time.time()
retained_counts = []

for _, row in sample_test_100.iterrows():
    candidate_ids = list(get_candidates_test_v3(row['block_key']))
    valid_ids, cand_names, _ = get_cand_arrays(candidate_ids)
    if not valid_ids:
        retained_counts.append(0)
        continue
    top_ids = retrieve_top_n(row['business_name'], np.array(cand_names), np.array(valid_ids), top_n=100)
    retained_counts.append(len(top_ids))

elapsed = time.time() - start
print(f"Retrieval time for 100 test entities: {elapsed:.2f} sec")
print(f"Avg candidates retained: {sum(retained_counts)/len(retained_counts):.1f}")
print(f"Estimated full-test retrieval time: {elapsed/100*len(test_source1_keyed_v3)/3600:.2f} hours")

Retrieval time for 100 test entities: 3.48 sec
Avg candidates retained: 81.5
Estimated full-test retrieval time: 16.75 hours


In [102]:
import random
random.seed(42)

val_sample = val_source1_keyed_v3.sample(n=3000, random_state=42)
topn_values = [50, 100, 200]
recalls_by_n = {n: [] for n in topn_values}

start = time.time()
for _, row in val_sample.iterrows():
    entity_id = row['entity_id']
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    if len(true_set) == 0:
        continue  # consistent with earlier candidate-recall convention: skip singletons

    candidate_ids = list(get_candidates_v3(row['block_key']))
    names, ids = [], []
    for cid in candidate_ids:
        lookup = source2_lookup if cid.startswith('S2-') else source3_lookup
        d = lookup.get(cid)
        if d is None:
            continue
        ids.append(cid)
        names.append(str(d['business_name']) if pd.notna(d['business_name']) else "")

    if not ids:
        for n in topn_values:
            recalls_by_n[n].append(0.0)
        continue

    ranked_ids = retrieve_top_n(row['business_name'], np.array(names), np.array(ids), top_n=max(topn_values))

    for n in topn_values:
        found = len(true_set & set(ranked_ids[:n]))
        recalls_by_n[n].append(found / len(true_set))

elapsed_val = time.time() - start
print(f"Validation check time: {elapsed_val:.1f}s ({len(val_sample)} entities sampled)")
for n in topn_values:
    print(f"Top-{n}: candidate recall = {sum(recalls_by_n[n])/len(recalls_by_n[n]):.4f}")

Validation check time: 2749.3s (3000 entities sampled)
Top-50: candidate recall = 0.5761
Top-100: candidate recall = 0.6163
Top-200: candidate recall = 0.6493


## Test-Time Candidate Ranking — Name Length Prefilter + Top-N RapidFuzz

We tested a two-stage candidate reduction strategy after V3 blocking.

### Approach

1. Compare candidate name length with the Source 1 name using a vectorized NumPy length-ratio filter.
2. Rank the surviving candidates using name-only `token_sort_ratio`.
3. Retain the top 50, 100, or 200 candidates for subsequent combined name/address scoring.

### Test Runtime

- Test sample: 100 Source 1 entities
- Average candidates retained: 8.5
- Runtime: 3.48 seconds
- Estimated full-test runtime: ~16.75 hours

The ranking stage therefore did not provide the required computational reduction because RapidFuzz still had to score a large candidate set before selecting the top candidates.

### Validation Recall

Using a 3,000-entity validation sample:

- Top-50 recall: 0.5761
- Top-100 recall: 0.6163
- Top-200 recall: 0.6493

### Result

This approach was rejected for S1.

The candidate-ranking stage remained computationally expensive while substantially reducing candidate recall. Even Top-200 retained only 64.93% of true matches in the validation sample.

The V3 blocking stage remains frozen. Future candidate reduction should use a substantially cheaper retrieval mechanism rather than RapidFuzz ranking over the full V3 candidate set.

In [104]:
import time
import numpy as np
from scipy.sparse import vstack

def transform_in_batches(vectorizer, texts, batch_size=200000):
    texts = list(texts)
    parts = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        parts.append(vectorizer.transform(batch))
    return vstack(parts)

start = time.time()

M2 = transform_in_batches(vectorizer, test_source2_keyed_v3['norm_name'])
print(f"M2 done: {M2.shape}, {time.time()-start:.1f}s elapsed")

M3 = transform_in_batches(vectorizer, test_source3_keyed_v3['norm_name'])
print(f"M3 done: {M3.shape}, {time.time()-start:.1f}s elapsed")

id_to_pos_s2 = pd.Series(np.arange(len(test_source2_keyed_v3)), index=test_source2_keyed_v3['entity_id'].values)
id_to_pos_s3 = pd.Series(np.arange(len(test_source3_keyed_v3)), index=test_source3_keyed_v3['entity_id'].values)

print(f"Setup done in {time.time()-start:.1f}s total.")

MemoryError: 

In [ ]:
def retrieve_top_n_tfidf(name1_norm, candidate_ids, top_n=100):
    cand_arr = np.array(list(candidate_ids))
    if len(cand_arr) == 0:
        return cand_arr

    is_s2 = np.char.startswith(cand_arr, 'S2-')
    ids_s2, ids_s3 = cand_arr[is_s2], cand_arr[~is_s2]

    pos_s2 = id_to_pos_s2.reindex(ids_s2).dropna().values.astype(int)
    pos_s3 = id_to_pos_s3.reindex(ids_s3).dropna().values.astype(int)

    parts = []
    if len(pos_s2) > 0:
        parts.append(M2[pos_s2])
    if len(pos_s3) > 0:
        parts.append(M3[pos_s3])
    if not parts:
        return np.array([])

    sub_matrix = vstack(parts)
    combined_ids = np.concatenate([ids_s2[:len(pos_s2)], ids_s3[:len(pos_s3)]])

    query_vec = vectorizer.transform([name1_norm])
    sims = (sub_matrix @ query_vec.T).toarray().ravel()

    top_idx = np.argsort(-sims)[:top_n]
    return combined_ids[top_idx]


start = time.time()
retained_counts = []

for _, row in sample_test_100.iterrows():
    candidate_ids = get_candidates_test_v3(row['block_key'])
    top_ids = retrieve_top_n_tfidf(row['norm_name'], candidate_ids, top_n=100)
    retained_counts.append(len(top_ids))

elapsed = time.time() - start
print(f"TF-IDF retrieval time for 100 test entities: {elapsed:.2f} sec")
print(f"Avg candidates retained: {sum(retained_counts)/len(retained_counts):.1f}")
print(f"Estimated full-test retrieval time: {elapsed/100*len(test_source1_keyed_v3)/3600:.2f} hours")

## Test-Time Candidate Retrieval — Sparse TF-IDF

We investigated character-level TF-IDF retrieval as a first-stage candidate-ranking mechanism.

### Intended approach

Character n-gram TF-IDF vectors would be created for the full Source 2 and Source 3 test corpora, followed by sparse matrix multiplication to retrieve the most similar candidates before applying the final name/address matcher.

### Result

This approach was rejected before benchmarking retrieval.

Transforming the full Source 2 corpus into the sparse TF-IDF representation caused a `MemoryError` during vectorization. The full Source 2/Source 3 corpora are too large for the available local memory using this implementation.

No leaderboard inference was performed using this approach.

Future work may revisit indexed retrieval using a memory-efficient external or approximate-search system, but it is not suitable for the current local environment.

In [ ]:
"""import time
import numpy as np

def top_n_by_length(name1, cand_names, cand_ids, top_n=100):
    name1 = str(name1) if pd.notna(name1) else ""
    len1 = len(name1)
    if len(cand_ids) == 0:
        return []

    cand_lens = np.array([len(n) for n in cand_names])
    diff = np.abs(cand_lens - len1)
    order = np.argsort(diff)[:top_n]
    return [cand_ids[i] for i in order]


# --- TEST runtime benchmark (same 100 entities), reusing existing score_entity_candidates_test ---
start = time.time()
total_pairs = 0

for _, row in sample_test_100.iterrows():
    candidate_ids = list(get_candidates_test_v3(row['block_key']))
    valid_ids, cand_names, cand_addrs = [], [], []
    for cid in candidate_ids:
        lookup = source2_test_lookup if cid.startswith('S2-') else source3_test_lookup
        d = lookup.get(cid)
        if d is None:
            continue
        valid_ids.append(cid)
        cand_names.append(str(d['business_name']) if pd.notna(d['business_name']) else "")
        cand_addrs.append(str(d['business_address']) if pd.notna(d['business_address']) else "")

    top_ids = top_n_by_length(row['business_name'], cand_names, valid_ids, top_n=100)

    scored = score_entity_candidates_test(row['business_name'], row['business_address'], top_ids)
    total_pairs += len(scored)

elapsed = time.time() - start
print(f"Length-sort + existing scorer time for 100 entities: {elapsed:.2f} sec")
print(f"Estimated full-test time: {elapsed/100*len(test_source1_keyed_v3)/3600:.2f} hours")"""

KeyboardInterrupt: 

## Candidate Ranking — Length-Only Top-N

This experiment attempted to reduce the V3 candidate set by ranking candidates according to the absolute difference in business-name length and retaining the top N candidates before final name/address scoring.

### Result

The experiment was interrupted after more than 27 minutes on a 100-entity test sample. This was substantially slower than the earlier baseline and therefore was rejected.

No leaderboard inference was performed using this approach.

In [106]:
print("test_source2_keyed_v3:", len(test_source2_keyed_v3))
print("test_source3_keyed_v3:", len(test_source3_keyed_v3))
print("sample_test_100:", len(sample_test_100))

test_source2_keyed_v3: 4887273
test_source3_keyed_v3: 5082316
sample_test_100: 100


In [107]:
import time
import numpy as np

row = sample_test_100.iloc[0]

t0 = time.time()

candidate_ids = list(get_candidates_test_v3(row['block_key']))
print("1. Candidate retrieval:", time.time() - t0, "sec")
print("Candidates:", len(candidate_ids))

t1 = time.time()

valid_ids, cand_names, cand_addrs = [], [], []

for cid in candidate_ids:
    lookup = source2_test_lookup if cid.startswith('S2-') else source3_test_lookup
    d = lookup.get(cid)
    if d is None:
        continue

    valid_ids.append(cid)
    cand_names.append(
        str(d['business_name'])
        if pd.notna(d['business_name'])
        else ""
    )
    cand_addrs.append(
        str(d['business_address'])
        if pd.notna(d['business_address'])
        else ""
    )

print("2. Candidate lookup/build lists:", time.time() - t1, "sec")

t2 = time.time()

name1 = str(row['business_name']) if pd.notna(row['business_name']) else ""
len1 = len(name1)

cand_lens = np.array([len(n) for n in cand_names])
diff = np.abs(cand_lens - len1)
order = np.argsort(diff)[:100]
top_ids = [valid_ids[i] for i in order]

print("3. Length sorting:", time.time() - t2, "sec")

t3 = time.time()

scored = score_entity_candidates_test(
    row['business_name'],
    row['business_address'],
    top_ids
)

print("4. Final scorer:", time.time() - t3, "sec")
print("Total:", time.time() - t0, "sec")
print("Final candidates:", len(scored))

1. Candidate retrieval: 0.014676332473754883 sec
Candidates: 8255
2. Candidate lookup/build lists: 0.05489325523376465 sec
3. Length sorting: 0.0034096240997314453 sec
4. Final scorer: 0.0009021759033203125 sec
Total: 0.07487893104553223 sec
Final candidates: 100


In [110]:
def length_top_n_recall(sample_df, top_n=100):
    total_true = 0
    recovered_true = 0

    for _, row in sample_df.iterrows():
        true_ids = parse_ids(
            train_ground_truth.loc[
                train_ground_truth['source1_entity_id'] == row['entity_id'],
                'matched_entity_ids'
            ].iloc[0]
        )

        if not true_ids:
            continue

        candidate_ids = list(get_candidates_v3(row['block_key']))

        valid_ids = []
        cand_names = []

        for cid in candidate_ids:
            lookup = source2_lookup if cid.startswith('S2-') else source3_lookup
            d = lookup.get(cid)

            if d is None:
                continue

            valid_ids.append(cid)
            cand_names.append(
                str(d['business_name'])
                if pd.notna(d['business_name'])
                else ""
            )

        name1 = str(row['business_name']) if pd.notna(row['business_name']) else ""
        len1 = len(name1)

        cand_lens = np.array([len(n) for n in cand_names])
        diff = np.abs(cand_lens - len1)

        top_indices = np.argsort(diff)[:top_n]
        top_ids = {valid_ids[i] for i in top_indices}

        total_true += len(true_ids)
        recovered_true += len(true_ids & top_ids)

    return recovered_true / total_true if total_true else 0

In [111]:
validation_sample_3000 = train_source1_keyed_v3.sample(
    n=3000,
    random_state=42
).reset_index(drop=True)

recall_100 = length_top_n_recall(validation_sample_3000, top_n=100)

print("Top-100 length candidate recall:", recall_100)

NameError: name 'train_source1_keyed_v3' is not defined

In [112]:
print('train_source1' in globals())
print('train_source1_keyed_v3' in globals())

True
False


In [ ]:
train_source1_keyed_v3 = build_block_key(train_source1)

## Final Inference Pipeline — Production Run

At this stage, the V3 blocking strategy and threshold-75 RapidFuzz matcher have been selected as the baseline pipeline.

Previous optimization experiments were benchmarked and rejected where they did not provide sufficient speed or validation recall.

The production pipeline will:
1. Rebuild the required test-time lookup structures.
2. Generate candidates using the frozen V3 blocking strategy.
3. Apply a lightweight candidate reduction step.
4. Apply the existing name/address RapidFuzz scorer.
5. Keep predictions meeting the threshold-75 decision boundary.
6. Generate `matching_results.tsv` and `candidate_pairs.tsv`.

In [1]:
import pandas as pd
import numpy as np
import time
from rapidfuzz import process, fuzz

PROJECT_DIR = r"E:\Amazon-ML-Challenge-2026"
TEST_DIR = rf"{PROJECT_DIR}\data\student_resource\dataset\test"

test_source1 = pd.read_csv(
    rf"{TEST_DIR}\test_source1.tsv",
    sep="\t"
)

test_source2 = pd.read_csv(
    rf"{TEST_DIR}\test_source2.tsv",
    sep="\t"
)

test_source3 = pd.read_csv(
    rf"{TEST_DIR}\test_source3.tsv",
    sep="\t"
)

print("Loaded:")
print("Source 1:", len(test_source1))
print("Source 2:", len(test_source2))
print("Source 3:", len(test_source3))

Loaded:
Source 1: 1732544
Source 2: 4887273
Source 3: 5082316


In [2]:
import re
import unicodedata

def normalize_name(name):
    if pd.isna(name):
        return ""

    name = name.lower()

    name = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in name
    )

    stopwords = {
        'inc','incorporated','ltd','limited','pvt','private',
        'corp','corporation','llc','llp','co','company',
        'dr','mr','mrs','ms','smt','sri','shri','prof','m','s',
        'the'
    }

    tokens = [t for t in name.split() if t not in stopwords]

    return ' '.join(tokens).strip()


def first_token(normalized_name):
    tokens = normalized_name.split()
    return tokens[0] if tokens else ""


def build_block_key(df):
    df = df.copy()
    df['norm_name'] = df['business_name'].apply(normalize_name)
    df['block_key'] = (
        df['country'] + '||' +
        df['norm_name'].apply(first_token)
    )
    return df

print("Helper functions restored.")

Helper functions restored.


In [3]:
test_source1_keyed_v3 = build_block_key(test_source1)
test_source2_keyed_v3 = build_block_key(test_source2)
test_source3_keyed_v3 = build_block_key(test_source3)

index_source2_test_v3 = (
    test_source2_keyed_v3
    .groupby('block_key')['entity_id']
    .apply(list)
    .to_dict()
)

index_source3_test_v3 = (
    test_source3_keyed_v3
    .groupby('block_key')['entity_id']
    .apply(list)
    .to_dict()
)

print("V3 blocking structures rebuilt.")
print("S2 block keys:", len(index_source2_test_v3))
print("S3 block keys:", len(index_source3_test_v3))

V3 blocking structures rebuilt.
S2 block keys: 343967
S3 block keys: 384817


In [4]:
# Build fast integer-indexed candidate structures

test_source2_keyed_v3 = test_source2_keyed_v3.reset_index(drop=True)
test_source3_keyed_v3 = test_source3_keyed_v3.reset_index(drop=True)

s2_names = (
    test_source2_keyed_v3['business_name']
    .fillna('')
    .astype(str)
    .to_numpy()
)
s2_addrs = (
    test_source2_keyed_v3['business_address']
    .fillna('')
    .astype(str)
    .to_numpy()
)
s2_ids = test_source2_keyed_v3['entity_id'].to_numpy()

s3_names = (
    test_source3_keyed_v3['business_name']
    .fillna('')
    .astype(str)
    .to_numpy()
)
s3_addrs = (
    test_source3_keyed_v3['business_address']
    .fillna('')
    .astype(str)
    .to_numpy()
)
s3_ids = test_source3_keyed_v3['entity_id'].to_numpy()

index_source2_test_int = test_source2_keyed_v3.groupby('block_key').indices
index_source3_test_int = test_source3_keyed_v3.groupby('block_key').indices

print("Integer-index structures ready.")
print("S2 rows:", len(s2_ids))
print("S3 rows:", len(s3_ids))

Integer-index structures ready.
S2 rows: 4887273
S3 rows: 5082316


In [5]:
def get_candidate_arrays_fast(block_key):
    s2_idx = index_source2_test_int.get(block_key, np.array([], dtype=np.int64))
    s3_idx = index_source3_test_int.get(block_key, np.array([], dtype=np.int64))

    return (
        np.asarray(s2_idx, dtype=np.int64),
        np.asarray(s3_idx, dtype=np.int64)
    )


def score_top_length_candidates(
    name1,
    addr1,
    s2_idx,
    s3_idx,
    top_n=100
):
    if len(s2_idx) + len(s3_idx) == 0:
        return []

    names = np.concatenate([
        s2_names[s2_idx],
        s3_names[s3_idx]
    ]).astype(str)

    ids = np.concatenate([
        s2_ids[s2_idx],
        s3_ids[s3_idx]
    ])

    addrs = np.concatenate([
        s2_addrs[s2_idx],
        s3_addrs[s3_idx]
    ]).astype(str)

    name1 = str(name1) if pd.notna(name1) else ""
    addr1 = str(addr1) if pd.notna(addr1) else ""

    len1 = len(name1)

    # Convert explicitly to Unicode before np.char.str_len
    candidate_lengths = np.char.str_len(
        names.astype("U")
    )

    differences = np.abs(candidate_lengths - len1)

    k = min(top_n, len(ids))

    top_idx = np.argpartition(
        differences,
        k - 1
    )[:k]

    top_names = names[top_idx].tolist()
    top_addrs = addrs[top_idx].tolist()
    top_ids = ids[top_idx]

    name_scores = process.cdist(
        [name1],
        top_names,
        scorer=fuzz.token_sort_ratio
    )[0]

    addr_scores = process.cdist(
        [addr1],
        top_addrs,
        scorer=fuzz.token_sort_ratio
    )[0]

    combined = 0.6 * name_scores + 0.4 * addr_scores

    matched = top_ids[combined >= 75]

    return matched.tolist()

In [13]:
sample_test_10 = test_source1_keyed_v3.sample(
    n=10,
    random_state=42
).reset_index(drop=True)

start = time.time()

results_10 = []

for _, row in sample_test_10.iterrows():

    s2_idx, s3_idx = get_candidate_arrays_fast(
        row['block_key']
    )

    matches = score_top_length_candidates(
        row['business_name'],
        row['business_address'],
        s2_idx,
        s3_idx,
        top_n=100
    )

    results_10.append(matches)

elapsed = time.time() - start

total_candidates = 0

for _, row in sample_test_10.iterrows():
    s2_idx, s3_idx = get_candidate_arrays_fast(row['block_key'])
    total_candidates += len(s2_idx) + len(s3_idx)

print("10-entity benchmark")
print("--------------------")
print("Runtime:", elapsed, "seconds")
print("Candidate pairs:", total_candidates)
print("Average candidates/entity:", total_candidates / 10)
print("Matches produced:", sum(len(x) for x in results_10))
print("Estimated full-test runtime:",
      elapsed / 10 * len(test_source1) / 3600,
      "hours")

10-entity benchmark
--------------------
Runtime: 0.42816925048828125 seconds
Candidate pairs: 59354
Average candidates/entity: 5935.4
Matches produced: 10
Estimated full-test runtime: 20.606168497721356 hours


In [14]:
# Benchmark: length filtering BEFORE constructing candidate strings

def score_length_filtered(
    name1,
    addr1,
    s2_idx,
    s3_idx,
    length_tolerance=0.30,
    top_n=100
):
    name1 = str(name1) if pd.notna(name1) else ""
    addr1 = str(addr1) if pd.notna(addr1) else ""

    len1 = len(name1)

    if len1 == 0:
        return []

    # Work directly with integer-indexed arrays
    s2_lengths = np.char.str_len(s2_names[s2_idx].astype("U"))
    s3_lengths = np.char.str_len(s3_names[s3_idx].astype("U"))

    s2_mask = (
        np.abs(s2_lengths - len1) / max(len1, 1)
        <= length_tolerance
    )

    s3_mask = (
        np.abs(s3_lengths - len1) / max(len1, 1)
        <= length_tolerance
    )

    filtered_s2 = s2_idx[s2_mask]
    filtered_s3 = s3_idx[s3_mask]

    # Now construct strings only for survivors
    names = np.concatenate([
        s2_names[filtered_s2],
        s3_names[filtered_s3]
    ]).astype(str)

    ids = np.concatenate([
        s2_ids[filtered_s2],
        s3_ids[filtered_s3]
    ])

    addrs = np.concatenate([
        s2_addrs[filtered_s2],
        s3_addrs[filtered_s3]
    ]).astype(str)

    if len(ids) == 0:
        return []

    # Keep at most top_n by exact length difference
    differences = np.abs(
        np.char.str_len(names.astype("U")) - len1
    )

    k = min(top_n, len(ids))

    top_idx = np.argpartition(
        differences,
        k - 1
    )[:k]

    top_names = names[top_idx].tolist()
    top_addrs = addrs[top_idx].tolist()
    top_ids = ids[top_idx]

    name_scores = process.cdist(
        [name1],
        top_names,
        scorer=fuzz.token_sort_ratio
    )[0]

    addr_scores = process.cdist(
        [addr1],
        top_addrs,
        scorer=fuzz.token_sort_ratio
    )[0]

    combined = 0.6 * name_scores + 0.4 * addr_scores

    return top_ids[combined >= 75].tolist()

In [15]:
start = time.time()

results_10_filtered = []
filtered_counts = []

for _, row in sample_test_10.iterrows():

    s2_idx, s3_idx = get_candidate_arrays_fast(
        row['block_key']
    )

    before = len(s2_idx) + len(s3_idx)

    matches = score_length_filtered(
        row['business_name'],
        row['business_address'],
        s2_idx,
        s3_idx,
        length_tolerance=0.30,
        top_n=100
    )

    results_10_filtered.append(matches)

elapsed = time.time() - start

print("10-entity length-filter benchmark")
print("----------------------------------")
print("Runtime:", elapsed, "seconds")
print("Original candidate pairs:", total_candidates)
print("Average original candidates:", total_candidates / 10)
print("Matches produced:", sum(len(x) for x in results_10_filtered))

10-entity length-filter benchmark
----------------------------------
Runtime: 0.4690275192260742 seconds
Original candidate pairs: 59354
Average original candidates: 5935.4
Matches produced: 10


## Production Candidate Resolution — Exact Normalized-Name Matching

Because full V3 candidate scoring is too expensive for the complete test set, we introduce a deterministic fast path.

For each Source 1 entity, Source 2 and Source 3 records are first grouped by the existing V3 block key and normalized business name.

Records with the same country, first normalized-name token, and complete normalized business name can be resolved without pairwise RapidFuzz scoring.

This fast path is intended to reduce the number of entities requiring expensive approximate matching while preserving the frozen V3 blocking strategy.

In [ ]:
# Build exact normalized-name indexes inside each V3 block.
# This allows exact normalized-name matches without RapidFuzz.

s2_exact_index = (
    test_source2_keyed_v3
    .groupby(['block_key', 'norm_name'])['entity_id']
    .apply(list)
    .to_dict()
)

s3_exact_index = (
    test_source3_keyed_v3
    .groupby(['block_key', 'norm_name'])['entity_id']
    .apply(list)
    .to_dict()
)

print("Exact normalized-name indexes built.")
print("S2 exact groups:", len(s2_exact_index))
print("S3 exact groups:", len(s3_exact_index))

In [ ]:
start = time.time()

exact_match_counts = []
exact_matches = []

for _, row in test_source1_keyed_v3.iterrows():
    block_key = row['block_key']
    norm_name = row['norm_name']

    s2_matches = s2_exact_index.get((block_key, norm_name), [])
    s3_matches = s3_exact_index.get((block_key, norm_name), [])

    matches = s2_matches + s3_matches

    exact_matches.append(matches)
    exact_match_counts.append(len(matches))

elapsed = time.time() - start

exact_match_counts = np.array(exact_match_counts)

print("Exact normalized-name coverage")
print("--------------------------------")
print("Runtime:", elapsed, "seconds")
print("Total Source 1 entities:", len(test_source1_keyed_v3))
print(
    "Entities with >=1 exact match:",
    np.sum(exact_match_counts > 0)
)
print(
    "Coverage:",
    np.mean(exact_match_counts > 0)
)
print(
    "Entities with >1 exact match:",
    np.sum(exact_match_counts > 1)
)
print(
    "Maximum exact matches for one entity:",
    exact_match_counts.max()
)

Exact normalized-name coverage
--------------------------------
Runtime: 100.5589771270752 seconds
Total Source 1 entities: 1732544
Entities with >=1 exact match: 1560169
Coverage: 0.9005075772967382
Entities with >1 exact match: 1281872
Maximum exact matches for one entity: 758


## Validation — Exact Normalized-Name Matching

We now validate the deterministic exact normalized-name rule on held-out training data.

For each validation Source 1 entity, candidates are generated using the same
`(country + first normalized-name token, normalized full name)` key used for
the test set.

This measures whether exact normalized-name matching provides sufficient
precision and F0.5 performance to serve as the primary matching rule.

In [21]:
import pandas as pd
import numpy as np
import time

BASE_DIR = r"E:\Amazon-ML-Challenge-2026\data\student_resource"

print("Loading training data...")

train_source1 = pd.read_csv(
    rf"{BASE_DIR}\dataset\train\train_source1.tsv",
    sep="\t"
)

train_source2 = pd.read_csv(
    rf"{BASE_DIR}\dataset\train\train_source2.tsv",
    sep="\t"
)

train_source3 = pd.read_csv(
    rf"{BASE_DIR}\dataset\train\train_source3.tsv",
    sep="\t"
)

train_ground_truth = pd.read_csv(
    rf"{BASE_DIR}\dataset\train\train_ground_truth.tsv",
    sep="\t"
)

print("S1:", train_source1.shape)
print("S2:", train_source2.shape)
print("S3:", train_source3.shape)
print("GT:", train_ground_truth.shape)

Loading training data...
S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)
GT: (2206821, 2)


In [22]:
print("Building V3 normalized/block keys...")

train_source1_keyed_v3 = build_block_key(train_source1)
train_source2_keyed_v3 = build_block_key(train_source2)
train_source3_keyed_v3 = build_block_key(train_source3)

print("Done.")
print("S1:", train_source1_keyed_v3.shape)
print("S2:", train_source2_keyed_v3.shape)
print("S3:", train_source3_keyed_v3.shape)

Building V3 normalized/block keys...
Done.
S1: (2206821, 6)
S2: (5034616, 6)
S3: (5285603, 6)


In [23]:
# Rebuild training V3 keyed data after kernel restart

train_source1_keyed_v3 = build_block_key(train_source1)
train_source2_keyed_v3 = build_block_key(train_source2)
train_source3_keyed_v3 = build_block_key(train_source3)

print("Training V3 data rebuilt")
print("S1:", train_source1_keyed_v3.shape)
print("S2:", train_source2_keyed_v3.shape)
print("S3:", train_source3_keyed_v3.shape)

Training V3 data rebuilt
S1: (2206821, 6)
S2: (5034616, 6)
S3: (5285603, 6)


In [24]:
# Build exact normalized-name indexes from training data

train_s2_exact_index = (
    train_source2_keyed_v3
    .groupby(['block_key', 'norm_name'])['entity_id']
    .apply(list)
    .to_dict()
)

train_s3_exact_index = (
    train_source3_keyed_v3
    .groupby(['block_key', 'norm_name'])['entity_id']
    .apply(list)
    .to_dict())

print("Training exact indexes built")
print("S2 groups:", len(train_s2_exact_index))
print("S3 groups:", len(train_s3_exact_index))

Training exact indexes built
S2 groups: 3368060
S3 groups: 3595824


In [26]:
def parse_ids(cell):
    if pd.isna(cell) or cell == '':
        return set()
    return set(cell.split(','))

In [28]:
def f_beta_score(true_set, pred_set, beta=0.5):
    if len(true_set) == 0:
        return 1.0 if len(pred_set) == 0 else 0.0

    if len(pred_set) == 0:
        return 0.0

    tp = len(true_set & pred_set)

    precision = tp / len(pred_set)
    recall = tp / len(true_set)

    if precision == 0 and recall == 0:
        return 0.0

    beta_sq = beta ** 2

    return (
        (1 + beta_sq) * precision * recall
        / (beta_sq * precision + recall)
    )


# Calculate F0.5 from the already-computed results
scores = [
    f_beta_score(true_set, predicted, beta=0.5)
    for _, true_set, predicted in exact_val_results
]

tp = 0
pred_total = 0
true_total = 0

for _, true_set, predicted in exact_val_results:
    tp += len(true_set & predicted)
    pred_total += len(predicted)
    true_total += len(true_set)

precision = tp / pred_total if pred_total else 0
recall = tp / true_total if true_total else 0

coverage = np.mean([
    len(predicted) > 0
    for _, _, predicted in exact_val_results
])

avg_predictions = np.mean([
    len(predicted)
    for _, _, predicted in exact_val_results
])

print("Exact normalized-name validation")
print("----------------------------------")
print("F0.5:", round(np.mean(scores), 6))
print("Precision:", round(precision, 6))
print("Recall:", round(recall, 6))
print("Coverage:", round(coverage, 6))
print("Average predicted matches/entity:", round(avg_predictions, 3))

Exact normalized-name validation
----------------------------------
F0.5: 0.416253
Precision: 0.042232
Recall: 0.448223
Coverage: 0.913
Average predicted matches/entity: 36.829


## Exact Name + Address Disambiguation

Exact normalized-name matching has high candidate coverage but very low
precision because many businesses share the same normalized name.

We therefore use normalized address agreement as a deterministic
disambiguation signal before considering a candidate a final match.

This keeps the existing V3 blocking strategy unchanged.

In [30]:
def normalize_address(address):
    if pd.isna(address):
        return ""

    address = str(address).lower()

    address = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in address
    )

    return ' '.join(address.split()).strip()

In [31]:
# Add normalized addresses to the TRAINING data

train_source1_keyed_v3['norm_address'] = (
    train_source1_keyed_v3['business_address']
    .apply(normalize_address)
)

train_source2_keyed_v3['norm_address'] = (
    train_source2_keyed_v3['business_address']
    .apply(normalize_address)
)

train_source3_keyed_v3['norm_address'] = (
    train_source3_keyed_v3['business_address']
    .apply(normalize_address)
)

print("Normalized addresses created.")

Normalized addresses created.


In [32]:
train_s2_name_addr_index = (
    train_source2_keyed_v3[
        train_source2_keyed_v3['norm_address'] != ""
    ]
    .groupby(['block_key', 'norm_name', 'norm_address'])['entity_id']
    .apply(list)
    .to_dict()
)

train_s3_name_addr_index = (
    train_source3_keyed_v3[
        train_source3_keyed_v3['norm_address'] != ""
    ]
    .groupby(['block_key', 'norm_name', 'norm_address'])['entity_id']
    .apply(list)
    .to_dict()
)

print("S2 name+address groups:", len(train_s2_name_addr_index))
print("S3 name+address groups:", len(train_s3_name_addr_index))

S2 name+address groups: 4711722
S3 name+address groups: 4999307


In [34]:
val_sample['norm_address'] = (
    val_sample['business_address']
    .apply(normalize_address)
)

print("Validation sample updated:", val_sample.shape)

Validation sample updated: (3000, 7)


In [35]:
name_addr_val_results = []

start = time.time()

for _, row in val_sample.iterrows():

    key = (
        row['block_key'],
        row['norm_name'],
        row['norm_address']
    )

    s2_matches = train_s2_name_addr_index.get(key, [])
    s3_matches = train_s3_name_addr_index.get(key, [])

    predicted = set(s2_matches + s3_matches)

    true_cell = train_ground_truth_lookup.get(row['entity_id'], "")
    true_set = parse_ids(true_cell)

    name_addr_val_results.append(
        (row['entity_id'], true_set, predicted)
    )

elapsed = time.time() - start

scores = [
    f_beta_score(true_set, predicted, beta=0.5)
    for _, true_set, predicted in name_addr_val_results
]

tp = sum(
    len(true_set & predicted)
    for _, true_set, predicted in name_addr_val_results
)

pred_total = sum(
    len(predicted)
    for _, _, predicted in name_addr_val_results
)

true_total = sum(
    len(true_set)
    for _, true_set, _ in name_addr_val_results
)

precision = tp / pred_total if pred_total else 0
recall = tp / true_total if true_total else 0

coverage = np.mean([
    len(predicted) > 0
    for _, _, predicted in name_addr_val_results
])

avg_predictions = np.mean([
    len(predicted)
    for _, _, predicted in name_addr_val_results
])

print("Exact normalized-name + address validation")
print("--------------------------------------------")
print("Runtime:", round(elapsed, 2), "seconds")
print("F0.5:", round(np.mean(scores), 6))
print("Precision:", round(precision, 6))
print("Recall:", round(recall, 6))
print("Coverage:", round(coverage, 6))
print("Average predicted matches/entity:", round(avg_predictions, 3))

Exact normalized-name + address validation
--------------------------------------------
Runtime: 104.81 seconds
F0.5: 0.110244
Precision: 1.0
Recall: 0.027858
Coverage: 0.083333
Average predicted matches/entity: 0.097


## Candidate Reduction — Name Block + Address Token Overlap

V3 blocking is frozen.

Exact normalized-name matching provides high candidate coverage but produces
large ambiguous candidate groups. Exact address matching is too strict.

We therefore use token-level address overlap as a lightweight candidate
reduction stage before the final similarity model.

Only candidates with sufficient address-token overlap will proceed to
expensive similarity scoring.

In [36]:
import re

def address_tokens(address):
    if pd.isna(address):
        return set()

    text = normalize_address(address)

    # Keep reasonably informative tokens
    tokens = {
        token
        for token in text.split()
        if len(token) >= 2
    }

    return tokens

In [37]:
# Create address token sets for the TEST sources

test_source1_keyed_v3['address_tokens'] = (
    test_source1_keyed_v3['business_address']
    .apply(address_tokens)
)

test_source2_keyed_v3['address_tokens'] = (
    test_source2_keyed_v3['business_address']
    .apply(address_tokens)
)

test_source3_keyed_v3['address_tokens'] = (
    test_source3_keyed_v3['business_address']
    .apply(address_tokens)
)

print("Address token sets created.")

KeyboardInterrupt: 

> **Experiment note:** Full-dataset Python-level address-token construction was interrupted because it was too slow and resource-intensive on the available environment. A smaller benchmark will be used before attempting any scaled version.

## Small Benchmark — Address Token Candidate Reduction

The full-dataset Python-level address-token construction was too slow for the
available environment, so we first benchmark the idea on 100 Source 1 test
entities.

No full test-set transformation is performed until the benchmark demonstrates
that the filtering approach is computationally practical.

In [38]:
# Select only 100 test Source 1 entities
benchmark_s1 = test_source1_keyed_v3.head(100).copy()

# Get the V3 name-block candidates
benchmark_results = []

start = time.time()

for _, row in benchmark_s1.iterrows():

    block_key = row['block_key']

    # Candidates from S2 and S3 within the V3 block
    s2_candidates = test_source2_keyed_v3[
        test_source2_keyed_v3['block_key'] == block_key
    ]

    s3_candidates = test_source3_keyed_v3[
        test_source3_keyed_v3['block_key'] == block_key
    ]

    original_count = len(s2_candidates) + len(s3_candidates)

    # Simple address tokenization ONLY for these candidates
    source1_tokens = address_tokens(row['business_address'])

    filtered_s2 = []
    for _, cand in s2_candidates.iterrows():
        cand_tokens = address_tokens(cand['business_address'])

        if source1_tokens and cand_tokens:
            overlap = len(source1_tokens & cand_tokens)
            if overlap >= 1:
                filtered_s2.append(cand['entity_id'])

    filtered_s3 = []
    for _, cand in s3_candidates.iterrows():
        cand_tokens = address_tokens(cand['business_address'])

        if source1_tokens and cand_tokens:
            overlap = len(source1_tokens & cand_tokens)
            if overlap >= 1:
                filtered_s3.append(cand['entity_id'])

    filtered_count = len(filtered_s2) + len(filtered_s3)

    benchmark_results.append(
        (row['entity_id'], original_count, filtered_count)
    )

elapsed = time.time() - start

original_total = sum(x[1] for x in benchmark_results)
filtered_total = sum(x[2] for x in benchmark_results)

print("100-entity address filter benchmark")
print("------------------------------------")
print("Runtime:", round(elapsed, 2), "seconds")
print("Original candidates:", original_total)
print("Filtered candidates:", filtered_total)
print("Average original/entity:", round(original_total / 100, 2))
print("Average filtered/entity:", round(filtered_total / 100, 2))
print(
    "Candidate reduction:",
    round(1 - filtered_total / original_total, 4)
    if original_total else 0
)

KeyboardInterrupt: 

## End of Day 2 — Experiment Status

The following approaches were evaluated:

- V3 blocking: country + first normalized-name token.
- Exact normalized-name matching: high coverage but very low precision.
- Exact normalized-name + exact address: very high precision but insufficient recall.
- RapidFuzz name/address similarity: threshold 75 gave the strongest validated F0.5 result among tested approaches.
- Full-scale Python-level address-token filtering was attempted but was too computationally expensive and was abandoned.

The next session will focus on turning the best validated approach into a computationally practical production pipeline and generating the required submission files.

No further experiments were run after the computationally expensive address-token benchmark was interrupted.

In [6]:
import pandas as pd
import numpy as np
import time

PROJECT_DIR = r"E:\Amazon-ML-Challenge-2026"
TRAIN_DIR = rf"{PROJECT_DIR}\data\student_resource\dataset\train"

t0 = time.time()
train_source1 = pd.read_csv(rf"{TRAIN_DIR}\train_source1.tsv", sep="\t")
train_source2 = pd.read_csv(rf"{TRAIN_DIR}\train_source2.tsv", sep="\t")
train_source3 = pd.read_csv(rf"{TRAIN_DIR}\train_source3.tsv", sep="\t")
train_ground_truth = pd.read_csv(rf"{TRAIN_DIR}\train_ground_truth.tsv", sep="\t")
print(f"Train data loaded in {time.time()-t0:.1f}s")

# Reuse normalize_name / first_token / build_block_key already defined in cell 128
t0 = time.time()
train_source2_keyed_v3 = build_block_key(train_source2)
print(f"train_source2 keyed in {time.time()-t0:.1f}s")

t0 = time.time()
train_source3_keyed_v3 = build_block_key(train_source3)
print(f"train_source3 keyed in {time.time()-t0:.1f}s")

# Integer-indexed arrays (same pattern as cell 130, applied to train)
train_s2_names = train_source2_keyed_v3['business_name'].fillna('').astype(str).to_numpy()
train_s2_ids = train_source2_keyed_v3['entity_id'].to_numpy()

train_s3_names = train_source3_keyed_v3['business_name'].fillna('').astype(str).to_numpy()
train_s3_ids = train_source3_keyed_v3['entity_id'].to_numpy()

index_source2_train_int = train_source2_keyed_v3.groupby('block_key').indices
index_source3_train_int = train_source3_keyed_v3.groupby('block_key').indices

empty_idx = np.array([], dtype=int)

def get_candidate_arrays_fast_train(block_key):
    idx2 = index_source2_train_int.get(block_key, empty_idx)
    idx3 = index_source3_train_int.get(block_key, empty_idx)
    return idx2, idx3

# Ground truth lookup + parser
gt_lookup = train_ground_truth.set_index('source1_entity_id')['matched_entity_ids'].to_dict()

def parse_ids(cell):
    if pd.isna(cell) or cell == '':
        return set()
    return set(cell.split(','))

# Small held-out sample from train_source1
sample_s1 = train_source1.sample(n=3000, random_state=42).reset_index(drop=True)
sample_s1 = build_block_key(sample_s1)  # cheap: only 3000 rows

print("Setup complete.")

Train data loaded in 172.5s
train_source2 keyed in 49.3s
train_source3 keyed in 45.5s
Setup complete.


In [7]:
N_SMALL = 100
N_LARGE = 500

baseline_recalls = []
recalls_n_small = []
recalls_n_large = []

cand_counts_before = []
cand_counts_after_small = []
cand_counts_after_large = []

skipped_no_match = 0

t0 = time.time()

for row in sample_s1.itertuples(index=False):
    entity_id = row.entity_id
    name1 = row.business_name if pd.notna(row.business_name) else ""
    len1 = len(str(name1))
    block_key = row.block_key

    true_set = parse_ids(gt_lookup.get(entity_id))

    if not true_set:
        skipped_no_match += 1
        continue

    idx2, idx3 = get_candidate_arrays_fast_train(block_key)

    cand_ids = np.concatenate([
        train_s2_ids[idx2],
        train_s3_ids[idx3]
    ])

    cand_names = np.concatenate([
        train_s2_names[idx2],
        train_s3_names[idx3]
    ])

    n_total = len(cand_ids)
    cand_counts_before.append(n_total)

    # -----------------------------
    # V3 baseline recall
    # -----------------------------
    true_ids = np.array(list(true_set), dtype=object)

    baseline_hits = np.isin(true_ids, cand_ids)
    baseline_recalls.append(baseline_hits.mean())

    if n_total == 0:
        recalls_n_small.append(0.0)
        recalls_n_large.append(0.0)
        cand_counts_after_small.append(0)
        cand_counts_after_large.append(0)
        continue

    # -----------------------------
    # Cheap name-length reduction
    # -----------------------------
    cand_lens = np.char.str_len(cand_names.astype("U"))
    diff = np.abs(cand_lens - len1)

    for N, recall_list, count_list in [
        (N_SMALL, recalls_n_small, cand_counts_after_small),
        (N_LARGE, recalls_n_large, cand_counts_after_large),
    ]:
        k = min(N, n_total)

        top_idx = np.argpartition(diff, k - 1)[:k]

        top_ids = cand_ids[top_idx]

        hits = np.isin(true_ids, top_ids)
        recall_list.append(hits.mean())

        count_list.append(k)

elapsed = time.time() - t0

print("===== TOP-N CANDIDATE REDUCTION BENCHMARK =====")
print()
print("V3 blocking baseline recall:",
      round(np.mean(baseline_recalls), 4))

print(f"Top-{N_SMALL} recall:",
      round(np.mean(recalls_n_small), 4))

print(f"Top-{N_LARGE} recall:",
      round(np.mean(recalls_n_large), 4))

print()

print("Avg candidates before reduction:",
      round(np.mean(cand_counts_before), 1))

print(f"Avg candidates after top-{N_SMALL}:",
      round(np.mean(cand_counts_after_small), 1))

print(f"Avg candidates after top-{N_LARGE}:",
      round(np.mean(cand_counts_after_large), 1))

print()

print("Entities evaluated:", len(baseline_recalls))
print("Entities skipped (no GT match):", skipped_no_match)
print(f"Benchmark runtime: {elapsed:.2f}s")

===== TOP-N CANDIDATE REDUCTION BENCHMARK =====

V3 blocking baseline recall: 0.7893
Top-100 recall: 0.3953
Top-500 recall: 0.532

Avg candidates before reduction: 5910.2
Avg candidates after top-100: 88.1
Avg candidates after top-500: 342.8

Entities evaluated: 2820
Entities skipped (no GT match): 180
Benchmark runtime: 64.86s


## Experiment: Name-Token Jaccard Candidate Ranking

Test whether name-token Jaccard similarity can cheaply reduce the V3
candidate pool while retaining true-match recall.

Compare:
- V3 blocking baseline
- Top-100 candidates ranked by name-token Jaccard
- Top-500 candidates ranked by name-token Jaccard

This is a validation benchmark only. No production inference is performed.

In [8]:
import time

# --- Cheap setup: reuse already-computed norm_name columns, just as numpy arrays ---
train_s2_norm_names = train_source2_keyed_v3['norm_name'].to_numpy()
train_s3_norm_names = train_source3_keyed_v3['norm_name'].to_numpy()

N_SMALL = 100
N_LARGE = 500

baseline_recalls = []
recalls_n_small = []
recalls_n_large = []

cand_counts_before = []
cand_counts_after_small = []
cand_counts_after_large = []

skipped_no_match = 0

t0 = time.time()

for row in sample_s1.itertuples(index=False):
    entity_id = row.entity_id
    block_key = row.block_key
    name1_tokens = set(row.norm_name.split()) if row.norm_name else set()

    true_set = parse_ids(gt_lookup.get(entity_id))
    if not true_set:
        skipped_no_match += 1
        continue

    idx2, idx3 = get_candidate_arrays_fast_train(block_key)

    cand_ids = np.concatenate([train_s2_ids[idx2], train_s3_ids[idx3]])
    cand_norm_names = np.concatenate([train_s2_norm_names[idx2], train_s3_norm_names[idx3]])

    n_total = len(cand_ids)
    cand_counts_before.append(n_total)

    true_ids = np.array(list(true_set), dtype=object)

    # Baseline (V3 block) recall — recomputed here for a clean same-loop comparison
    baseline_recalls.append(np.isin(true_ids, cand_ids).mean())

    if n_total == 0:
        recalls_n_small.append(0.0)
        recalls_n_large.append(0.0)
        cand_counts_after_small.append(0)
        cand_counts_after_large.append(0)
        continue

    # --- Name-token Jaccard scoring ---
    jaccard_scores = np.empty(n_total, dtype=np.float64)
    for i, cname in enumerate(cand_norm_names):
        cand_tokens = set(cname.split()) if cname else set()
        union_size = len(name1_tokens | cand_tokens)
        jaccard_scores[i] = (
            len(name1_tokens & cand_tokens) / union_size if union_size else 0.0
        )

    order = np.argsort(-jaccard_scores)  # descending

    for N, recall_list, count_list in [
        (N_SMALL, recalls_n_small, cand_counts_after_small),
        (N_LARGE, recalls_n_large, cand_counts_after_large),
    ]:
        k = min(N, n_total)
        top_ids = cand_ids[order[:k]]
        recall_list.append(np.isin(true_ids, top_ids).mean())
        count_list.append(k)

elapsed = time.time() - t0

print("===== NAME-TOKEN JACCARD RANKING BENCHMARK =====")
print()
print("V3 blocking baseline recall:", round(np.mean(baseline_recalls), 4))
print(f"Top-{N_SMALL} Jaccard recall:", round(np.mean(recalls_n_small), 4))
print(f"Top-{N_LARGE} Jaccard recall:", round(np.mean(recalls_n_large), 4))
print()
print("Avg candidates before reduction:", round(np.mean(cand_counts_before), 1))
print(f"Avg candidates after top-{N_SMALL}:", round(np.mean(cand_counts_after_small), 1))
print(f"Avg candidates after top-{N_LARGE}:", round(np.mean(cand_counts_after_large), 1))
print()
print("Entities evaluated:", len(baseline_recalls))
print("Entities skipped (no GT match):", skipped_no_match)
print(f"Benchmark runtime: {elapsed:.2f}s")

===== NAME-TOKEN JACCARD RANKING BENCHMARK =====

V3 blocking baseline recall: 0.7893
Top-100 Jaccard recall: 0.6646
Top-500 Jaccard recall: 0.7274

Avg candidates before reduction: 5910.2
Avg candidates after top-100: 88.1
Avg candidates after top-500: 342.8

Entities evaluated: 2820
Entities skipped (no GT match): 180
Benchmark runtime: 80.13s


## Experiment: Vectorized Name-Token Jaccard Ranking

Optimize the validated name-token Jaccard candidate-ranking method using
a compact sparse token-presence matrix over only the candidate universe
touched by the validation sample.

The metric and tokenization remain equivalent to the previous Jaccard
experiment. This benchmark measures whether the same ranking can be
computed efficiently enough for production-scale inference.

Compare:
- V3 blocking baseline
- Top-100 Jaccard
- Top-500 Jaccard

No production inference is performed.

In [9]:
import time
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# =========================================================
# Phase 1 — cheap: find the unique candidate universe touched
#           by this sample's unique block keys (no per-candidate work)
# =========================================================
t0 = time.time()

unique_block_keys = sample_s1['block_key'].unique()

s2_pos_list, s3_pos_list = [], []
for bk in unique_block_keys:
    idx2, idx3 = get_candidate_arrays_fast_train(bk)
    if len(idx2):
        s2_pos_list.append(idx2)
    if len(idx3):
        s3_pos_list.append(idx3)

unique_s2_pos = np.unique(np.concatenate(s2_pos_list)) if s2_pos_list else np.array([], dtype=int)
unique_s3_pos = np.unique(np.concatenate(s3_pos_list)) if s3_pos_list else np.array([], dtype=int)

n_unique = len(unique_s2_pos) + len(unique_s3_pos)

print(f"Unique block keys touched: {len(unique_block_keys)}")
print(f"Unique S2 candidates: {len(unique_s2_pos)} | Unique S3 candidates: {len(unique_s3_pos)}")
print(f"Total unique candidates to vectorize: {n_unique}")
print(f"Phase 1 (candidate universe) time: {time.time()-t0:.2f}s")

SAFE_LIMIT = 2_000_000
if n_unique > SAFE_LIMIT:
    raise RuntimeError(
        f"Unique candidate universe ({n_unique}) exceeds safety limit "
        f"({SAFE_LIMIT}). Stopping before building the matrix — redesign needed."
    )

# =========================================================
# Phase 2 — build ONE small sparse matrix over just this candidate universe
# =========================================================
t0 = time.time()

uniq_ids = np.concatenate([train_s2_ids[unique_s2_pos], train_s3_ids[unique_s3_pos]])
uniq_names = np.concatenate([train_s2_norm_names[unique_s2_pos], train_s3_norm_names[unique_s3_pos]])

vectorizer = CountVectorizer(
    tokenizer=str.split,     # exact same tokenization as the earlier set(name.split()) version
    token_pattern=None,
    lowercase=False,         # names already normalized/lowercased
    binary=True,
    dtype=np.int32,
)
cand_matrix = vectorizer.fit_transform(uniq_names)  # (n_unique x vocab_size), sparse CSR
cand_matrix = cand_matrix.tocsr()

vocab_size = len(vectorizer.vocabulary_)
nnz = cand_matrix.nnz
approx_mb = nnz * 12 / (1024**2)

print(f"Vocabulary size: {vocab_size}")
print(f"Matrix nnz: {nnz} (~{approx_mb:.1f} MB)")
print(f"Phase 2 (vectorization) time: {time.time()-t0:.2f}s")

# Precompute each candidate's token count ONCE (vectorized)
cand_token_counts = np.asarray(cand_matrix.sum(axis=1)).ravel()

# Map original (train_s2/train_s3) integer positions -> compact matrix row
compact_row_for_s2 = np.full(len(train_s2_ids), -1, dtype=np.int64)
compact_row_for_s3 = np.full(len(train_s3_ids), -1, dtype=np.int64)
compact_row_for_s2[unique_s2_pos] = np.arange(len(unique_s2_pos))
compact_row_for_s3[unique_s3_pos] = len(unique_s2_pos) + np.arange(len(unique_s3_pos))

# =========================================================
# Phase 3 — main benchmark loop: vectorized Jaccard per query
# =========================================================
N_SMALL = 100
N_LARGE = 500

baseline_recalls = []
recalls_n_small = []
recalls_n_large = []
cand_counts_before = []
skipped_no_match = 0

t0 = time.time()

for row in sample_s1.itertuples(index=False):
    entity_id = row.entity_id
    block_key = row.block_key

    true_set = parse_ids(gt_lookup.get(entity_id))
    if not true_set:
        skipped_no_match += 1
        continue

    idx2, idx3 = get_candidate_arrays_fast_train(block_key)
    cand_ids = np.concatenate([train_s2_ids[idx2], train_s3_ids[idx3]])
    n_total = len(cand_ids)
    cand_counts_before.append(n_total)

    true_ids = np.array(list(true_set), dtype=object)
    baseline_recalls.append(np.isin(true_ids, cand_ids).mean())

    if n_total == 0:
        recalls_n_small.append(0.0)
        recalls_n_large.append(0.0)
        continue

    # Vectorized gather of this query's candidate rows in the compact matrix
    compact_rows = np.concatenate([compact_row_for_s2[idx2], compact_row_for_s3[idx3]])

    # Query token vector (from the already-fit vocabulary)
    query_vec = vectorizer.transform([row.norm_name])
    len_a = query_vec.sum()

    cand_sub = cand_matrix[compact_rows]              # sparse row gather
    intersection = cand_sub.dot(query_vec.T).toarray().ravel()  # one sparse dot product

    len_b = cand_token_counts[compact_rows]
    union = len_a + len_b - intersection
    jaccard = np.divide(intersection, union, out=np.zeros_like(union, dtype=float), where=union > 0)

    order = np.argsort(-jaccard)

    for N, recall_list in [(N_SMALL, recalls_n_small), (N_LARGE, recalls_n_large)]:
        k = min(N, n_total)
        top_ids = cand_ids[order[:k]]
        recall_list.append(np.isin(true_ids, top_ids).mean())

elapsed = time.time() - t0

print()
print("===== VECTORIZED JACCARD BENCHMARK =====")
print("V3 blocking baseline recall:", round(np.mean(baseline_recalls), 4))
print(f"Top-{N_SMALL} Jaccard recall (vectorized):", round(np.mean(recalls_n_small), 4))
print(f"Top-{N_LARGE} Jaccard recall (vectorized):", round(np.mean(recalls_n_large), 4))
print()
print("Entities evaluated:", len(baseline_recalls))
print("Entities skipped (no GT match):", skipped_no_match)
print(f"Scoring loop runtime: {elapsed:.2f}s")
print(f"Per-query rate: {elapsed/len(baseline_recalls)*1000:.2f} ms/entity")
print(f"Estimated full-test scoring time (loop only): "
      f"{elapsed/len(baseline_recalls)*1732544/3600:.2f} hours")

Unique block keys touched: 1848
Unique S2 candidates: 1849190 | Unique S3 candidates: 2054190
Total unique candidates to vectorize: 3903380
Phase 1 (candidate universe) time: 2.62s


RuntimeError: Unique candidate universe (3903380) exceeds safety limit (2000000). Stopping before building the matrix — redesign needed.

In [12]:
import time
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

N_SMALL = 100
N_LARGE = 500
SAFE_BLOCK_LIMIT = 300_000  # per-block guard; adjust only after seeing Phase 0 output

# =========================================================
# Phase 0 — inspect block sizes ONLY (no matrix building)
# =========================================================
t0 = time.time()

unique_block_keys = sample_s1['block_key'].unique()
block_sizes = {}

for bk in unique_block_keys:
    idx2, idx3 = get_candidate_arrays_fast_train(bk)
    block_sizes[bk] = len(idx2) + len(idx3)

sizes_arr = np.array(list(block_sizes.values()))
sorted_blocks = sorted(block_sizes.items(), key=lambda x: -x[1])

print("===== PHASE 0: BLOCK SIZE INSPECTION =====")
print(f"Unique blocks: {len(block_sizes)}")
print(f"Sum of block sizes (should match earlier 3,903,380): {sizes_arr.sum()}")
print(f"Max block size: {sizes_arr.max()}")
print(f"Mean block size: {sizes_arr.mean():.1f} | Median: {np.median(sizes_arr):.1f}")
print()
print("Top 5 largest blocks:")
for bk, size in sorted_blocks[:5]:
    est_nnz = size * 3
    est_mb = est_nnz * 12 / (1024**2)
    print(f"  block_key={bk!r} | candidates={size} | est. matrix ~{est_mb:.2f} MB")
print(f"Phase 0 time: {time.time()-t0:.2f}s")

if sizes_arr.max() > SAFE_BLOCK_LIMIT:
    print(f"\nWARNING: at least one block exceeds SAFE_BLOCK_LIMIT={SAFE_BLOCK_LIMIT}.")
    print("These blocks will be SKIPPED (not vectorized) in Phase 1 below.")
else:
    print(f"\nAll blocks are within SAFE_BLOCK_LIMIT={SAFE_BLOCK_LIMIT}. Safe to proceed.")

# =========================================================
# Phase 1 — block-by-block vectorized Jaccard
# =========================================================
grouped = sample_s1.groupby('block_key')

baseline_recalls = []
recalls_n_small = []
recalls_n_large = []

skipped_no_match = 0
skipped_large_block_entities = 0
skipped_large_block_count = 0
processed_candidates_total = 0
entities_processed = 0

t0 = time.time()

for block_key, group in grouped:
    idx2, idx3 = get_candidate_arrays_fast_train(block_key)
    block_size = len(idx2) + len(idx3)

    if block_size == 0:
        # No candidates at all in this block — every entity here gets recall 0
        for row in group.itertuples(index=False):
            true_set = parse_ids(gt_lookup.get(row.entity_id))
            if not true_set:
                skipped_no_match += 1
                continue
            baseline_recalls.append(0.0)
            recalls_n_small.append(0.0)
            recalls_n_large.append(0.0)
        continue

    if block_size > SAFE_BLOCK_LIMIT:
        skipped_large_block_count += 1
        for row in group.itertuples(index=False):
            true_set = parse_ids(gt_lookup.get(row.entity_id))
            if not true_set:
                skipped_no_match += 1
                continue
            skipped_large_block_entities += 1
        continue

    cand_ids = np.concatenate([train_s2_ids[idx2], train_s3_ids[idx3]])
    cand_names = np.concatenate([train_s2_norm_names[idx2], train_s3_norm_names[idx3]])

    # Fresh, tiny vectorizer for THIS block only — same tokenization as before
    vectorizer = CountVectorizer(
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        binary=True,
        dtype=np.int32,
    )
    cand_matrix = vectorizer.fit_transform(cand_names).tocsr()
    cand_token_counts = np.asarray(cand_matrix.sum(axis=1)).ravel()

    processed_candidates_total += block_size

    for row in group.itertuples(index=False):
        entity_id = row.entity_id
        true_set = parse_ids(gt_lookup.get(entity_id))
        if not true_set:
            skipped_no_match += 1
            continue

        true_ids = np.array(list(true_set), dtype=object)
        baseline_recalls.append(np.isin(true_ids, cand_ids).mean())

        query_vec = vectorizer.transform([row.norm_name])
        len_a = query_vec.sum()

        intersection = cand_matrix.dot(query_vec.T).toarray().ravel()
        union = len_a + cand_token_counts - intersection
        jaccard = np.divide(
            intersection, union,
            out=np.zeros_like(union, dtype=float),
            where=union > 0
        )

        order = np.argsort(-jaccard)

        for N, recall_list in [(N_SMALL, recalls_n_small), (N_LARGE, recalls_n_large)]:
            k = min(N, block_size)
            top_ids = cand_ids[order[:k]]
            recall_list.append(np.isin(true_ids, top_ids).mean())

        entities_processed += 1

    # Discard this block's matrix before moving on
    del cand_matrix, vectorizer, cand_token_counts, cand_ids, cand_names

elapsed = time.time() - t0

print()
print("===== PHASE 1: BLOCK-BY-BLOCK VECTORIZED JACCARD =====")
print("V3 blocking baseline recall:", round(np.mean(baseline_recalls), 4) if baseline_recalls else None)
print(f"Top-{N_SMALL} Jaccard recall (block-wise):", round(np.mean(recalls_n_small), 4) if recalls_n_small else None)
print(f"Top-{N_LARGE} Jaccard recall (block-wise):", round(np.mean(recalls_n_large), 4) if recalls_n_large else None)
print()
print("Entities evaluated:", entities_processed)
print("Entities skipped (no GT match):", skipped_no_match)
print("Blocks skipped (over SAFE_BLOCK_LIMIT):", skipped_large_block_count)
print("Entities skipped due to oversized block:", skipped_large_block_entities)
print("Total candidates actually vectorized:", processed_candidates_total)
print(f"Phase 1 runtime: {elapsed:.2f}s")
if entities_processed:
    per_entity_ms = elapsed / entities_processed * 1000
    print(f"Per-entity rate: {per_entity_ms:.2f} ms/entity")
    print(f"Naive full-test extrapolation (upper bound, ignores block reuse savings): "
          f"{elapsed/entities_processed*1732544/3600:.2f} hours")

===== PHASE 0: BLOCK SIZE INSPECTION =====
Unique blocks: 1848
Sum of block sizes (should match earlier 3,903,380): 3903380
Max block size: 62320
Mean block size: 2112.2 | Median: 170.0

Top 5 largest blocks:
  block_key='US||pediatric' | candidates=62320 | est. matrix ~2.14 MB
  block_key='India||new' | candidates=31506 | est. matrix ~1.08 MB
  block_key='US||family' | candidates=26804 | est. matrix ~0.92 MB
  block_key='US||a' | candidates=24451 | est. matrix ~0.84 MB
  block_key='India||shree' | candidates=22339 | est. matrix ~0.77 MB
Phase 0 time: 0.01s

All blocks are within SAFE_BLOCK_LIMIT=300000. Safe to proceed.

===== PHASE 1: BLOCK-BY-BLOCK VECTORIZED JACCARD =====
V3 blocking baseline recall: 0.7893
Top-100 Jaccard recall (block-wise): 0.6646
Top-500 Jaccard recall (block-wise): 0.7274

Entities evaluated: 2820
Entities skipped (no GT match): 180
Blocks skipped (over SAFE_BLOCK_LIMIT): 0
Entities skipped due to oversized block: 0
Total candidates actually vectorized: 390338

In [13]:
import time
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

N_SMALL = 100
N_LARGE = 500
SAFE_BLOCK_LIMIT = 300_000

grouped = sample_s1.groupby('block_key')

baseline_recalls = []
recalls_n_small_final = []
recalls_n_large_final = []
recalls_n_small_shortlist_only = []
recalls_n_large_shortlist_only = []

full_block_sizes = []
shortlist_sizes = []
scored_candidate_counts = []

backfill_needed_small = 0
backfill_needed_large = 0

skipped_no_match = 0
skipped_large_block_count = 0
skipped_large_block_entities = 0
entities_processed = 0

t0 = time.time()

for block_key, group in grouped:
    idx2, idx3 = get_candidate_arrays_fast_train(block_key)
    block_size = len(idx2) + len(idx3)

    if block_size == 0:
        for row in group.itertuples(index=False):
            true_set = parse_ids(gt_lookup.get(row.entity_id))
            if not true_set:
                skipped_no_match += 1
                continue
            baseline_recalls.append(0.0)
            recalls_n_small_final.append(0.0)
            recalls_n_large_final.append(0.0)
            recalls_n_small_shortlist_only.append(0.0)
            recalls_n_large_shortlist_only.append(0.0)
        continue

    if block_size > SAFE_BLOCK_LIMIT:
        skipped_large_block_count += 1
        for row in group.itertuples(index=False):
            true_set = parse_ids(gt_lookup.get(row.entity_id))
            if not true_set:
                skipped_no_match += 1
                continue
            skipped_large_block_entities += 1
        continue

    cand_ids = np.concatenate([train_s2_ids[idx2], train_s3_ids[idx3]])
    cand_names = np.concatenate([train_s2_norm_names[idx2], train_s3_norm_names[idx3]])

    vectorizer = CountVectorizer(
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        binary=True,
        dtype=np.int32,
    )
    cand_matrix_csr = vectorizer.fit_transform(cand_names)
    cand_matrix_csc = cand_matrix_csr.tocsc()
    cand_token_counts = np.asarray(cand_matrix_csr.sum(axis=1)).ravel()
    vocab = vectorizer.vocabulary_

    all_idx = np.arange(block_size)

    for row in group.itertuples(index=False):
        entity_id = row.entity_id
        true_set = parse_ids(gt_lookup.get(entity_id))
        if not true_set:
            skipped_no_match += 1
            continue

        true_ids = np.array(list(true_set), dtype=object)
        baseline_recalls.append(np.isin(true_ids, cand_ids).mean())
        full_block_sizes.append(block_size)

        tokens = row.norm_name.split() if row.norm_name else []
        retrieval_tokens = set(tokens[1:])  # exclude block-defining first token

        cols = [vocab[t] for t in retrieval_tokens if t in vocab] if retrieval_tokens else []

        if cols:
            shortlist_idx = np.unique(
                np.concatenate([
                    cand_matrix_csc.indices[cand_matrix_csc.indptr[c]:cand_matrix_csc.indptr[c + 1]]
                    for c in cols
                ])
            )
        else:
            shortlist_idx = np.array([], dtype=np.int64)

        shortlist_size = len(shortlist_idx)
        shortlist_sizes.append(shortlist_size)

        query_vec = vectorizer.transform([row.norm_name])
        len_a = query_vec.sum()

        if shortlist_size > 0:
            intersection_sl = cand_matrix_csr[shortlist_idx].dot(query_vec.T).toarray().ravel()
            len_b_sl = cand_token_counts[shortlist_idx]
            union_sl = len_a + len_b_sl - intersection_sl
            jaccard_sl = np.divide(
                intersection_sl, union_sl,
                out=np.zeros_like(union_sl, dtype=float),
                where=union_sl > 0
            )
            order_sl = np.argsort(-jaccard_sl)
            ranked_shortlist_ids = cand_ids[shortlist_idx[order_sl]]
        else:
            jaccard_sl = np.array([], dtype=float)
            ranked_shortlist_ids = np.array([], dtype=cand_ids.dtype)

        scored_so_far = shortlist_size

        for N, recall_list in [(N_SMALL, recalls_n_small_shortlist_only),
                                (N_LARGE, recalls_n_large_shortlist_only)]:
            k = min(N, shortlist_size)
            top_ids = ranked_shortlist_ids[:k]
            recall_list.append(np.isin(true_ids, top_ids).mean())

        need_backfill_small = shortlist_size < N_SMALL
        need_backfill_large = shortlist_size < N_LARGE
        backfill_needed_small += int(need_backfill_small)
        backfill_needed_large += int(need_backfill_large)

        if need_backfill_large:
            leftover_idx = np.setdiff1d(all_idx, shortlist_idx, assume_unique=False)

            if len(leftover_idx) > 0:
                intersection_lo = cand_matrix_csr[leftover_idx].dot(query_vec.T).toarray().ravel()
                len_b_lo = cand_token_counts[leftover_idx]
                union_lo = len_a + len_b_lo - intersection_lo
                jaccard_lo = np.divide(
                    intersection_lo, union_lo,
                    out=np.zeros_like(union_lo, dtype=float),
                    where=union_lo > 0
                )
            else:
                jaccard_lo = np.array([], dtype=float)

            # --- FIX: merge scores + ids BEFORE ranking, then do ONE global argsort ---
            combined_ids = np.concatenate([cand_ids[shortlist_idx], cand_ids[leftover_idx]])
            combined_scores = np.concatenate([jaccard_sl, jaccard_lo])

            global_order = np.argsort(-combined_scores)
            full_ranked_ids = combined_ids[global_order]

            scored_so_far += len(leftover_idx)
        else:
            full_ranked_ids = ranked_shortlist_ids

        scored_candidate_counts.append(scored_so_far)

        for N, recall_list in [(N_SMALL, recalls_n_small_final),
                                (N_LARGE, recalls_n_large_final)]:
            k = min(N, len(full_ranked_ids))
            top_ids = full_ranked_ids[:k]
            recall_list.append(np.isin(true_ids, top_ids).mean())

        entities_processed += 1

    del cand_matrix_csr, cand_matrix_csc, vectorizer, cand_token_counts, cand_ids, cand_names

elapsed = time.time() - t0

print("===== TOKEN-INVERTED-INDEX (CSC) RETRIEVAL BENCHMARK =====")
print()
print("V3 blocking baseline recall:", round(np.mean(baseline_recalls), 4))
print()
print(f"Top-{N_SMALL} recall — shortlist only (no backfill):", round(np.mean(recalls_n_small_shortlist_only), 4))
print(f"Top-{N_LARGE} recall — shortlist only (no backfill):", round(np.mean(recalls_n_large_shortlist_only), 4))
print()
print(f"Top-{N_SMALL} recall — FINAL (after backfill):", round(np.mean(recalls_n_small_final), 4))
print(f"Top-{N_LARGE} recall — FINAL (after backfill):", round(np.mean(recalls_n_large_final), 4))
print()
print("Average full V3 block size:", round(np.mean(full_block_sizes), 1))
print("Average retrieved shortlist size:", round(np.mean(shortlist_sizes), 1))
print("Average candidates actually Jaccard-scored:", round(np.mean(scored_candidate_counts), 1))
print()
print(f"Queries requiring backfill for Top-{N_SMALL}: {backfill_needed_small} "
      f"({backfill_needed_small/entities_processed*100:.1f}%)")
print(f"Queries requiring backfill for Top-{N_LARGE}: {backfill_needed_large} "
      f"({backfill_needed_large/entities_processed*100:.1f}%)")
print()
print("Entities evaluated:", entities_processed)
print("Entities skipped (no GT match):", skipped_no_match)
print("Blocks skipped (over SAFE_BLOCK_LIMIT):", skipped_large_block_count)
print("Entities skipped due to oversized block:", skipped_large_block_entities)
print()
print(f"Total runtime: {elapsed:.2f}s")
print(f"Per-entity rate: {elapsed/entities_processed*1000:.2f} ms/entity")
print(f"Estimated full-test runtime (linear extrapolation): "
      f"{elapsed/entities_processed*1732544/3600:.2f} hours")

===== TOKEN-INVERTED-INDEX (CSC) RETRIEVAL BENCHMARK =====

V3 blocking baseline recall: 0.7893

Top-100 recall — shortlist only (no backfill): 0.6097
Top-500 recall — shortlist only (no backfill): 0.6491

Top-100 recall — FINAL (after backfill): 0.6666
Top-500 recall — FINAL (after backfill): 0.7279

Average full V3 block size: 5910.2
Average retrieved shortlist size: 838.5
Average candidates actually Jaccard-scored: 4735.0

Queries requiring backfill for Top-100: 1821 (64.6%)
Queries requiring backfill for Top-500: 2539 (90.0%)

Entities evaluated: 2820
Entities skipped (no GT match): 180
Blocks skipped (over SAFE_BLOCK_LIMIT): 0
Entities skipped due to oversized block: 0

Total runtime: 74.81s
Per-entity rate: 26.53 ms/entity
Estimated full-test runtime (linear extrapolation): 12.77 hours


In [14]:
import time
import unicodedata
import numpy as np
import pandas as pd

# =========================================================
# Step 0 — normalize_address (reused definition, consistent with your logged experiment 8)
# =========================================================
def normalize_address(address):
    if pd.isna(address):
        return ""
    address = str(address).lower()
    address = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in address
    )
    return ' '.join(address.split()).strip()

def parse_ids(cell):
    if pd.isna(cell) or cell == '':
        return set()
    return set(cell.split(','))

def f_beta_score(true_set, pred_set, beta=0.5):
    if len(true_set) == 0:
        return 1.0 if len(pred_set) == 0 else 0.0
    if len(pred_set) == 0:
        return 0.0
    tp = len(true_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(true_set)
    if precision == 0 and recall == 0:
        return 0.0
    beta_sq = beta ** 2
    return (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)

t0 = time.time()
train_source2_keyed_v3['norm_address'] = train_source2_keyed_v3['business_address'].apply(normalize_address)
print(f"train_source2 norm_address built in {time.time()-t0:.1f}s")

t0 = time.time()
train_source3_keyed_v3['norm_address'] = train_source3_keyed_v3['business_address'].apply(normalize_address)
print(f"train_source3 norm_address built in {time.time()-t0:.1f}s")

sample_s1 = sample_s1.copy()
sample_s1['norm_address'] = sample_s1['business_address'].apply(normalize_address)

# =========================================================
# Step 1 — build the three exact indexes (name-only, name+address, address-only)
# =========================================================
t0 = time.time()

s2_name_idx = train_source2_keyed_v3.groupby(['block_key', 'norm_name'])['entity_id'].apply(list).to_dict()
s3_name_idx = train_source3_keyed_v3.groupby(['block_key', 'norm_name'])['entity_id'].apply(list).to_dict()

s2_name_addr_idx = train_source2_keyed_v3.groupby(['block_key', 'norm_name', 'norm_address'])['entity_id'].apply(list).to_dict()
s3_name_addr_idx = train_source3_keyed_v3.groupby(['block_key', 'norm_name', 'norm_address'])['entity_id'].apply(list).to_dict()

s2_addr_idx = train_source2_keyed_v3.groupby(['block_key', 'norm_address'])['entity_id'].apply(list).to_dict()
s3_addr_idx = train_source3_keyed_v3.groupby(['block_key', 'norm_address'])['entity_id'].apply(list).to_dict()

# entity_id -> norm_address lookup, for cheap address-token overlap on pool members only
s2_addr_text = train_source2_keyed_v3.set_index('entity_id')['norm_address'].to_dict()
s3_addr_text = train_source3_keyed_v3.set_index('entity_id')['norm_address'].to_dict()

print(f"All exact indexes + address-text lookups built in {time.time()-t0:.1f}s")
print("S2 name idx groups:", len(s2_name_idx), "| S3 name idx groups:", len(s3_name_idx))
print("S2 name+addr idx groups:", len(s2_name_addr_idx), "| S3 name+addr idx groups:", len(s3_name_addr_idx))
print("S2 addr idx groups:", len(s2_addr_idx), "| S3 addr idx groups:", len(s3_addr_idx))

# =========================================================
# Step 2 — per-entity candidate-pool construction + ceiling measurement
#          + simple-rule evaluation (no weighted score yet)
# =========================================================
pool_sizes = []
pool_recall_hits = []

# Prediction sets per rule, per entity — collected for F-beta scoring
results_exact_name = []
results_exact_name_or_addr = []
results_name_plus_addr_support = {1: [], 2: []}  # threshold -> list of (true_set, predicted)

skipped_no_match = 0
entities_processed = 0

t0 = time.time()

for row in sample_s1.itertuples(index=False):
    entity_id = row.entity_id
    true_set = parse_ids(gt_lookup.get(entity_id))
    if not true_set:
        skipped_no_match += 1
        continue

    block_key = row.block_key
    norm_name = row.norm_name
    norm_addr = row.norm_address
    query_addr_tokens = set(norm_addr.split()) if norm_addr else set()

    # --- Three index lookups (O(1) dict gets, no scanning) ---
    name_hits = set(s2_name_idx.get((block_key, norm_name), [])) | set(s3_name_idx.get((block_key, norm_name), []))
    name_addr_hits = set(s2_name_addr_idx.get((block_key, norm_name, norm_addr), [])) | set(s3_name_addr_idx.get((block_key, norm_name, norm_addr), []))
    addr_hits = set(s2_addr_idx.get((block_key, norm_addr), [])) | set(s3_addr_idx.get((block_key, norm_addr), []))

    union_pool = name_hits | name_addr_hits | addr_hits
    pool_sizes.append(len(union_pool))

    # --- Candidate-pool ceiling: are all GT matches present? ---
    true_ids_arr = np.array(list(true_set), dtype=object)
    pool_arr = np.array(list(union_pool), dtype=object) if union_pool else np.array([], dtype=object)
    hit_fraction = np.isin(true_ids_arr, pool_arr).mean() if len(true_ids_arr) else 1.0
    pool_recall_hits.append(hit_fraction)

    # --- Rule 1: exact_name only ---
    pred_exact_name = name_hits
    results_exact_name.append((true_set, pred_exact_name))

    # --- Rule 2: exact_name OR exact_addr ---
    pred_exact_name_or_addr = name_hits | addr_hits
    results_exact_name_or_addr.append((true_set, pred_exact_name_or_addr))

    # --- Rule 3: exact_name plus address-token-overlap support from remaining pool ---
    remaining_pool = union_pool - name_hits
    if remaining_pool and query_addr_tokens:

        for thresh in (1, 2):
            support_ids = set()
            for cid in remaining_pool:
                cand_addr = s2_addr_text.get(cid) if cid.startswith('S2-') else s3_addr_text.get(cid)
                if not cand_addr:
                    continue
                overlap = len(query_addr_tokens & set(cand_addr.split()))
                if overlap >= thresh:
                    support_ids.add(cid)
            predicted = name_hits | support_ids
            results_name_plus_addr_support[thresh].append((true_set, predicted))
    else:
        for thresh in (1, 2):
            results_name_plus_addr_support[thresh].append((true_set, name_hits))

    entities_processed += 1

elapsed = time.time() - t0

# =========================================================
# Step 3 — reporting
# =========================================================
pool_sizes_arr = np.array(pool_sizes)

print()
print("===== CANDIDATE-POOL CEILING =====")
print("Entities evaluated:", entities_processed)
print("Entities skipped (no GT match):", skipped_no_match)
print("Candidate-pool recall (macro-avg):", round(np.mean(pool_recall_hits), 4))
print("Avg pool size:", round(pool_sizes_arr.mean(), 2))
print("Median pool size:", np.median(pool_sizes_arr))
print("P95 pool size:", np.percentile(pool_sizes_arr, 95))
print("Max pool size:", pool_sizes_arr.max())

def evaluate_rule(name, results):
    scores = [f_beta_score(t, p, beta=0.5) for t, p in results]
    tp = sum(len(t & p) for t, p in results)
    pred_total = sum(len(p) for t, p in results)
    true_total = sum(len(t) for t, p in results)
    precision = tp / pred_total if pred_total else 0
    recall = tp / true_total if true_total else 0
    coverage = np.mean([len(p) > 0 for t, p in results])
    avg_pred = np.mean([len(p) for t, p in results])
    print(f"\n--- {name} ---")
    print("F0.5:", round(np.mean(scores), 6))
    print("Precision:", round(precision, 6))
    print("Recall:", round(recall, 6))
    print("Coverage:", round(coverage, 6))
    print("Avg predictions/entity:", round(avg_pred, 3))

print()
print("===== SIMPLE-RULE EVALUATION =====")
evaluate_rule("exact_name", results_exact_name)
evaluate_rule("exact_name OR exact_addr", results_exact_name_or_addr)
evaluate_rule("exact_name + addr_overlap>=1 support", results_name_plus_addr_support[1])
evaluate_rule("exact_name + addr_overlap>=2 support", results_name_plus_addr_support[2])

print()
print("===== RUNTIME =====")
print(f"Total loop runtime: {elapsed:.2f}s")
print(f"Per-entity rate: {elapsed/entities_processed*1000:.2f} ms/entity")
print(f"Estimated full-test runtime (linear extrapolation): "
      f"{elapsed/entities_processed*1732544/3600:.2f} hours")

train_source2 norm_address built in 59.2s
train_source3 norm_address built in 58.1s


KeyboardInterrupt: 

In [16]:
import numpy as np
import pandas as pd

def build_fast_index(df, key_cols):
    combined_key = df[key_cols[0]].astype(str)

    for col in key_cols[1:]:
        combined_key = combined_key + '||' + df[col].astype(str)

    combined_key = combined_key.to_numpy()
    entity_ids = df['entity_id'].to_numpy()

    codes, uniques = pd.factorize(combined_key, sort=True)

    sort_idx = np.argsort(codes, kind='stable')
    sorted_codes = codes[sort_idx]
    sorted_entity_ids = entity_ids[sort_idx]

    counts = np.bincount(sorted_codes, minlength=len(uniques))

    indptr = np.zeros(len(uniques) + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])

    return uniques, indptr, sorted_entity_ids

In [17]:
t0 = time.time()

s2_name_u, s2_name_ptr, s2_name_ids = build_fast_index(
    train_source2_keyed_v3,
    ['block_key', 'norm_name']
)

print(
    f"S2 name index built in {time.time()-t0:.2f}s, "
    f"{len(s2_name_u):,} unique keys"
)

S2 name index built in 54.34s, 3,368,060 unique keys


In [18]:
t0 = time.time()

s3_name_u, s3_name_ptr, s3_name_ids = build_fast_index(
    train_source3_keyed_v3,
    ['block_key', 'norm_name']
)

print(
    f"S3 name index built in {time.time()-t0:.2f}s, "
    f"{len(s3_name_u):,} unique keys"
)

S3 name index built in 51.17s, 3,595,824 unique keys


In [19]:
t0 = time.time()

s2_addr_u, s2_addr_ptr, s2_addr_ids = build_fast_index(
    train_source2_keyed_v3,
    ['block_key', 'norm_address']
)

print(
    f"S2 address index built in {time.time()-t0:.2f}s, "
    f"{len(s2_addr_u):,} unique keys"
)

S2 address index built in 35.76s, 4,519,259 unique keys


In [20]:
t0 = time.time()

s3_addr_u, s3_addr_ptr, s3_addr_ids = build_fast_index(
    train_source3_keyed_v3,
    ['block_key', 'norm_address']
)

print(
    f"S3 address index built in {time.time()-t0:.2f}s, "
    f"{len(s3_addr_u):,} unique keys"
)

S3 address index built in 35.82s, 4,849,114 unique keys


In [21]:
import time
import numpy as np

def lookup_fast_index(key, uniques, indptr, sorted_entity_ids):
    pos = np.searchsorted(uniques, key)
    if pos < len(uniques) and uniques[pos] == key:
        return sorted_entity_ids[indptr[pos]:indptr[pos + 1]]
    return np.empty(0, dtype=sorted_entity_ids.dtype)


pool_sizes = []
pool_recall_hits = []

results_exact_name = []
results_exact_name_or_addr = []

skipped_no_match = 0
entities_processed = 0

t0 = time.time()

for row in sample_s1.itertuples(index=False):
    entity_id = row.entity_id
    true_set = parse_ids(gt_lookup.get(entity_id))
    if not true_set:
        skipped_no_match += 1
        continue

    block_key = row.block_key
    norm_name = row.norm_name
    norm_address = row.norm_address if pd.notna(row.norm_address) else ""

    name_key = f"{block_key}||{norm_name}"
    addr_key = f"{block_key}||{norm_address}"

    s2_n = lookup_fast_index(name_key, s2_name_u, s2_name_ptr, s2_name_ids)
    s3_n = lookup_fast_index(name_key, s3_name_u, s3_name_ptr, s3_name_ids)
    name_hits = set(s2_n.tolist()) | set(s3_n.tolist())

    s2_a = lookup_fast_index(addr_key, s2_addr_u, s2_addr_ptr, s2_addr_ids)
    s3_a = lookup_fast_index(addr_key, s3_addr_u, s3_addr_ptr, s3_addr_ids)
    addr_hits = set(s2_a.tolist()) | set(s3_a.tolist())

    union_pool = name_hits | addr_hits
    pool_sizes.append(len(union_pool))

    true_ids_arr = np.array(list(true_set), dtype=object)
    pool_arr = np.array(list(union_pool), dtype=object) if union_pool else np.array([], dtype=object)
    hit_fraction = np.isin(true_ids_arr, pool_arr).mean() if len(true_ids_arr) else 1.0
    pool_recall_hits.append(hit_fraction)

    results_exact_name.append((true_set, name_hits))
    results_exact_name_or_addr.append((true_set, name_hits | addr_hits))

    entities_processed += 1

elapsed = time.time() - t0

pool_sizes_arr = np.array(pool_sizes)

def evaluate_rule(name, results):
    scores = [f_beta_score(t, p, beta=0.5) for t, p in results]
    tp = sum(len(t & p) for t, p in results)
    pred_total = sum(len(p) for t, p in results)
    true_total = sum(len(t) for t, p in results)
    precision = tp / pred_total if pred_total else 0
    recall = tp / true_total if true_total else 0
    coverage = np.mean([len(p) > 0 for t, p in results])
    avg_pred = np.mean([len(p) for t, p in results])
    print(f"\n--- {name} ---")
    print("F0.5:", round(np.mean(scores), 6))
    print("Precision:", round(precision, 6))
    print("Recall:", round(recall, 6))
    print("Coverage:", round(coverage, 6))
    print("Avg predictions/entity:", round(avg_pred, 3))

print("===== EXACT-INDEX CANDIDATE POOL =====")
print("Entities evaluated:", entities_processed)
print("Entities skipped:", skipped_no_match)
print("Candidate-pool recall:", round(np.mean(pool_recall_hits), 4))
print("Avg pool size:", round(pool_sizes_arr.mean(), 2))
print("Median pool size:", np.median(pool_sizes_arr))
print("P95 pool size:", np.percentile(pool_sizes_arr, 95))
print("Max pool size:", pool_sizes_arr.max())

print()
print("===== SIMPLE-RULE EVALUATION =====")
evaluate_rule("exact_name", results_exact_name)
evaluate_rule("exact_name OR exact_addr", results_exact_name_or_addr)

print()
print("===== RUNTIME =====")
print(f"Total loop runtime: {elapsed:.2f}s")
print(f"Per-entity rate: {elapsed/entities_processed*1000:.2f} ms/entity")
print(f"Estimated full-test runtime (linear extrapolation): "
      f"{elapsed/entities_processed*1732544/3600:.2f} hours")

===== EXACT-INDEX CANDIDATE POOL =====
Entities evaluated: 2820
Entities skipped: 180
Candidate-pool recall: 0.4659
Avg pool size: 40.08
Median pool size: 3.0
P95 pool size: 163.0
Max pool size: 1374

===== SIMPLE-RULE EVALUATION =====

--- exact_name ---
F0.5: 0.411849
Precision: 0.041337
Recall: 0.446476
Coverage: 0.919504
Avg predictions/entity: 40.002

--- exact_name OR exact_addr ---
F0.5: 0.426581
Precision: 0.043321
Recall: 0.468882
Coverage: 0.928723
Avg predictions/entity: 40.085

===== RUNTIME =====
Total loop runtime: 24.33s
Per-entity rate: 8.63 ms/entity
Estimated full-test runtime (linear extrapolation): 4.15 hours


In [22]:
import time
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

SAFE_BLOCK_LIMIT = 300_000

grouped = sample_s1.groupby('block_key')

pool_sizes_t1, pool_sizes_t2 = [], []
recall_hits_t1, recall_hits_t2 = [], []

skipped_no_match = 0
skipped_large_block = 0
entities_processed = 0

t0 = time.time()

for block_key, group in grouped:
    idx2, idx3 = get_candidate_arrays_fast_train(block_key)
    block_size = len(idx2) + len(idx3)

    if block_size == 0:
        for row in group.itertuples(index=False):
            if not parse_ids(gt_lookup.get(row.entity_id)):
                skipped_no_match += 1
                continue
            pool_sizes_t1.append(0); pool_sizes_t2.append(0)
            recall_hits_t1.append(0.0); recall_hits_t2.append(0.0)
        continue

    if block_size > SAFE_BLOCK_LIMIT:
        skipped_large_block += len(group)
        continue

    cand_ids = np.concatenate([train_s2_ids[idx2], train_s3_ids[idx3]])
    cand_names = np.concatenate([train_s2_norm_names[idx2], train_s3_norm_names[idx3]])

    vectorizer = CountVectorizer(
        analyzer='char', ngram_range=(3, 3),
        lowercase=False, binary=True, dtype=np.int32,
    )
    cand_matrix_csc = vectorizer.fit_transform(cand_names).tocsc()

    for row in group.itertuples(index=False):
        entity_id = row.entity_id
        true_set = parse_ids(gt_lookup.get(entity_id))
        if not true_set:
            skipped_no_match += 1
            continue

        true_ids = np.array(list(true_set), dtype=object)

        query_vec = vectorizer.transform([row.norm_name])
        cols = query_vec.indices  # trigram columns present in this query

        if len(cols) == 0:
            pool_sizes_t1.append(0); pool_sizes_t2.append(0)
            recall_hits_t1.append(0.0); recall_hits_t2.append(0.0)
            entities_processed += 1
            continue

        rows_concat = np.concatenate([
            cand_matrix_csc.indices[cand_matrix_csc.indptr[c]:cand_matrix_csc.indptr[c + 1]]
            for c in cols
        ])

        counts = np.bincount(rows_concat, minlength=block_size)

        idx_t1 = np.nonzero(counts >= 1)[0]
        idx_t2 = np.nonzero(counts >= 2)[0]

        ids_t1 = cand_ids[idx_t1]
        ids_t2 = cand_ids[idx_t2]

        pool_sizes_t1.append(len(ids_t1))
        pool_sizes_t2.append(len(ids_t2))

        recall_hits_t1.append(np.isin(true_ids, ids_t1).mean())
        recall_hits_t2.append(np.isin(true_ids, ids_t2).mean())

        entities_processed += 1

    del cand_matrix_csc, vectorizer, cand_ids, cand_names

elapsed = time.time() - t0

def report(name, sizes, hits):
    sizes_arr = np.array(sizes)
    print(f"\n--- Trigram overlap {name} ---")
    print("Candidate-pool recall:", round(np.mean(hits), 4))
    print("Avg pool size:", round(sizes_arr.mean(), 2))
    print("Median pool size:", np.median(sizes_arr))
    print("P95 pool size:", np.percentile(sizes_arr, 95))
    print("Max pool size:", sizes_arr.max())

print("===== CHARACTER TRIGRAM CANDIDATE-POOL BENCHMARK =====")
print("Entities evaluated:", entities_processed)
print("Entities skipped (no GT):", skipped_no_match)
print("Entities skipped (oversized block):", skipped_large_block)

report(">=1 shared trigram", pool_sizes_t1, recall_hits_t1)
report(">=2 shared trigrams", pool_sizes_t2, recall_hits_t2)

print()
print("===== RUNTIME =====")
print(f"Total loop runtime: {elapsed:.2f}s")
print(f"Per-entity rate: {elapsed/entities_processed*1000:.2f} ms/entity")
print(f"Estimated full-test runtime (linear extrapolation): "
      f"{elapsed/entities_processed*1732544/3600:.2f} hours")

===== CHARACTER TRIGRAM CANDIDATE-POOL BENCHMARK =====
Entities evaluated: 2820
Entities skipped (no GT): 180
Entities skipped (oversized block): 0

--- Trigram overlap >=1 shared trigram ---
Candidate-pool recall: 0.7889
Avg pool size: 5833.53
Median pool size: 1684.5
P95 pool size: 20127.84999999997
Max pool size: 62320

--- Trigram overlap >=2 shared trigrams ---
Candidate-pool recall: 0.7852
Avg pool size: 5759.96
Median pool size: 1553.5
P95 pool size: 20127.84999999997
Max pool size: 62320

===== RUNTIME =====
Total loop runtime: 168.52s
Per-entity rate: 59.76 ms/entity
Estimated full-test runtime (linear extrapolation): 28.76 hours


In [24]:
# ============================================================
# REBUILD FAST EXACT NAME + ADDRESS INDEXES FOR VALIDATION
# ============================================================

import numpy as np
import time

def build_fast_index(df, key_cols):
    combined_key = df[key_cols[0]].astype(str)

    for col in key_cols[1:]:
        combined_key = combined_key + '||' + df[col].astype(str)

    combined_key = combined_key.to_numpy()
    entity_ids = df['entity_id'].to_numpy()

    codes, uniques = pd.factorize(combined_key, sort=True)

    sort_idx = np.argsort(codes, kind='stable')

    sorted_codes = codes[sort_idx]
    sorted_entity_ids = entity_ids[sort_idx]

    counts = np.bincount(
        sorted_codes,
        minlength=len(uniques)
    )

    indptr = np.zeros(
        len(uniques) + 1,
        dtype=np.int64
    )

    np.cumsum(counts, out=indptr[1:])

    return (
        uniques,
        indptr,
        sorted_entity_ids
    )


t0 = time.time()

print("Building S2 exact-name index...")
(
    s2_name_uniques,
    s2_name_indptr,
    s2_name_ids
) = build_fast_index(
    train_source2_keyed_v3,
    ['block_key', 'norm_name']
)

print("Building S3 exact-name index...")
(
    s3_name_uniques,
    s3_name_indptr,
    s3_name_ids
) = build_fast_index(
    train_source3_keyed_v3,
    ['block_key', 'norm_name']
)

print("Building S2 exact-address index...")
(
    s2_addr_uniques,
    s2_addr_indptr,
    s2_addr_ids
) = build_fast_index(
    train_source2_keyed_v3,
    ['block_key', 'norm_address']
)

print("Building S3 exact-address index...")
(
    s3_addr_uniques,
    s3_addr_indptr,
    s3_addr_ids
) = build_fast_index(
    train_source3_keyed_v3,
    ['block_key', 'norm_address']
)

print()
print("FAST INDEXES READY")
print("S2 name keys:", len(s2_name_uniques))
print("S3 name keys:", len(s3_name_uniques))
print("S2 address keys:", len(s2_addr_uniques))
print("S3 address keys:", len(s3_addr_uniques))
print("Build time:", round(time.time() - t0, 2), "seconds")

Building S2 exact-name index...
Building S3 exact-name index...
Building S2 exact-address index...
Building S3 exact-address index...

FAST INDEXES READY
S2 name keys: 3368060
S3 name keys: 3595824
S2 address keys: 4519259
S3 address keys: 4849114
Build time: 171.4 seconds


In [28]:
# ============================================================
# SMALL, MEMORY-SAFE RECORD LOOKUP FOR VALIDATION
# ============================================================

# We only need candidate records encountered in the validation
# sample, so first collect those candidate IDs.

candidate_ids_needed_s2 = set()
candidate_ids_needed_s3 = set()

for row in sample_s1.itertuples(index=False):

    block_key = row.block_key
    norm_name = row.norm_name
    norm_address = row.norm_address

    name_key = str(block_key) + "||" + str(norm_name)
    addr_key = str(block_key) + "||" + str(norm_address)

    name_s2 = lookup_fast_index_local(
        name_key,
        s2_name_uniques,
        s2_name_indptr,
        s2_name_ids
    )

    name_s3 = lookup_fast_index_local(
        name_key,
        s3_name_uniques,
        s3_name_indptr,
        s3_name_ids
    )

    addr_s2 = lookup_fast_index_local(
        addr_key,
        s2_addr_uniques,
        s2_addr_indptr,
        s2_addr_ids
    )

    addr_s3 = lookup_fast_index_local(
        addr_key,
        s3_addr_uniques,
        s3_addr_indptr,
        s3_addr_ids
    )

    candidate_ids_needed_s2.update(map(str, name_s2))
    candidate_ids_needed_s2.update(map(str, addr_s2))

    candidate_ids_needed_s3.update(map(str, name_s3))
    candidate_ids_needed_s3.update(map(str, addr_s3))

print("Unique S2 candidate IDs needed:", len(candidate_ids_needed_s2))
print("Unique S3 candidate IDs needed:", len(candidate_ids_needed_s3))

# ------------------------------------------------------------
# Extract only those records from the training data
# ------------------------------------------------------------

s2_needed = train_source2_keyed_v3[
    train_source2_keyed_v3["entity_id"].isin(candidate_ids_needed_s2)
][["entity_id", "business_name", "business_address"]].copy()

s3_needed = train_source3_keyed_v3[
    train_source3_keyed_v3["entity_id"].isin(candidate_ids_needed_s3)
][["entity_id", "business_name", "business_address"]].copy()

print("S2 records loaded:", len(s2_needed))
print("S3 records loaded:", len(s3_needed))

# Small dictionaries: only validation candidates
source2_lookup = (
    s2_needed
    .set_index("entity_id")
    .to_dict("index")
)

source3_lookup = (
    s3_needed
    .set_index("entity_id")
    .to_dict("index")
)

print("Validation record lookup ready.")

Unique S2 candidate IDs needed: 52255
Unique S3 candidate IDs needed: 58987
S2 records loaded: 52255
S3 records loaded: 58987
Validation record lookup ready.


In [30]:
# Build sorted ID arrays for fast record lookup

train_s2_ids_sorted = np.array(
    sorted(source2_lookup.keys()),
    dtype=object
)

train_s3_ids_sorted = np.array(
    sorted(source3_lookup.keys()),
    dtype=object
)

print("S2 lookup records:", len(train_s2_ids_sorted))
print("S3 lookup records:", len(train_s3_ids_sorted))

S2 lookup records: 52255
S3 lookup records: 58987


In [33]:
# Build sorted ID + record arrays for fast validation lookup

s2_items = sorted(source2_lookup.items(), key=lambda x: x[0])
s3_items = sorted(source3_lookup.items(), key=lambda x: x[0])

train_s2_ids_sorted = np.array(
    [item[0] for item in s2_items],
    dtype=object
)

train_s2_records_sorted = [
    item[1] for item in s2_items
]

train_s3_ids_sorted = np.array(
    [item[0] for item in s3_items],
    dtype=object
)

train_s3_records_sorted = [
    item[1] for item in s3_items
]

print("S2 records:", len(train_s2_ids_sorted))
print("S3 records:", len(train_s3_ids_sorted))

S2 records: 52255
S3 records: 58987


In [34]:
# ============================================================
# VALIDATION: Exact Name OR Exact Address Candidate Pool
# + RapidFuzz final filtering
# ============================================================

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
import time

# ------------------------------------------------------------
# Helper: retrieve IDs from our flat exact indexes
# ------------------------------------------------------------

def lookup_fast_index_local(key, uniques, indptr, sorted_entity_ids):
    pos = np.searchsorted(uniques, key)

    if pos < len(uniques) and uniques[pos] == key:
        return sorted_entity_ids[indptr[pos]:indptr[pos + 1]]

    return np.empty(0, dtype=object)


# ------------------------------------------------------------
# Candidate retrieval
# ------------------------------------------------------------

def get_exact_name_address_candidates(row):
    block_key = row.block_key
    norm_name = row.norm_name
    norm_address = row.norm_address

    # Exact normalized NAME candidates
    name_key = str(block_key) + "||" + str(norm_name)

    name_s2 = lookup_fast_index_local(
        name_key,
        s2_name_uniques,
        s2_name_indptr,
        s2_name_ids
    )

    name_s3 = lookup_fast_index_local(
        name_key,
        s3_name_uniques,
        s3_name_indptr,
        s3_name_ids
    )

    # Exact normalized ADDRESS candidates
    addr_key = str(block_key) + "||" + str(norm_address)

    addr_s2 = lookup_fast_index_local(
        addr_key,
        s2_addr_uniques,
        s2_addr_indptr,
        s2_addr_ids
    )

    addr_s3 = lookup_fast_index_local(
        addr_key,
        s3_addr_uniques,
        s3_addr_indptr,
        s3_addr_ids
    )

    candidates = np.unique(
        np.concatenate([
            name_s2,
            name_s3,
            addr_s2,
            addr_s3
        ])
    )

    return candidates


# ------------------------------------------------------------
# Candidate record lookup
# ------------------------------------------------------------

def get_record(candidate_id):
    candidate_id = str(candidate_id)

    if candidate_id.startswith("S2-"):
        idx = np.searchsorted(
            train_s2_ids_sorted,
            candidate_id
        )
        if idx < len(train_s2_ids_sorted) and train_s2_ids_sorted[idx] == candidate_id:
            return train_s2_records_sorted[idx]
        return None

    if candidate_id.startswith("S3-"):
        idx = np.searchsorted(
            train_s3_ids_sorted,
            candidate_id
        )
        if idx < len(train_s3_ids_sorted) and train_s3_ids_sorted[idx] == candidate_id:
            return train_s3_records_sorted[idx]
        return None

    return None


# ------------------------------------------------------------
# Final matching rule
#
# Keep candidate when:
#
#   exact NAME + address similarity >= 70
#
# OR
#
#   exact ADDRESS + name similarity >= 70
#
# Exact name AND exact address automatically survives.
# ------------------------------------------------------------

def predict_for_row(row):

    candidates = get_exact_name_address_candidates(row)

    if len(candidates) == 0:
        return set(), 0

    query_name = str(row.norm_name)
    query_addr = str(row.norm_address)

    predicted = set()

    for candidate_id in candidates:

        record = get_record(candidate_id)

        if record is None:
            continue

        cand_name_raw = record.get("business_name", "")
        cand_addr_raw = record.get("business_address", "")

        # Use the already normalized fields when available
        cand_name = normalize_name(cand_name_raw)
        cand_addr = normalize_address(cand_addr_raw)

        exact_name = (query_name == cand_name)
        exact_addr = (query_addr == cand_addr)

        # Both exact
        if exact_name and exact_addr:
            predicted.add(candidate_id)
            continue

        # Exact name -> check address similarity
        if exact_name:
            addr_score = fuzz.token_sort_ratio(
                query_addr,
                cand_addr
            )

            if addr_score >= 70:
                predicted.add(candidate_id)
                continue

        # Exact address -> check name similarity
        if exact_addr:
            name_score = fuzz.token_sort_ratio(
                query_name,
                cand_name
            )

            if name_score >= 70:
                predicted.add(candidate_id)
                continue

    return predicted, len(candidates)


# ============================================================
# RUN VALIDATION
# ============================================================

t0 = time.time()

results = []

f_scores = []
precisions = []
recalls = []

pool_sizes = []

total_predicted = 0
total_true = 0
total_tp = 0

for row in sample_s1.itertuples(index=False):

    entity_id = row.entity_id

    true_set = parse_ids(gt_lookup.get(entity_id))

    predicted_set, pool_size = predict_for_row(row)

    pool_sizes.append(pool_size)

    # F0.5 for this entity
    score = f_beta_score(
        true_set,
        predicted_set,
        beta=0.5
    )

    f_scores.append(score)

    # Precision / recall bookkeeping
    if len(predicted_set) == 0:
        precision = 1.0 if len(true_set) == 0 else 0.0
    else:
        precision = len(true_set & predicted_set) / len(predicted_set)

    if len(true_set) == 0:
        recall = 1.0 if len(predicted_set) == 0 else 0.0
    else:
        recall = len(true_set & predicted_set) / len(true_set)

    precisions.append(precision)
    recalls.append(recall)

    total_predicted += len(predicted_set)
    total_true += len(true_set)
    total_tp += len(true_set & predicted_set)

    results.append(
        (
            entity_id,
            true_set,
            predicted_set,
            pool_size
        )
    )

elapsed = time.time() - t0

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

pool_arr = np.array(pool_sizes)

print("=" * 70)
print("EXACT NAME OR ADDRESS + RAPIDFUZZ VALIDATION")
print("=" * 70)

print(f"Entities evaluated: {len(results)}")

print()
print("--- Candidate Pool ---")
print(f"Average pool size: {pool_arr.mean():.2f}")
print(f"Median pool size: {np.median(pool_arr):.2f}")
print(f"P95 pool size: {np.percentile(pool_arr, 95):.2f}")
print(f"Maximum pool size: {pool_arr.max()}")

print()
print("--- Macro Metrics ---")
print(f"F0.5:     {np.mean(f_scores):.4f}")
print(f"Precision: {np.mean(precisions):.4f}")
print(f"Recall:    {np.mean(recalls):.4f}")

print()
print("--- Micro-style totals (diagnostic only) ---")
print(f"Total predicted matches: {total_predicted}")
print(f"Total true matches:      {total_true}")
print(f"Total true positives:    {total_tp}")

print()
print("--- Runtime ---")
print(f"Runtime: {elapsed:.2f}s")
print(f"Per entity: {elapsed / len(results) * 1000:.2f} ms")

print()
print("DONE")

EXACT NAME OR ADDRESS + RAPIDFUZZ VALIDATION
Entities evaluated: 3000

--- Candidate Pool ---
Average pool size: 40.15
Median pool size: 3.00
P95 pool size: 163.10
Maximum pool size: 1374

--- Macro Metrics ---
F0.5:     0.5769
Precision: 0.6989
Recall:    0.4031

--- Micro-style totals (diagnostic only) ---
Total predicted matches: 4519
Total true matches:      10444
Total true positives:    4008

--- Runtime ---
Runtime: 20.79s
Per entity: 6.93 ms

DONE


In [36]:
# ============================================================
# REBUILD EXACT-NAME TEST MATCHES
# ============================================================

start = time.time()

exact_matches = []
exact_match_counts = []

for _, row in test_source1_keyed_v3.iterrows():
    block_key = row["block_key"]
    norm_name = row["norm_name"]

    s2_matches = s2_exact_index.get((block_key, norm_name), [])
    s3_matches = s3_exact_index.get((block_key, norm_name), [])

    matches = s2_matches + s3_matches

    exact_matches.append(matches)
    exact_match_counts.append(len(matches))

exact_match_counts = np.array(exact_match_counts)

elapsed = time.time() - start

print("=" * 70)
print("EXACT-NAME TEST MATCHES REBUILT")
print("=" * 70)
print("Entities:", len(exact_matches))
print("Entities with >=1 match:", np.sum(exact_match_counts > 0))
print("Coverage:", np.mean(exact_match_counts > 0))
print("Maximum matches:", exact_match_counts.max())
print("Runtime:", round(elapsed, 2), "seconds")

NameError: name 's2_exact_index' is not defined

In [ ]:
# ============================================================
# SUBMISSION 1 — EXACT NORMALIZED NAME BASELINE
# ============================================================

from pathlib import Path
import pandas as pd
import time

OUTPUT_DIR = Path(PROJECT_DIR) / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

# ------------------------------------------------------------
# Build matching_results.tsv
# ------------------------------------------------------------

matching_results = pd.DataFrame({
    "source1_entity_id": test_source1_keyed_v3["entity_id"].values,
    "matched_entity_ids": [
        ",".join(dict.fromkeys(map(str, matches)))
        for matches in exact_matches
    ]
})

matching_path = OUTPUT_DIR / "matching_results.tsv"

matching_results.to_csv(
    matching_path,
    sep="\t",
    index=False
)

# ------------------------------------------------------------
# Candidate pairs
#
# For this baseline, the exact-name candidates ARE the final
# candidate set because there is no later filtering stage.
# ------------------------------------------------------------

candidate_pairs = pd.DataFrame({
    "source1_entity_id": test_source1_keyed_v3["entity_id"].values,
    "candidate_entity_ids": [
        ",".join(dict.fromkeys(map(str, matches)))
        for matches in exact_matches
    ]
})

candidate_path = OUTPUT_DIR / "candidate_pairs.tsv"

candidate_pairs.to_csv(
    candidate_path,
    sep="\t",
    index=False
)

elapsed = time.time() - t0

print("=" * 70)
print("SUBMISSION 1 FILES CREATED")
print("=" * 70)

print("matching_results:", matching_path)
print("candidate_pairs: ", candidate_path)

print()
print("Rows:", len(matching_results))
print(
    "Entities with >=1 predicted match:",
    (matching_results["matched_entity_ids"] != "").sum()
)

print("Runtime:", round(elapsed, 2), "seconds")

NameError: name 'exact_matches' is not defined

In [ ]:
# ============================================================
# VALIDATE SUBMISSION 1
# ============================================================

import subprocess
import sys

validator = Path(PROJECT_DIR) / "data" / "student_resource" / "utils" / "validate_submission.py"
test_dir = Path(PROJECT_DIR) / "data" / "student_resource" / "dataset" / "test"

cmd = [
    sys.executable,
    str(validator),
    "--matching", str(matching_path),
    "--candidate", str(candidate_path),
    "--test-dir", str(test_dir)
]

print("Running submission validator...\n")

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Exit code:", result.returncode)

In [1]:
print("kernal is alive")

kernal is alive


In [2]:
import os
import time
import numpy as np
import pandas as pd
import unicodedata

from pathlib import Path

BASE = Path(r"E:\Amazon-ML-Challenge-2026")
TEST_DIR = BASE / "data" / "student_resource" / "dataset" / "test"
OUTPUT_DIR = BASE / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Ready")
print(TEST_DIR)

Ready
E:\Amazon-ML-Challenge-2026\data\student_resource\dataset\test


In [3]:
def normalize_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower()

    name = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in name
    )

    stopwords = {
        'inc','incorporated','ltd','limited','pvt','private',
        'corp','corporation','llc','llp','co','company',
        'dr','mr','mrs','ms','smt','sri','shri','prof','m','s',
        'the'
    }

    tokens = [t for t in name.split() if t not in stopwords]

    return ' '.join(tokens).strip()


def first_token(normalized_name):
    tokens = normalized_name.split()
    return tokens[0] if tokens else ""


def build_block_key(df):
    df = df.copy()
    df["norm_name"] = df["business_name"].apply(normalize_name)
    df["block_key"] = (
        df["country"].astype(str)
        + "||"
        + df["norm_name"].apply(first_token)
    )
    return df


print("Normalization ready")

Normalization ready


In [4]:
t0 = time.time()

test_source1 = pd.read_csv(
    TEST_DIR / "test_source1.tsv",
    sep="\t"
)

test_source2 = pd.read_csv(
    TEST_DIR / "test_source2.tsv",
    sep="\t"
)

test_source3 = pd.read_csv(
    TEST_DIR / "test_source3.tsv",
    sep="\t"
)

print("S1:", test_source1.shape)
print("S2:", test_source2.shape)
print("S3:", test_source3.shape)
print("Load time:", round(time.time() - t0, 1), "sec")

S1: (1732544, 4)
S2: (4887273, 4)
S3: (5082316, 4)
Load time: 80.3 sec


In [5]:
t0 = time.time()

test_source1_keyed_v3 = build_block_key(test_source1)
test_source2_keyed_v3 = build_block_key(test_source2)
test_source3_keyed_v3 = build_block_key(test_source3)

print("V3 keys built")
print("Time:", round(time.time() - t0, 1), "sec")

V3 keys built
Time: 96.7 sec


In [6]:
def build_fast_index(df, key_cols):
    combined_key = df[key_cols[0]].astype(str)

    for col in key_cols[1:]:
        combined_key = combined_key + "||" + df[col].astype(str)

    combined_key = combined_key.to_numpy()
    entity_ids = df["entity_id"].to_numpy()

    codes, uniques = pd.factorize(combined_key, sort=True)

    sort_idx = np.argsort(codes, kind="stable")

    sorted_codes = codes[sort_idx]
    sorted_entity_ids = entity_ids[sort_idx]

    counts = np.bincount(
        sorted_codes,
        minlength=len(uniques)
    )

    indptr = np.zeros(len(uniques) + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])

    return uniques, indptr, sorted_entity_ids


t0 = time.time()

s2_name_uniques, s2_name_indptr, s2_name_ids = build_fast_index(
    test_source2_keyed_v3,
    ["block_key", "norm_name"]
)

print("S2 index built:", round(time.time() - t0, 1), "sec")

t0 = time.time()

s3_name_uniques, s3_name_indptr, s3_name_ids = build_fast_index(
    test_source3_keyed_v3,
    ["block_key", "norm_name"]
)

print("S3 index built:", round(time.time() - t0, 1), "sec")

print("S2 unique keys:", len(s2_name_uniques))
print("S3 unique keys:", len(s3_name_uniques))

S2 index built: 21.0 sec
S3 index built: 21.5 sec
S2 unique keys: 3460415
S3 unique keys: 3627062


In [7]:
def lookup_fast_index(key, uniques, indptr, entity_ids):
    pos = np.searchsorted(uniques, key)

    if pos < len(uniques) and uniques[pos] == key:
        return entity_ids[indptr[pos]:indptr[pos + 1]]

    return np.empty(0, dtype=entity_ids.dtype)

In [8]:
t0 = time.time()

matching_path = OUTPUT_DIR / "matching_results.tsv"
candidate_path = OUTPUT_DIR / "candidate_pairs.tsv"

s1_ids = test_source1_keyed_v3["entity_id"].to_numpy()
s1_blocks = test_source1_keyed_v3["block_key"].to_numpy()
s1_names = test_source1_keyed_v3["norm_name"].to_numpy()

matched_count = 0
total_predicted = 0

MATCH_HEADER = "source1_entity_id\tmatched_entity_ids\n"
CAND_HEADER = "source1_entity_id\tcandidate_entity_ids\n"

with open(matching_path, "w", encoding="utf-8", buffering=1024*1024) as fm, \
     open(candidate_path, "w", encoding="utf-8", buffering=1024*1024) as fc:

    fm.write(MATCH_HEADER)
    fc.write(CAND_HEADER)

    buffer_matching = []
    buffer_candidates = []

    for entity_id, block, norm_name in zip(
        s1_ids,
        s1_blocks,
        s1_names
    ):
        key = str(block) + "||" + str(norm_name)

        s2 = lookup_fast_index(
            key,
            s2_name_uniques,
            s2_name_indptr,
            s2_name_ids
        )

        s3 = lookup_fast_index(
            key,
            s3_name_uniques,
            s3_name_indptr,
            s3_name_ids
        )

        if len(s2) or len(s3):
            matched_count += 1

        total_predicted += len(s2) + len(s3)

        ids = np.concatenate((s2, s3))

        id_string = ",".join(str(x) for x in ids)

        line = f"{entity_id}\t{id_string}\n"

        buffer_matching.append(line)
        buffer_candidates.append(line)

        if len(buffer_matching) >= 10000:
            fm.write("".join(buffer_matching))
            fc.write("".join(buffer_candidates))
            buffer_matching.clear()
            buffer_candidates.clear()

    if buffer_matching:
        fm.write("".join(buffer_matching))
        fc.write("".join(buffer_candidates))

print("DONE")
print("Entities:", len(s1_ids))
print("Entities with matches:", matched_count)
print("Total predicted IDs:", total_predicted)
print("Runtime:", round(time.time() - t0, 1), "sec")
print("Matching file:", matching_path)
print("Candidate file:", candidate_path)

DONE
Entities: 1732544
Entities with matches: 1560169
Total predicted IDs: 42651215
Runtime: 54.6 sec
Matching file: E:\Amazon-ML-Challenge-2026\output\matching_results.tsv
Candidate file: E:\Amazon-ML-Challenge-2026\output\candidate_pairs.tsv


In [9]:
import time
import numpy as np

t0 = time.time()

matching_path_small = OUTPUT_DIR / "matching_results.tsv"
candidate_path_small = OUTPUT_DIR / "candidate_pairs.tsv"

MAX_IDS = 10

matched_count = 0
total_predicted = 0

s1_ids = test_source1_keyed_v3["entity_id"].to_numpy()
s1_blocks = test_source1_keyed_v3["block_key"].to_numpy()
s1_names = test_source1_keyed_v3["norm_name"].to_numpy()

with open(
    matching_path_small,
    "w",
    encoding="utf-8",
    buffering=1024 * 1024
) as fm, open(
    candidate_path_small,
    "w",
    encoding="utf-8",
    buffering=1024 * 1024
) as fc:

    fm.write("source1_entity_id\tmatched_entity_ids\n")
    fc.write("source1_entity_id\tcandidate_entity_ids\n")

    buffer_m = []
    buffer_c = []

    for entity_id, block, norm_name in zip(
        s1_ids, s1_blocks, s1_names
    ):
        key = str(block) + "||" + str(norm_name)

        s2 = lookup_fast_index(
            key,
            s2_name_uniques,
            s2_name_indptr,
            s2_name_ids
        )

        s3 = lookup_fast_index(
            key,
            s3_name_uniques,
            s3_name_indptr,
            s3_name_ids
        )

        ids = np.concatenate((s2, s3))

        # Precision-oriented size cap
        if len(ids) > MAX_IDS:
            ids = ids[:MAX_IDS]

        if len(ids):
            matched_count += 1

        total_predicted += len(ids)

        id_string = ",".join(str(x) for x in ids)

        line = f"{entity_id}\t{id_string}\n"

        buffer_m.append(line)
        buffer_c.append(line)

        if len(buffer_m) >= 10000:
            fm.write("".join(buffer_m))
            fc.write("".join(buffer_c))
            buffer_m.clear()
            buffer_c.clear()

    if buffer_m:
        fm.write("".join(buffer_m))
        fc.write("".join(buffer_c))

print("DONE")
print("Entities:", len(s1_ids))
print("Entities with predictions:", matched_count)
print("Total predicted IDs:", total_predicted)
print("Runtime:", round(time.time() - t0, 1), "sec")
print("Matching:", matching_path_small)
print("Candidate:", candidate_path_small)

DONE
Entities: 1732544
Entities with predictions: 1560169
Total predicted IDs: 8050866
Runtime: 226.8 sec
Matching: E:\Amazon-ML-Challenge-2026\output\matching_results.tsv
Candidate: E:\Amazon-ML-Challenge-2026\output\candidate_pairs.tsv


## Submission 1 — Exact Normalized-Name Baseline

### Method
V3 blocking was used to identify records sharing the same country and first normalized-name token. Within this candidate space, exact normalized-name matches were generated.

Because the unrestricted exact-name predictions produced an excessively large submission file, a maximum of 10 predicted IDs per Source 1 entity was applied for the submitted version.

### Submission Result
- Leaderboard score: **0.416**
- Rank: **6276**
- Validator: **PASS**

### Observation
Exact normalized-name matching alone produced substantial ambiguity because many Source 2/3 records can share the same normalized business name. Limiting predictions reduced the output size but did not provide sufficient matching quality.

### Decision
**Reject as the final approach.**

Next experiment: selective multi-attribute blocking with frequency pruning, followed by pairwise scoring if candidate recall remains high.

## Submission 2 — Selective Multi-Attribute Blocking

**Objective:** Replace the broad V3 first-token blocking with multiple selective blocking keys.

The blocker uses name and address signals while pruning high-frequency keys to avoid large candidate blocks.

**Plan:**
1. Generate multiple blocking keys for Source 1, Source 2, and Source 3.
2. Remove overly frequent keys.
3. Retrieve candidates through all surviving keys.
4. Union and deduplicate candidates.
5. Measure candidate recall and candidate-set size on validation data.
6. Proceed to pairwise scoring only if the candidate-recall/size trade-off is promising.

In [11]:
# Submission 2 — load training data

TRAIN_DIR = BASE / "data" / "student_resource" / "dataset" / "train"

train_source1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t"
)

train_source2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t"
)

train_source3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t"
)

train_ground_truth = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t"
)

print("S1:", train_source1.shape)
print("S2:", train_source2.shape)
print("S3:", train_source3.shape)
print("GT:", train_ground_truth.shape)

S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)
GT: (2206821, 2)


In [12]:
# Submission 2 — normalized fields for selective blocking

def compact_name(name):
    if pd.isna(name):
        return ""
    
    tokens = normalize_name(name).split()
    return "".join(tokens)


def first_two_tokens(name):
    if pd.isna(name):
        return ""
    
    tokens = normalize_name(name).split()
    return " ".join(tokens[:2])


def name_tokens(name):
    if pd.isna(name):
        return []
    
    return normalize_name(name).split()


def address_number(address):
    if pd.isna(address):
        return ""
    
    import re
    
    match = re.search(r'\b\d+[A-Za-z]?\b', str(address))
    return match.group(0) if match else ""


def address_tokens(address):
    if pd.isna(address):
        return []
    
    address = str(address).lower()
    
    address = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in address
    )
    
    return address.split()


print("Blocking helper functions ready")

Blocking helper functions ready


In [13]:
import time

t0 = time.time()

def prepare_block_fields(df):
    out = pd.DataFrame(index=df.index)
    
    out["entity_id"] = df["entity_id"].values
    out["country"] = df["country"].fillna("").astype(str).values
    
    out["norm_name"] = df["business_name"].apply(normalize_name)
    out["compact_name"] = out["norm_name"].str.replace(" ", "", regex=False)
    out["first_two"] = out["norm_name"].str.split().str[:2].str.join(" ")
    
    out["address"] = df["business_address"].fillna("").astype(str).values
    
    out["address_number"] = (
        out["address"]
        .str.extract(r"\b(\d+[A-Za-z]?)\b", expand=False)
        .fillna("")
    )
    
    return out


train_s1_b = prepare_block_fields(train_source1)
train_s2_b = prepare_block_fields(train_source2)
train_s3_b = prepare_block_fields(train_source3)

print("Prepared")
print("S1:", train_s1_b.shape)
print("S2:", train_s2_b.shape)
print("S3:", train_s3_b.shape)
print("Runtime:", round(time.time() - t0, 1), "sec")

Prepared
S1: (2206821, 7)
S2: (5034616, 7)
S3: (5285603, 7)
Runtime: 201.4 sec


In [14]:
# ---- Minimal missing pieces (not yet in this session) ----
def parse_ids(cell):
    if pd.isna(cell) or cell == '':
        return set()
    return set(str(cell).split(','))

def f_beta_score(true_set, pred_set, beta=0.5):
    if not true_set:
        return 1.0 if not pred_set else 0.0
    if not pred_set:
        return 0.0
    tp = len(true_set & pred_set)
    precision, recall = tp / len(pred_set), tp / len(true_set)
    if precision == 0 and recall == 0:
        return 0.0
    b2 = beta ** 2
    return (1 + b2) * precision * recall / (b2 * precision + recall)

gt_lookup = train_ground_truth.set_index('source1_entity_id')['matched_entity_ids'].to_dict()

# ---- Build the 4 selective keys on S2/S3 (reusing existing build_fast_index) ----
KEY_DEFS = {
    'K1_compact': ['country', 'compact_name'],
    'K2_first_two': ['country', 'first_two'],
    'K3_addr_num': ['country', 'address_number'],
    'K4_name_addr': ['country', 'norm_name', 'address_number'],
}

indexes = {}
t0 = time.time()
for name, cols in KEY_DEFS.items():
    u2, p2, id2 = build_fast_index(train_s2_b, cols)
    u3, p3, id3 = build_fast_index(train_s3_b, cols)
    indexes[name] = {'s2': (u2, p2, id2), 's3': (u3, p3, id3), 'cols': cols}
    bucket_sizes_2 = np.diff(p2)
    bucket_sizes_3 = np.diff(p3)
    print(f"{name}: S2 keys={len(u2)} (max bucket={bucket_sizes_2.max()}, "
          f"p99={np.percentile(bucket_sizes_2,99):.0f}) | "
          f"S3 keys={len(u3)} (max bucket={bucket_sizes_3.max()}, "
          f"p99={np.percentile(bucket_sizes_3,99):.0f})")
print(f"All indexes built in {time.time()-t0:.1f}s")

MAX_BUCKET = 500  # frequency-pruning cap; revisit after seeing the printed distribution above

def pruned_lookup(key, uniques, indptr, ids, cap):
    pos = np.searchsorted(uniques, key)
    if pos < len(uniques) and uniques[pos] == key:
        size = indptr[pos+1] - indptr[pos]
        if size <= cap:
            return ids[indptr[pos]:indptr[pos+1]]
    return np.empty(0, dtype=ids.dtype)

# ---- Validation: union across all 4 keys, measure recall BEFORE any scoring ----
sample_s1 = train_s1_b.sample(n=3000, random_state=42)

pool_sizes, recall_hits = [], []
skipped = 0
t0 = time.time()

for row in sample_s1.itertuples(index=False):
    true_set = parse_ids(gt_lookup.get(row.entity_id))
    if not true_set:
        skipped += 1
        continue

    union_pool = set()
    for name, idx in indexes.items():
        cols = idx['cols']
        key = '||'.join(str(getattr(row, c)) for c in cols)
        u2, p2, id2 = idx['s2']; u3, p3, id3 = idx['s3']
        union_pool.update(pruned_lookup(key, u2, p2, id2, MAX_BUCKET).tolist())
        union_pool.update(pruned_lookup(key, u3, p3, id3, MAX_BUCKET).tolist())

    pool_sizes.append(len(union_pool))
    true_arr = np.array(list(true_set), dtype=object)
    pool_arr = np.array(list(union_pool), dtype=object) if union_pool else np.array([], dtype=object)
    recall_hits.append(np.isin(true_arr, pool_arr).mean())

elapsed = time.time() - t0
sizes = np.array(pool_sizes)

print("\n===== MULTI-KEY SELECTIVE BLOCKING (PRUNED) =====")
print("Entities evaluated:", len(recall_hits), "| skipped (no GT):", skipped)
print("Candidate-pool recall:", round(np.mean(recall_hits), 4))
print("Avg pool:", round(sizes.mean(),2), "| Median:", np.median(sizes),
      "| P95:", np.percentile(sizes,95), "| Max:", sizes.max())
print(f"Runtime: {elapsed:.2f}s | Full-test est: {elapsed/len(recall_hits)*1732544/3600:.2f}h")

K1_compact: S2 keys=3349727 (max bucket=719, p99=8) | S3 keys=3575798 (max bucket=657, p99=7)
K2_first_two: S2 keys=2051593 (max bucket=9375, p99=24) | S3 keys=2192871 (max bucket=9541, p99=22)
K3_addr_num: S2 keys=107315 (max bucket=321088, p99=640) | S3 keys=110930 (max bucket=300817, p99=649)
K4_name_addr: S2 keys=4455314 (max bucket=56, p99=3) | S3 keys=4730110 (max bucket=57, p99=3)
All indexes built in 152.6s

===== MULTI-KEY SELECTIVE BLOCKING (PRUNED) =====
Entities evaluated: 2820 | skipped (no GT): 180
Candidate-pool recall: 0.6979
Avg pool: 191.05 | Median: 73.0 | P95: 853.0499999999997 | Max: 1805
Runtime: 5.61s | Full-test est: 0.96h


In [15]:
# Submission 2 — per-key diagnostic
# No new indexes are built.

def evaluate_key_recall(key_name, cap=500):
    idx = indexes[key_name]
    hits = []
    pools = []

    u2, p2, id2 = idx["s2"]
    u3, p3, id3 = idx["s3"]

    for row in sample_s1.itertuples(index=False):
        true_set = parse_ids(gt_lookup.get(row.entity_id))

        if not true_set:
            continue

        cols = idx["cols"]
        key = "||".join(str(getattr(row, c)) for c in cols)

        cand = set(
            pruned_lookup(key, u2, p2, id2, cap).tolist()
        )
        cand.update(
            pruned_lookup(key, u3, p3, id3, cap).tolist()
        )

        pools.append(len(cand))

        if cand:
            hits.append(
                len(true_set.intersection(cand)) / len(true_set)
            )
        else:
            hits.append(0.0)

    return np.mean(hits), np.mean(pools)


print("===== PER-KEY RECALL @ CAP=500 =====")

for key_name in indexes:
    recall, avg_pool = evaluate_key_recall(key_name, 500)
    print(
        f"{key_name:15s} | "
        f"recall={recall:.4f} | "
        f"avg_pool={avg_pool:.2f}"
    )

print("\n===== UNION AT DIFFERENT CAPS =====")

for cap in [500, 1000, 2000]:
    pool_sizes = []
    recall_hits = []

    for row in sample_s1.itertuples(index=False):
        true_set = parse_ids(gt_lookup.get(row.entity_id))

        if not true_set:
            continue

        union_pool = set()

        for name, idx in indexes.items():
            cols = idx["cols"]
            key = "||".join(str(getattr(row, c)) for c in cols)

            u2, p2, id2 = idx["s2"]
            u3, p3, id3 = idx["s3"]

            union_pool.update(
                pruned_lookup(key, u2, p2, id2, cap).tolist()
            )
            union_pool.update(
                pruned_lookup(key, u3, p3, id3, cap).tolist()
            )

        pool_sizes.append(len(union_pool))
        recall_hits.append(
            len(true_set.intersection(union_pool)) / len(true_set)
        )

    print(
        f"CAP={cap:4d} | "
        f"recall={np.mean(recall_hits):.4f} | "
        f"avg_pool={np.mean(pool_sizes):.2f} | "
        f"median={np.median(pool_sizes):.0f} | "
        f"P95={np.percentile(pool_sizes,95):.0f} | "
        f"max={np.max(pool_sizes):.0f}"
    )

===== PER-KEY RECALL @ CAP=500 =====
K1_compact      | recall=0.4485 | avg_pool=39.74
K2_first_two    | recall=0.5732 | avg_pool=54.64
K3_addr_num     | recall=0.2488 | avg_pool=125.24
K4_name_addr    | recall=0.3061 | avg_pool=1.42

===== UNION AT DIFFERENT CAPS =====
CAP= 500 | recall=0.6979 | avg_pool=191.05 | median=73 | P95=853 | max=1805
CAP=1000 | recall=0.7319 | avg_pool=378.10 | median=103 | P95=1598 | max=3078
CAP=2000 | recall=0.7641 | avg_pool=721.80 | median=192 | P95=3052 | max=6357


In [18]:
import time
import numpy as np
import pandas as pd

CHUNK_ROWS = 1_000_000
MAX_TOKEN_ESTIMATE_GUARD = 100_000_000  # abort if wildly exceeded

def estimate_token_count(df):
    total = 0
    for start in range(0, len(df), CHUNK_ROWS):
        chunk = df['norm_name'].iloc[start:start+CHUNK_ROWS]
        total += int(chunk.str.split().str.len().fillna(0).sum())
    return total

t0 = time.time()
est_s2 = estimate_token_count(train_s2_b)
est_s3 = estimate_token_count(train_s3_b)
print(f"Estimated tokens — S2: {est_s2:,} | S3: {est_s3:,} | total: {est_s2+est_s3:,}")
print(f"Estimated int32-code memory: {(est_s2+est_s3)*4/1e6:.1f} MB")
print(f"Estimation time: {time.time()-t0:.1f}s")

if (est_s2 + est_s3) > MAX_TOKEN_ESTIMATE_GUARD:
    raise RuntimeError("Token count far exceeds expectations — stopping before build. Redesign needed.")

# ---- Build tiny country vocabulary ONCE from the small, row-level country columns ----
# (row-level country columns are only ~5-10M short strings total, not per-token — cheap)
country_codes_s2_full, country_uniques = pd.factorize(train_s2_b['country'].to_numpy())
country_codes_s3_full = pd.Categorical(
    train_s3_b['country'].to_numpy(), categories=country_uniques
).codes.astype(np.int64)

if np.any(country_codes_s2_full < 0) or np.any(country_codes_s3_full < 0):
    raise RuntimeError("Found a country in S2/S3 that is not in the shared country vocabulary.")

print("Country coding verified — no unknown countries.")
# country_codes_s2_full / country_codes_s3_full are now int-coded, ROW-aligned with train_s2_b/train_s3_b
country_to_code = {c: i for i, c in enumerate(country_uniques)}
NUM_COUNTRIES = len(country_uniques)
print(f"Country vocabulary size: {NUM_COUNTRIES}")


def build_token_postings_chunked(df, row_country_codes):
    """
    Memory-conscious token postings index.

    Empty names are removed BEFORE explode(), so the token rows
    and repeated country codes remain perfectly aligned.
    """
    global_vocab = {}

    token_id_parts = []
    entity_id_parts = []
    country_code_parts = []

    entity_ids = df["entity_id"].to_numpy()
    norm_names = df["norm_name"].to_numpy()

    for start in range(0, len(df), CHUNK_ROWS):

        end = min(start + CHUNK_ROWS, len(df))

        chunk_names = pd.Series(
            norm_names[start:end]
        ).fillna("").str.split()

        lengths = chunk_names.str.len().to_numpy(dtype=np.int32)

        # Remove rows with zero name tokens BEFORE explode.
        nonempty = lengths > 0

        if not np.any(nonempty):
            continue

        token_lists = chunk_names.iloc[np.flatnonzero(nonempty)]
        chunk_entity_ids = entity_ids[start:end][nonempty]
        chunk_country_codes = row_country_codes[start:end][nonempty]
        chunk_lengths = lengths[nonempty]

        exploded = pd.DataFrame({
            "entity_id": chunk_entity_ids,
            "tokens": token_lists.to_numpy()
        }).explode(
            "tokens",
            ignore_index=True
        )

        tokens = exploded["tokens"].to_numpy()

        # These now have exactly the same length.
        exploded_country_codes = np.repeat(
            chunk_country_codes,
            chunk_lengths
        )

        # Local token -> global token vocabulary
        local_codes, local_uniques = pd.factorize(tokens)

        remap = np.empty(
            len(local_uniques),
            dtype=np.int32
        )

        for i, tok in enumerate(local_uniques):
            gid = global_vocab.get(tok)

            if gid is None:
                gid = len(global_vocab)
                global_vocab[tok] = gid

            remap[i] = gid

        token_id_parts.append(
            remap[local_codes].astype(np.int32)
        )

        entity_id_parts.append(
            exploded["entity_id"].to_numpy()
        )

        country_code_parts.append(
            exploded_country_codes.astype(np.int32)
        )

    token_ids = np.concatenate(token_id_parts)
    rep_entity_ids = np.concatenate(entity_id_parts)
    rep_country_codes = np.concatenate(country_code_parts)

    inv_vocab = np.empty(
        len(global_vocab),
        dtype=object
    )

    for tok, gid in global_vocab.items():
        inv_vocab[gid] = tok

    return (
        token_ids,
        rep_entity_ids,
        rep_country_codes,
        inv_vocab
    )
    


def build_index_from_codes(combined_codes, entity_ids):
    codes, uniques = pd.factorize(combined_codes, sort=True)
    sort_idx = np.argsort(codes, kind='stable')
    sorted_codes = codes[sort_idx]
    sorted_entity_ids = entity_ids[sort_idx]
    counts = np.bincount(sorted_codes, minlength=len(uniques))
    indptr = np.zeros(len(uniques) + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    return uniques, indptr, sorted_entity_ids


t0 = time.time()

token_ids_s2, ent_s2, country_codes_s2, vocab_s2 = build_token_postings_chunked(
    train_s2_b, country_codes_s2_full
)
token_ids_s3, ent_s3, country_codes_s3, vocab_s3 = build_token_postings_chunked(
    train_s3_b, country_codes_s3_full
)

VOCAB_SIZE_S2 = len(vocab_s2) + 1
VOCAB_SIZE_S3 = len(vocab_s3) + 1

# int64 arithmetic combination — no strings at any point in this step
combined_s2 = country_codes_s2.astype(np.int64) * VOCAB_SIZE_S2 + token_ids_s2
combined_s3 = country_codes_s3.astype(np.int64) * VOCAB_SIZE_S3 + token_ids_s3

s2_tok_uniques, s2_tok_indptr, s2_tok_ids = build_index_from_codes(combined_s2, ent_s2)
s3_tok_uniques, s3_tok_indptr, s3_tok_ids = build_index_from_codes(combined_s3, ent_s3)

print(f"Token index build time: {time.time()-t0:.1f}s")
print(f"S2 unique tokens: {len(vocab_s2):,} | S3 unique tokens: {len(vocab_s3):,}")

bucket_sizes_s2 = np.diff(s2_tok_indptr)
bucket_sizes_s3 = np.diff(s3_tok_indptr)
all_bucket_sizes = np.concatenate([bucket_sizes_s2, bucket_sizes_s3])

percentiles = np.percentile(all_bucket_sizes, [50, 75, 90, 95, 99])
print("\n===== TOKEN DOCUMENT-FREQUENCY DISTRIBUTION =====")
print(f"P50: {percentiles[0]:.0f} | P75: {percentiles[1]:.0f} | "
      f"P90: {percentiles[2]:.0f} | P95: {percentiles[3]:.0f} | P99: {percentiles[4]:.0f}")
print(f"Max bucket size: {all_bucket_sizes.max()}")

Estimated tokens — S2: 14,733,627 | S3: 15,469,560 | total: 30,203,187
Estimated int32-code memory: 120.8 MB
Estimation time: 36.3s
Country coding verified — no unknown countries.
Country vocabulary size: 2
Token index build time: 93.9s
S2 unique tokens: 811,017 | S3 unique tokens: 863,517

===== TOKEN DOCUMENT-FREQUENCY DISTRIBUTION =====
P50: 1 | P75: 1 | P90: 6 | P95: 19 | P99: 116
Max bucket size: 190091


In [19]:
def token_country_lookup(country_code, token_id, uniques, indptr, ids, vocab_size, cap):
    if token_id is None:
        return np.empty(0, dtype=ids.dtype)
    key = country_code * vocab_size + token_id
    pos = np.searchsorted(uniques, key)
    if pos < len(uniques) and uniques[pos] == key:
        size = indptr[pos+1] - indptr[pos]
        if size <= cap:
            return ids[indptr[pos]:indptr[pos+1]]
    return np.empty(0, dtype=ids.dtype)

country_to_code = {c: i for i, c in enumerate(country_uniques)}
vocab_to_id_s2 = {tok: i for i, tok in enumerate(vocab_s2)}
vocab_to_id_s3 = {tok: i for i, tok in enumerate(vocab_s3)}
CAPS_TO_TEST = [6, 20, 100]

for CAP in CAPS_TO_TEST:
    # --- Rare-token pool alone ---
    pool_sizes, recall_hits = [], []
    skipped = 0
    t0 = time.time()

    for row in sample_s1.itertuples(index=False):
        true_set = parse_ids(gt_lookup.get(row.entity_id))
        if not true_set:
            skipped += 1
            continue

        ccode = country_to_code.get(row.country)
        union_pool = set()
        if ccode is not None:
            for tok in row.norm_name.split():
                tid2 = vocab_to_id_s2.get(tok)
                tid3 = vocab_to_id_s3.get(tok)
                union_pool.update(token_country_lookup(ccode, tid2, s2_tok_uniques, s2_tok_indptr, s2_tok_ids, VOCAB_SIZE_S2, CAP).tolist())
                union_pool.update(token_country_lookup(ccode, tid3, s3_tok_uniques, s3_tok_indptr, s3_tok_ids, VOCAB_SIZE_S3, CAP).tolist())

        pool_sizes.append(len(union_pool))
        true_arr = np.array(list(true_set), dtype=object)
        pool_arr = np.array(list(union_pool), dtype=object) if union_pool else np.array([], dtype=object)
        recall_hits.append(np.isin(true_arr, pool_arr).mean())

    elapsed = time.time() - t0
    sizes = np.array(pool_sizes)
    print(f"\n===== RARE-TOKEN ALONE (cap={CAP}) =====")
    print("Recall:", round(np.mean(recall_hits), 4))
    print("Avg:", round(sizes.mean(),2), "Median:", np.median(sizes), "P95:", np.percentile(sizes,95), "Max:", sizes.max())
    print(f"Runtime: {elapsed:.2f}s | Full-test est: {elapsed/len(recall_hits)*1732544/3600:.2f}h")

    # --- Union with K2 (first_two) + K4 (name+address_number) ---
    pool_sizes_u, recall_hits_u = [], []
    t0 = time.time()

    for row in sample_s1.itertuples(index=False):
        true_set = parse_ids(gt_lookup.get(row.entity_id))
        if not true_set:
            continue

        union_pool = set()
        ccode = country_to_code.get(row.country)
        if ccode is not None:
            for tok in row.norm_name.split():
                tid2 = vocab_to_id_s2.get(tok)
                tid3 = vocab_to_id_s3.get(tok)
                union_pool.update(token_country_lookup(ccode, tid2, s2_tok_uniques, s2_tok_indptr, s2_tok_ids, VOCAB_SIZE_S2, CAP).tolist())
                union_pool.update(token_country_lookup(ccode, tid3, s3_tok_uniques, s3_tok_indptr, s3_tok_ids, VOCAB_SIZE_S3, CAP).tolist())

        for kname in ['K2_first_two', 'K4_name_addr']:
            cols = indexes[kname]['cols']
            key = '||'.join(str(getattr(row, c)) for c in cols)
            u2, p2, id2 = indexes[kname]['s2']; u3, p3, id3 = indexes[kname]['s3']
            union_pool.update(pruned_lookup(key, u2, p2, id2, MAX_BUCKET).tolist())
            union_pool.update(pruned_lookup(key, u3, p3, id3, MAX_BUCKET).tolist())

        pool_sizes_u.append(len(union_pool))
        true_arr = np.array(list(true_set), dtype=object)
        pool_arr = np.array(list(union_pool), dtype=object) if union_pool else np.array([], dtype=object)
        recall_hits_u.append(np.isin(true_arr, pool_arr).mean())

    elapsed_u = time.time() - t0
    sizes_u = np.array(pool_sizes_u)
    print(f"\n===== RARE-TOKEN (cap={CAP}) + K2 + K4 UNION =====")
    print("Recall:", round(np.mean(recall_hits_u), 4))
    print("Avg:", round(sizes_u.mean(),2), "Median:", np.median(sizes_u), "P95:", np.percentile(sizes_u,95), "Max:", sizes_u.max())
    print(f"Runtime: {elapsed_u:.2f}s | Full-test est: {elapsed_u/len(recall_hits_u)*1732544/3600:.2f}h")
    print(f"vs. current best (cap 2000 union): 0.7641 recall @ 722 avg")
    print(f"vs. V3 baseline: 0.7893 recall @ ~5830 avg")


===== RARE-TOKEN ALONE (cap=6) =====
Recall: 0.0359
Avg: 0.33 Median: 0.0 P95: 3.0 Max: 12
Runtime: 0.52s | Full-test est: 0.09h

===== RARE-TOKEN (cap=6) + K2 + K4 UNION =====
Recall: 0.5963
Avg: 54.9 Median: 10.0 P95: 210.04999999999973 Max: 971
Runtime: 1.83s | Full-test est: 0.31h
vs. current best (cap 2000 union): 0.7641 recall @ 722 avg
vs. V3 baseline: 0.7893 recall @ ~5830 avg

===== RARE-TOKEN ALONE (cap=20) =====
Recall: 0.1095
Avg: 2.4 Median: 0.0 P95: 20.0 Max: 47
Runtime: 0.32s | Full-test est: 0.05h

===== RARE-TOKEN (cap=20) + K2 + K4 UNION =====
Recall: 0.6137
Avg: 56.51 Median: 14.0 P95: 210.04999999999973 Max: 971
Runtime: 1.09s | Full-test est: 0.19h
vs. current best (cap 2000 union): 0.7641 recall @ 722 avg
vs. V3 baseline: 0.7893 recall @ ~5830 avg

===== RARE-TOKEN ALONE (cap=100) =====
Recall: 0.298
Avg: 32.84 Median: 0.0 P95: 163.0 Max: 356
Runtime: 0.47s | Full-test est: 0.08h

===== RARE-TOKEN (cap=100) + K2 + K4 UNION =====
Recall: 0.6642
Avg: 85.39 Median: 

In [20]:
import time
import numpy as np

# ---- Step 1: build the one missing piece — train-side V3 index (country || first_token) ----
def first_token_of(s):
    if not isinstance(s, str) or not s:
        return ""
    parts = s.split()
    return parts[0] if parts else ""

train_s2_b['first_token'] = train_s2_b['norm_name'].apply(first_token_of)
train_s3_b['first_token'] = train_s3_b['norm_name'].apply(first_token_of)
sample_s1 = sample_s1.copy()
sample_s1['first_token'] = sample_s1['norm_name'].apply(first_token_of)

t0 = time.time()
v3_s2_u, v3_s2_p, v3_s2_ids = build_fast_index(train_s2_b, ['country', 'first_token'])
v3_s3_u, v3_s3_p, v3_s3_ids = build_fast_index(train_s3_b, ['country', 'first_token'])
print(f"V3 train index built in {time.time()-t0:.1f}s | S2 keys={len(v3_s2_u):,} S3 keys={len(v3_s3_u):,}")

# ---- Step 2: per-true-match classification ----
CAP_RARE = 100

def get_pool(cols, row, cap=None):
    key = '||'.join(str(getattr(row, c)) for c in cols)
    idx = indexes[cols_to_key[tuple(cols)]] if tuple(cols) in cols_to_key else None
    return key  # placeholder, replaced below with direct index refs

# direct references (avoids re-deriving key names)
K2_cols = indexes['K2_first_two']['cols']
K4_cols = indexes['K4_name_addr']['cols']
k2_u2, k2_p2, k2_id2 = indexes['K2_first_two']['s2']
k2_u3, k2_p3, k2_id3 = indexes['K2_first_two']['s3']
k4_u2, k4_p2, k4_id2 = indexes['K4_name_addr']['s2']
k4_u3, k4_p3, k4_id3 = indexes['K4_name_addr']['s3']

total_true = 0
count_k2 = count_k4 = count_rare_union = count_v3 = count_none = 0
count_v3_only_missed_by_rare = 0   # V3 caught it, rare-union did not
count_rare_only_missed_by_v3 = 0   # rare-union caught it, V3 did not

missed_all_examples = []  # (s1 row, true_id) pairs where none of the 4 branches retrieved it

t0 = time.time()

for row in sample_s1.itertuples(index=False):
    true_set = parse_ids(gt_lookup.get(row.entity_id))
    if not true_set:
        continue

    # K2 pool (plain, no cap — matches original standalone K2 measurement)
    key_k2 = '||'.join(str(getattr(row, c)) for c in K2_cols)
    pool_k2 = set(lookup_fast_index(key_k2, k2_u2, k2_p2, k2_id2).tolist()) | \
              set(lookup_fast_index(key_k2, k2_u3, k2_p3, k2_id3).tolist())

    # K4 pool (plain, no cap)
    key_k4 = '||'.join(str(getattr(row, c)) for c in K4_cols)
    pool_k4 = set(lookup_fast_index(key_k4, k4_u2, k4_p2, k4_id2).tolist()) | \
              set(lookup_fast_index(key_k4, k4_u3, k4_p3, k4_id3).tolist())

    # Rare-token (cap 100) + K2(cap MAX_BUCKET) + K4(cap MAX_BUCKET) — same union as the approved experiment
    ccode = country_to_code.get(row.country)
    pool_rare = set()
    if ccode is not None:
        for tok in row.norm_name.split():
            tid2 = vocab_to_id_s2.get(tok)
            tid3 = vocab_to_id_s3.get(tok)
            pool_rare.update(token_country_lookup(ccode, tid2, s2_tok_uniques, s2_tok_indptr, s2_tok_ids, VOCAB_SIZE_S2, CAP_RARE).tolist())
            pool_rare.update(token_country_lookup(ccode, tid3, s3_tok_uniques, s3_tok_indptr, s3_tok_ids, VOCAB_SIZE_S3, CAP_RARE).tolist())
    pool_rare.update(set(pruned_lookup(key_k2, k2_u2, k2_p2, k2_id2, MAX_BUCKET).tolist()))
    pool_rare.update(set(pruned_lookup(key_k2, k2_u3, k2_p3, k2_id3, MAX_BUCKET).tolist()))
    pool_rare.update(set(pruned_lookup(key_k4, k4_u2, k4_p2, k4_id2, MAX_BUCKET).tolist()))
    pool_rare.update(set(pruned_lookup(key_k4, k4_u3, k4_p3, k4_id3, MAX_BUCKET).tolist()))

    # V3 pool (plain, no cap)
    key_v3 = f"{row.country}||{row.first_token}"
    pool_v3 = set(lookup_fast_index(key_v3, v3_s2_u, v3_s2_p, v3_s2_ids).tolist()) | \
              set(lookup_fast_index(key_v3, v3_s3_u, v3_s3_p, v3_s3_ids).tolist())

    for true_id in true_set:
        total_true += 1
        in_k2 = true_id in pool_k2
        in_k4 = true_id in pool_k4
        in_rare = true_id in pool_rare
        in_v3 = true_id in pool_v3

        count_k2 += in_k2
        count_k4 += in_k4
        count_rare_union += in_rare
        count_v3 += in_v3

        if in_v3 and not in_rare:
            count_v3_only_missed_by_rare += 1
        if in_rare and not in_v3:
            count_rare_only_missed_by_v3 += 1

        if not in_rare and not in_v3:
            count_none += 1
            if len(missed_all_examples) < 15:
                missed_all_examples.append((row, true_id))

elapsed = time.time() - t0

print("\n===== TRUE-MATCH RETRIEVAL COVERAGE (per true match, not per entity) =====")
print(f"Total true matches evaluated: {total_true}")
print(f"Retrieved by K2:                     {count_k2} ({count_k2/total_true*100:.1f}%)")
print(f"Retrieved by K4:                     {count_k4} ({count_k4/total_true*100:.1f}%)")
print(f"Retrieved by rare-token+K2+K4(cap100): {count_rare_union} ({count_rare_union/total_true*100:.1f}%)")
print(f"Retrieved by V3:                      {count_v3} ({count_v3/total_true*100:.1f}%)")
print(f"Missed by ALL FOUR:                   {count_none} ({count_none/total_true*100:.1f}%)")
print()
print(f"V3 caught it, rare-union did NOT:     {count_v3_only_missed_by_rare} ({count_v3_only_missed_by_rare/total_true*100:.1f}%)")
print(f"Rare-union caught it, V3 did NOT:     {count_rare_only_missed_by_v3} ({count_rare_only_missed_by_v3/total_true*100:.1f}%)")
print(f"\nRuntime: {elapsed:.2f}s")

# ---- Step 3: inspect a handful of true matches missed by everything ----
print("\n===== SAMPLE OF TRUE MATCHES MISSED BY ALL FOUR BRANCHES =====")
lookup_s2 = train_s2_b.set_index('entity_id')
lookup_s3 = train_s3_b.set_index('entity_id')

for row, true_id in missed_all_examples[:10]:
    print(f"\nS1 [{row.entity_id}] name='{row.norm_name}' addr='{row.address}' country='{row.country}'")
    if true_id.startswith('S2-') and true_id in lookup_s2.index:
        m = lookup_s2.loc[true_id]
    elif true_id in lookup_s3.index:
        m = lookup_s3.loc[true_id]
    else:
        print(f"  -> True match [{true_id}] not found in _b frames (unexpected)")
        continue
    print(f"  True match [{true_id}] name='{m['norm_name']}' addr='{m['address']}' country='{m['country']}'")

V3 train index built in 18.2s | S2 keys=372,673 S3 keys=420,923

===== TRUE-MATCH RETRIEVAL COVERAGE (per true match, not per entity) =====
Total true matches evaluated: 10444
Retrieved by K2:                     6366 (61.0%)
Retrieved by K4:                     3204 (30.7%)
Retrieved by rare-token+K2+K4(cap100): 6929 (66.3%)
Retrieved by V3:                      8249 (79.0%)
Missed by ALL FOUR:                   2031 (19.4%)

V3 caught it, rare-union did NOT:     1484 (14.2%)
Rare-union caught it, V3 did NOT:     164 (1.6%)

Runtime: 6.40s

===== SAMPLE OF TRUE MATCHES MISSED BY ALL FOUR BRANCHES =====

S1 [S1-53356671] name='pediatric medicine pllc' addr='4850 20, Otisco, NY' country='US'
  True match [S2-904412198] name='pllc pediatric medirine' addr='4850-C 20, OTISCO, NY' country='US'

S1 [S1-320151505] name='fetech national twin' addr='19034 Woodburn Road, Woodburn, IN' country='US'
  True match [S2-230628758] name='fetechnationaltwin com' addr='WOODBURN ROAD, MILAN, IN' country=

In [21]:
import unicodedata

def has_non_latin_script(text):
    for ch in text:
        if ch.isalpha() and unicodedata.name(ch, '').find('LATIN') == -1 and ch.isascii() is False:
            return True
    return False

def sorted_token_key(tokens):
    return ' '.join(sorted(tokens))

missed_all_detail = []
v3_only_detail = []

for row in sample_s1.itertuples(index=False):
    true_set = parse_ids(gt_lookup.get(row.entity_id))
    if not true_set:
        continue

    ccode = country_to_code.get(row.country)
    pool_rare = set()
    if ccode is not None:
        for tok in row.norm_name.split():
            tid2 = vocab_to_id_s2.get(tok); tid3 = vocab_to_id_s3.get(tok)
            pool_rare.update(token_country_lookup(ccode, tid2, s2_tok_uniques, s2_tok_indptr, s2_tok_ids, VOCAB_SIZE_S2, CAP_RARE).tolist())
            pool_rare.update(token_country_lookup(ccode, tid3, s3_tok_uniques, s3_tok_indptr, s3_tok_ids, VOCAB_SIZE_S3, CAP_RARE).tolist())
    key_k2 = '||'.join(str(getattr(row, c)) for c in K2_cols)
    key_k4 = '||'.join(str(getattr(row, c)) for c in K4_cols)
    pool_rare.update(pruned_lookup(key_k2, k2_u2, k2_p2, k2_id2, MAX_BUCKET).tolist())
    pool_rare.update(pruned_lookup(key_k2, k2_u3, k2_p3, k2_id3, MAX_BUCKET).tolist())
    pool_rare.update(pruned_lookup(key_k4, k4_u2, k4_p2, k4_id2, MAX_BUCKET).tolist())
    pool_rare.update(pruned_lookup(key_k4, k4_u3, k4_p3, k4_id3, MAX_BUCKET).tolist())

    key_v3 = f"{row.country}||{row.first_token}"
    pool_v3 = set(lookup_fast_index(key_v3, v3_s2_u, v3_s2_p, v3_s2_ids).tolist()) | \
              set(lookup_fast_index(key_v3, v3_s3_u, v3_s3_p, v3_s3_ids).tolist())

    for true_id in true_set:
        in_rare, in_v3 = true_id in pool_rare, true_id in pool_v3
        if not in_rare and not in_v3 and len(missed_all_detail) < 60:
            missed_all_detail.append((row, true_id))
        if in_v3 and not in_rare and len(v3_only_detail) < 60:
            v3_only_detail.append((row, true_id))

def classify_and_print(pairs, label):
    print(f"\n===== {label} (showing up to {len(pairs)}) =====")
    counts = {'non_latin': 0, 'reorder_candidate': 0, 'containment_candidate': 0, 'other': 0}
    for row, true_id in pairs[:20]:
        m = lookup_s2.loc[true_id] if true_id in lookup_s2.index else (lookup_s3.loc[true_id] if true_id in lookup_s3.index else None)
        if m is None: continue
        q_tokens, c_tokens = set(row.norm_name.split()), set(m['norm_name'].split())
        non_latin = has_non_latin_script(m['norm_name']) or has_non_latin_script(row.norm_name)
        reorder = sorted_token_key(q_tokens) == sorted_token_key(c_tokens) and row.norm_name != m['norm_name']
        containment = (q_tokens <= c_tokens or c_tokens <= q_tokens) and q_tokens != c_tokens
        cls = 'non_latin' if non_latin else ('reorder_candidate' if reorder else ('containment_candidate' if containment else 'other'))
        counts[cls] += 1
        print(f"  [{cls}] S1='{row.norm_name}' | match='{m['norm_name']}'")
    print("Class counts (sampled subset):", counts)

classify_and_print(missed_all_detail, "MISSED BY ALL (sample)")
classify_and_print(v3_only_detail, "V3-CATCHES, RARE-UNION-MISSES (sample)")


===== MISSED BY ALL (sample) (showing up to 60) =====
  [other] S1='pediatric medicine pllc' | match='pllc pediatric medirine'
  [other] S1='fetech national twin' | match='fetechnationaltwin com'
  [other] S1='general design innovations' | match='gildcalosyn'
  [containment_candidate] S1='general design innovations' | match='umbraveo formerly known as general design innovations'
  [non_latin] S1='creative projects' | match='क्रिएटिव प्रोजेक्ट्स लिमिटेड'
  [non_latin] S1='creative projects' | match='क्रिएटिव प्रोजेक्ट्स लिमिटेड'
  [other] S1='goar stone' | match='goarstone com'
  [non_latin] S1='lakshmi consultants' | match='लक्ष्मी कंसल्टेंट्स प्राइवेट लिमिटेड'
  [non_latin] S1='lakshmi consultants' | match='लक्ष्मी कंसल्टेंट्स प्राइवेट लिमिटेड'
  [non_latin] S1='lakshmi consultants' | match='लक्ष्मी कंसल्टेंट्स प्राइवेट लिमिटेड'
  [non_latin] S1='lakshmi consultants' | match='लक्ष्मी कंसल्टेंट्स प्राइवेट लिमिटेड'
  [other] S1='ss technology' | match='téchnologyprivatess com'
  [non_l

In [22]:
import time
import numpy as np

Ns = [6, 8, 10, 12]

# ---- Add prefix columns (train_s2_b/train_s3_b + sample_s1) ----
for N in Ns:
    col = f'compact_prefix_{N}'
    train_s2_b[col] = train_s2_b['compact_name'].str.slice(0, N)
    train_s3_b[col] = train_s3_b['compact_name'].str.slice(0, N)
    sample_s1[col] = sample_s1['compact_name'].str.slice(0, N)

prefix_indexes = {}

for N in Ns:
    col = f'compact_prefix_{N}'
    t0 = time.time()
    u2, p2, id2 = build_fast_index(train_s2_b, ['country', col])
    u3, p3, id3 = build_fast_index(train_s3_b, ['country', col])
    build_time = time.time() - t0

    bsz2, bsz3 = np.diff(p2), np.diff(p3)
    all_bsz = np.concatenate([bsz2, bsz3])

    prefix_indexes[N] = {'s2': (u2, p2, id2), 's3': (u3, p3, id3)}

    print(f"\n===== compact_name[:{N}] — TRAIN BUCKET STATS =====")
    print(f"Build time: {build_time:.1f}s")
    print(f"Unique keys: S2={len(u2):,} S3={len(u3):,}")
    print(f"P50={np.percentile(all_bsz,50):.0f} P90={np.percentile(all_bsz,90):.0f} "
          f"P95={np.percentile(all_bsz,95):.0f} P99={np.percentile(all_bsz,99):.0f} "
          f"Max={all_bsz.max()}")

# ---- Validation: recall/pool size for the prefix key ALONE, per N ----
for N in Ns:
    col = f'compact_prefix_{N}'
    u2, p2, id2 = prefix_indexes[N]['s2']
    u3, p3, id3 = prefix_indexes[N]['s3']

    pool_sizes, recall_hits = [], []
    skipped = 0
    t0 = time.time()

    for row in sample_s1.itertuples(index=False):
        true_set = parse_ids(gt_lookup.get(row.entity_id))
        if not true_set:
            skipped += 1
            continue

        key = f"{row.country}||{getattr(row, col)}"
        pool = set(lookup_fast_index(key, u2, p2, id2).tolist()) | \
               set(lookup_fast_index(key, u3, p3, id3).tolist())

        pool_sizes.append(len(pool))
        true_arr = np.array(list(true_set), dtype=object)
        pool_arr = np.array(list(pool), dtype=object) if pool else np.array([], dtype=object)
        recall_hits.append(np.isin(true_arr, pool_arr).mean())

    elapsed = time.time() - t0
    sizes = np.array(pool_sizes)

    print(f"\n===== compact_name[:{N}] ALONE — VALIDATION =====")
    print("Recall:", round(np.mean(recall_hits), 4))
    print("Avg:", round(sizes.mean(), 2), "Median:", np.median(sizes),
          "P95:", np.percentile(sizes, 95), "Max:", sizes.max())
    print(f"Runtime: {elapsed:.2f}s | Full-test est: {elapsed/len(recall_hits)*1732544/3600:.2f}h")


===== compact_name[:6] — TRAIN BUCKET STATS =====
Build time: 55.6s
Unique keys: S2=603,429 S3=614,342
P50=1 P90=8 P95=18 P99=95 Max=32844

===== compact_name[:8] — TRAIN BUCKET STATS =====
Build time: 21.2s
Unique keys: S2=1,198,202 S3=1,246,328
P50=1 P90=5 P95=10 P99=43 Max=32840

===== compact_name[:10] — TRAIN BUCKET STATS =====
Build time: 26.1s
Unique keys: S2=1,754,682 S3=1,867,473
P50=1 P90=4 P95=7 P99=27 Max=20564

===== compact_name[:12] — TRAIN BUCKET STATS =====
Build time: 29.9s
Unique keys: S2=2,153,885 S3=2,329,522
P50=1 P90=3 P95=5 P99=20 Max=18997

===== compact_name[:6] ALONE — VALIDATION =====
Recall: 0.7819
Avg: 4550.76 Median: 330.5 P95: 20758.0 Max: 65091
Runtime: 91.78s | Full-test est: 15.66h

===== compact_name[:8] ALONE — VALIDATION =====
Recall: 0.7221
Avg: 2508.7 Median: 57.0 P95: 17571.0 Max: 65084
Runtime: 18.10s | Full-test est: 3.09h

===== compact_name[:10] ALONE — VALIDATION =====
Recall: 0.6646
Avg: 1119.51 Median: 16.0 P95: 5445.0 Max: 41006
Runtime

In [23]:
import time
import numpy as np

u2_p10, p2_p10, id2_p10 = prefix_indexes[10]['s2']
u3_p10, p3_p10, id3_p10 = prefix_indexes[10]['s3']

pool_sizes, recall_hits = [], []
skipped = 0
t0 = time.time()

for row in sample_s1.itertuples(index=False):
    true_set = parse_ids(gt_lookup.get(row.entity_id))
    if not true_set:
        skipped += 1
        continue

    union_pool = set()

    # K2 (first_two)
    key_k2 = '||'.join(str(getattr(row, c)) for c in K2_cols)
    union_pool.update(pruned_lookup(key_k2, k2_u2, k2_p2, k2_id2, MAX_BUCKET).tolist())
    union_pool.update(pruned_lookup(key_k2, k2_u3, k2_p3, k2_id3, MAX_BUCKET).tolist())

    # K4 (name+address_number)
    key_k4 = '||'.join(str(getattr(row, c)) for c in K4_cols)
    union_pool.update(pruned_lookup(key_k4, k4_u2, k4_p2, k4_id2, MAX_BUCKET).tolist())
    union_pool.update(pruned_lookup(key_k4, k4_u3, k4_p3, k4_id3, MAX_BUCKET).tolist())

    # rare-token (cap 100)
    ccode = country_to_code.get(row.country)
    if ccode is not None:
        for tok in row.norm_name.split():
            tid2 = vocab_to_id_s2.get(tok)
            tid3 = vocab_to_id_s3.get(tok)
            union_pool.update(token_country_lookup(ccode, tid2, s2_tok_uniques, s2_tok_indptr, s2_tok_ids, VOCAB_SIZE_S2, CAP_RARE).tolist())
            union_pool.update(token_country_lookup(ccode, tid3, s3_tok_uniques, s3_tok_indptr, s3_tok_ids, VOCAB_SIZE_S3, CAP_RARE).tolist())

    # compact_prefix_10 (reused, not rebuilt)
    key_p10 = f"{row.country}||{row.compact_prefix_10}"
    union_pool.update(lookup_fast_index(key_p10, u2_p10, p2_p10, id2_p10).tolist())
    union_pool.update(lookup_fast_index(key_p10, u3_p10, p3_p10, id3_p10).tolist())

    pool_sizes.append(len(union_pool))
    true_arr = np.array(list(true_set), dtype=object)
    pool_arr = np.array(list(union_pool), dtype=object) if union_pool else np.array([], dtype=object)
    recall_hits.append(np.isin(true_arr, pool_arr).mean())

elapsed = time.time() - t0
sizes = np.array(pool_sizes)

print("===== K2 + K4 + rare-token(cap100) + compact_prefix_10 UNION =====")
print("Entities evaluated:", len(recall_hits), "| skipped:", skipped)
print("Recall:", round(np.mean(recall_hits), 4))
print("Avg:", round(sizes.mean(), 2), "Median:", np.median(sizes),
      "P95:", np.percentile(sizes, 95), "Max:", sizes.max())
print(f"Runtime: {elapsed:.2f}s | Full-test est: {elapsed/len(recall_hits)*1732544/3600:.2f}h")
print()
print("vs. rare-token(cap100)+K2+K4:        0.6642 recall @ 85.39 avg")
print("vs. K2+K4 cap2000 (previous best):   0.7641 recall @ 721.80 avg")
print("vs. V3 baseline:                     0.7893 recall @ ~5830 avg")

===== K2 + K4 + rare-token(cap100) + compact_prefix_10 UNION =====
Entities evaluated: 2820 | skipped: 180
Recall: 0.7461
Avg: 1152.97 Median: 96.0 P95: 5445.0 Max: 41064
Runtime: 55.91s | Full-test est: 9.54h

vs. rare-token(cap100)+K2+K4:        0.6642 recall @ 85.39 avg
vs. K2+K4 cap2000 (previous best):   0.7641 recall @ 721.80 avg
vs. V3 baseline:                     0.7893 recall @ ~5830 avg


In [24]:
import time
import numpy as np

CAP_K2K4 = 2000
CAP_RARE = 100

# One-time, cheap: entity_id -> norm_name (for length-diff feature only)
t0 = time.time()
s2_name_by_id = dict(zip(train_s2_b['entity_id'].to_numpy(), train_s2_b['norm_name'].to_numpy()))
s3_name_by_id = dict(zip(train_s3_b['entity_id'].to_numpy(), train_s3_b['norm_name'].to_numpy()))
print(f"Name lookup dicts built in {time.time()-t0:.1f}s")

def name_of(cid):
    return s2_name_by_id.get(cid) if cid.startswith('S2-') else s3_name_by_id.get(cid)

# ---- Main diagnostic loop ----
total_true = 0
subset_stats = {
    'K4': {'vol': 0, 'tp': 0, 'coverage_hits': 0},
    'K2': {'vol': 0, 'tp': 0, 'coverage_hits': 0},
    'rare>=1': {'vol': 0, 'tp': 0, 'coverage_hits': 0},
    'rare>=2': {'vol': 0, 'tp': 0, 'coverage_hits': 0},
    'rare>=3': {'vol': 0, 'tp': 0, 'coverage_hits': 0},
    'K4=0,K2=0,rare>=1': {'vol': 0, 'tp': 0, 'coverage_hits': 0},
    'K4=0,K2=0,rare>=2': {'vol': 0, 'tp': 0, 'coverage_hits': 0},
    'K4 OR K2': {'vol': 0, 'tp': 0, 'coverage_hits': 0},
    'full_union (K2+K4+rare)': {'vol': 0, 'tp': 0, 'coverage_hits': 0},
}
len_diff_true, len_diff_false = [], []

entities_evaluated = 0
t0 = time.time()

for row in sample_s1.itertuples(index=False):
    true_set = parse_ids(gt_lookup.get(row.entity_id))
    if not true_set:
        continue
    entities_evaluated += 1
    total_true += len(true_set)

    cand = {}  # cid -> {'k2':0,'k4':0,'rare':0}

    key_k2 = '||'.join(str(getattr(row, c)) for c in K2_cols)
    for cid in pruned_lookup(key_k2, k2_u2, k2_p2, k2_id2, CAP_K2K4).tolist() + \
               pruned_lookup(key_k2, k2_u3, k2_p3, k2_id3, CAP_K2K4).tolist():
        cand.setdefault(cid, {'k2': 0, 'k4': 0, 'rare': 0})['k2'] = 1

    key_k4 = '||'.join(str(getattr(row, c)) for c in K4_cols)
    for cid in pruned_lookup(key_k4, k4_u2, k4_p2, k4_id2, CAP_K2K4).tolist() + \
               pruned_lookup(key_k4, k4_u3, k4_p3, k4_id3, CAP_K2K4).tolist():
        cand.setdefault(cid, {'k2': 0, 'k4': 0, 'rare': 0})['k4'] = 1

    ccode = country_to_code.get(row.country)
    if ccode is not None:
        for tok in row.norm_name.split():
            tid2 = vocab_to_id_s2.get(tok)
            tid3 = vocab_to_id_s3.get(tok)
            for cid in token_country_lookup(ccode, tid2, s2_tok_uniques, s2_tok_indptr, s2_tok_ids, VOCAB_SIZE_S2, CAP_RARE).tolist():
                cand.setdefault(cid, {'k2': 0, 'k4': 0, 'rare': 0})['rare'] += 1
            for cid in token_country_lookup(ccode, tid3, s3_tok_uniques, s3_tok_indptr, s3_tok_ids, VOCAB_SIZE_S3, CAP_RARE).tolist():
                cand.setdefault(cid, {'k2': 0, 'k4': 0, 'rare': 0})['rare'] += 1

    len1 = len(row.norm_name)

    def mark(name, is_member):
        if is_member:
            subset_stats[name]['vol'] += 1
            if cid in true_set:
                subset_stats[name]['tp'] += 1

    covered = {k: False for k in subset_stats}

    for cid, flags in cand.items():
        is_true = cid in true_set
        cname = name_of(cid)
        if cname is not None:
            (len_diff_true if is_true else len_diff_false).append(abs(len(cname) - len1))

        checks = {
            'K4': flags['k4'] == 1,
            'K2': flags['k2'] == 1,
            'rare>=1': flags['rare'] >= 1,
            'rare>=2': flags['rare'] >= 2,
            'rare>=3': flags['rare'] >= 3,
            'K4=0,K2=0,rare>=1': flags['k4'] == 0 and flags['k2'] == 0 and flags['rare'] >= 1,
            'K4=0,K2=0,rare>=2': flags['k4'] == 0 and flags['k2'] == 0 and flags['rare'] >= 2,
            'K4 OR K2': flags['k4'] == 1 or flags['k2'] == 1,
            'full_union (K2+K4+rare)': True,
        }
        for name, is_member in checks.items():
            if is_member:
                subset_stats[name]['vol'] += 1
                if is_true:
                    subset_stats[name]['tp'] += 1
                    covered[name] = True

    for name, was_covered in covered.items():
        if was_covered:
            subset_stats[name]['coverage_hits'] += 1

elapsed = time.time() - t0

print("\n===== CANDIDATE-VS-TRUE-MATCH DIAGNOSTICS =====")
print(f"Entities evaluated: {entities_evaluated} | Total true matches: {total_true}")
print()
for name, s in subset_stats.items():
    recall_contribution = s['tp'] / total_true if total_true else 0
    precision = s['tp'] / s['vol'] if s['vol'] else 0
    entity_coverage = s['coverage_hits'] / entities_evaluated if entities_evaluated else 0
    avg_vol_per_entity = s['vol'] / entities_evaluated if entities_evaluated else 0
    print(f"--- {name} ---")
    print(f"  Candidate volume (avg/entity): {avg_vol_per_entity:.2f} | total: {s['vol']}")
    print(f"  True matches captured: {s['tp']} | recall contribution: {recall_contribution:.4f}")
    print(f"  Precision (tp/volume): {precision:.4f}")
    print(f"  Entity-level coverage (>=1 true match found): {entity_coverage:.4f}")
    print()

print(f"Runtime: {elapsed:.2f}s | Full-test est: {elapsed/entities_evaluated*1732544/3600:.2f}h")
print()
print("Name-length-diff distribution (diagnostic only, no threshold chosen):")
if len_diff_true:
    print(f"  True matches   — mean: {np.mean(len_diff_true):.2f}, median: {np.median(len_diff_true):.1f}, "
          f"P90: {np.percentile(len_diff_true,90):.1f}")
if len_diff_false:
    print(f"  Non-matches    — mean: {np.mean(len_diff_false):.2f}, median: {np.median(len_diff_false):.1f}, "
          f"P90: {np.percentile(len_diff_false,90):.1f}")

Name lookup dicts built in 50.4s

===== CANDIDATE-VS-TRUE-MATCH DIAGNOSTICS =====
Entities evaluated: 2820 | Total true matches: 10444

--- K4 ---
  Candidate volume (avg/entity): 1.42 | total: 4016
  True matches captured: 3204 | recall contribution: 0.3068
  Precision (tp/volume): 0.7978
  Entity-level coverage (>=1 true match found): 0.6450

--- K2 ---
  Candidate volume (avg/entity): 85.51 | total: 241147
  True matches captured: 6104 | recall contribution: 0.5845
  Precision (tp/volume): 0.0253
  Entity-level coverage (>=1 true match found): 0.8727

--- rare>=1 ---
  Candidate volume (avg/entity): 32.84 | total: 92612
  True matches captured: 3100 | recall contribution: 0.2968
  Precision (tp/volume): 0.0335
  Entity-level coverage (>=1 true match found): 0.3457

--- rare>=2 ---
  Candidate volume (avg/entity): 0.54 | total: 1516
  True matches captured: 380 | recall contribution: 0.0364
  Precision (tp/volume): 0.2507
  Entity-level coverage (>=1 true match found): 0.0543

--- ra

In [25]:
results = []
pool_size_total = 0
entities_evaluated = 0
t0 = time.time()

for row in sample_s1.itertuples(index=False):
    true_set = parse_ids(gt_lookup.get(row.entity_id))
    if not true_set:
        continue
    entities_evaluated += 1

    cand = {}
    key_k2 = '||'.join(str(getattr(row, c)) for c in K2_cols)
    for cid in pruned_lookup(key_k2, k2_u2, k2_p2, k2_id2, 2000).tolist() + \
               pruned_lookup(key_k2, k2_u3, k2_p3, k2_id3, 2000).tolist():
        cand.setdefault(cid, {'k2':0,'k4':0,'rare':0})['k2'] = 1

    key_k4 = '||'.join(str(getattr(row, c)) for c in K4_cols)
    for cid in pruned_lookup(key_k4, k4_u2, k4_p2, k4_id2, 2000).tolist() + \
               pruned_lookup(key_k4, k4_u3, k4_p3, k4_id3, 2000).tolist():
        cand.setdefault(cid, {'k2':0,'k4':0,'rare':0})['k4'] = 1

    ccode = country_to_code.get(row.country)
    if ccode is not None:
        for tok in row.norm_name.split():
            for cid in token_country_lookup(ccode, vocab_to_id_s2.get(tok), s2_tok_uniques, s2_tok_indptr, s2_tok_ids, VOCAB_SIZE_S2, 100).tolist():
                cand.setdefault(cid, {'k2':0,'k4':0,'rare':0})['rare'] += 1
            for cid in token_country_lookup(ccode, vocab_to_id_s3.get(tok), s3_tok_uniques, s3_tok_indptr, s3_tok_ids, VOCAB_SIZE_S3, 100).tolist():
                cand.setdefault(cid, {'k2':0,'k4':0,'rare':0})['rare'] += 1

    pool_size_total += len(cand)
    len1 = len(row.norm_name)
    predicted = set()
    for cid, f in cand.items():
        cname = name_of(cid)
        ldiff = abs(len(cname) - len1) if cname is not None else 99
        if f['k4'] == 1 or (f['k2'] == 1 and ldiff <= 2) or (f['rare'] >= 2 and ldiff <= 2):
            predicted.add(cid)

    results.append((true_set, predicted))

elapsed = time.time() - t0
scores = [f_beta_score(t, p, beta=0.5) for t, p in results]
tp = sum(len(t & p) for t, p in results); pred_total = sum(len(p) for t, p in results); true_total = sum(len(t) for t, p in results)
print("F0.5:", round(np.mean(scores),6), "Precision:", round(tp/pred_total,6) if pred_total else 0,
      "Recall:", round(tp/true_total,6) if true_total else 0,
      "Coverage:", round(np.mean([len(p)>0 for t,p in results]),4),
      "Avg predictions/entity:", round(pred_total/entities_evaluated,3))
print("Avg pre-filter pool:", round(pool_size_total/entities_evaluated,2))
print(f"Runtime: {elapsed:.2f}s | Full-test est: {elapsed/entities_evaluated*1732544/3600:.2f}h")

F0.5: 0.375604 Precision: 0.041295 Recall: 0.483723 Coverage: 0.9493 Avg predictions/entity: 43.383
Avg pre-filter pool: 116.23
Runtime: 1.96s | Full-test est: 0.34h


In [26]:
results_k4 = []
pred_total = 0
true_total = 0
tp_total = 0
entities_evaluated = 0
t0 = time.time()

for row in sample_s1.itertuples(index=False):
    true_set = parse_ids(gt_lookup.get(row.entity_id))
    if not true_set:
        continue

    entities_evaluated += 1

    key_k4 = '||'.join(str(getattr(row, c)) for c in K4_cols)

    predicted = set()

    ids2 = pruned_lookup(
        key_k4, k4_u2, k4_p2, k4_id2, 2000
    )
    ids3 = pruned_lookup(
        key_k4, k4_u3, k4_p3, k4_id3, 2000
    )

    predicted.update(ids2.tolist())
    predicted.update(ids3.tolist())

    results_k4.append((true_set, predicted))

    tp_total += len(true_set & predicted)
    pred_total += len(predicted)
    true_total += len(true_set)

scores_k4 = [
    f_beta_score(t, p, beta=0.5)
    for t, p in results_k4
]

print("===== K4-ONLY VALIDATION =====")
print("Entities:", entities_evaluated)
print("F0.5:", round(np.mean(scores_k4), 6))
print("Precision:", round(tp_total / pred_total, 6) if pred_total else 0)
print("Recall:", round(tp_total / true_total, 6) if true_total else 0)
print(
    "Coverage:",
    round(np.mean([len(p) > 0 for t, p in results_k4]), 4)
)
print(
    "Avg predictions/entity:",
    round(pred_total / entities_evaluated, 3)
)
print(f"Runtime: {time.time() - t0:.2f}s")

===== K4-ONLY VALIDATION =====
Entities: 2820
F0.5: 0.486437
Precision: 0.797809
Recall: 0.306779
Coverage: 0.6741
Avg predictions/entity: 1.424
Runtime: 0.26s


In [28]:
# Build address lookup dictionaries for K4 similarity validation

s2_addr_by_id = dict(
    zip(train_s2_b['entity_id'], train_s2_b['address'])
)

s3_addr_by_id = dict(
    zip(train_s3_b['entity_id'], train_s3_b['address'])
)

print("S2 address lookup:", len(s2_addr_by_id))
print("S3 address lookup:", len(s3_addr_by_id))

S2 address lookup: 5034616
S3 address lookup: 5285603


In [29]:
# ============================================================
# K4-RESTRICTED EXACT NAME + ADDRESS SIMILARITY VALIDATION
# ============================================================

from rapidfuzz import fuzz
import time
import numpy as np

results_k4_sim = []

pred_total = 0
true_total = 0
tp_total = 0
entities_evaluated = 0
pool_total = 0

t0 = time.time()

for row in sample_s1.itertuples(index=False):

    true_set = parse_ids(gt_lookup.get(row.entity_id))

    if not true_set:
        continue

    entities_evaluated += 1

    # --------------------------------------------------------
    # 1. Retrieve ONLY K4 candidates
    # --------------------------------------------------------
    key_k4 = '||'.join(str(getattr(row, c)) for c in K4_cols)

    ids2 = pruned_lookup(
        key_k4,
        k4_u2,
        k4_p2,
        k4_id2,
        2000
    )

    ids3 = pruned_lookup(
        key_k4,
        k4_u3,
        k4_p3,
        k4_id3,
        2000
    )

    candidates = set(ids2.tolist())
    candidates.update(ids3.tolist())

    pool_total += len(candidates)

    # --------------------------------------------------------
    # 2. Score K4 candidates using the previously validated
    #    exact-name + address/name similarity rule
    # --------------------------------------------------------

    name1 = str(row.norm_name)
    addr1 = str(row.address)

    predicted = set()

    for cid in candidates:

        # Candidate names/addresses are stored in the
        # existing dictionaries/data structures.
        name2 = name_of(cid)

        # Determine source from the existing lookup dictionaries.
        if cid in s2_name_by_id:
            addr2 = s2_addr_by_id.get(cid, "")
        else:
            addr2 = s3_addr_by_id.get(cid, "")

        name2 = "" if name2 is None else str(name2)
        addr2 = "" if addr2 is None else str(addr2)

        # Exact normalized name + fuzzy address
        name_exact = (name1 == name2)

        # Exact normalized address + fuzzy name
        addr_exact = (addr1 == addr2)

        if name_exact:
            addr_score = fuzz.token_sort_ratio(addr1, addr2)

            if addr_score >= 70:
                predicted.add(cid)
                continue

        if addr_exact:
            name_score = fuzz.token_sort_ratio(name1, name2)

            if name_score >= 70:
                predicted.add(cid)

    results_k4_sim.append((true_set, predicted))

    tp_total += len(true_set & predicted)
    pred_total += len(predicted)
    true_total += len(true_set)


# ------------------------------------------------------------
# 3. Metrics
# ------------------------------------------------------------

scores_k4_sim = [
    f_beta_score(t, p, beta=0.5)
    for t, p in results_k4_sim
]

elapsed = time.time() - t0

print("===== K4 + EXACT NAME/ADDRESS SIMILARITY VALIDATION =====")
print("Entities:", entities_evaluated)

print(
    "F0.5:",
    round(np.mean(scores_k4_sim), 6)
)

print(
    "Precision:",
    round(tp_total / pred_total, 6)
    if pred_total else 0
)

print(
    "Recall:",
    round(tp_total / true_total, 6)
    if true_total else 0
)

print(
    "Coverage:",
    round(
        np.mean([len(p) > 0 for t, p in results_k4_sim]),
        4
    )
)

print(
    "Avg predictions/entity:",
    round(
        pred_total / entities_evaluated,
        3
    )
)

print(
    "Avg K4 candidate pool/entity:",
    round(
        pool_total / entities_evaluated,
        3
    )
)

print(f"Runtime: {elapsed:.2f}s")

print("\nComparison:")
print("Previous exact-name/address rule F0.5: 0.5769")
print("K4-only F0.5: 0.4864")

===== K4 + EXACT NAME/ADDRESS SIMILARITY VALIDATION =====
Entities: 2820
F0.5: 0.252429
Precision: 0.994148
Recall: 0.130123
Coverage: 0.361
Avg predictions/entity: 0.485
Avg K4 candidate pool/entity: 1.424
Runtime: 1.34s

Comparison:
Previous exact-name/address rule F0.5: 0.5769
K4-only F0.5: 0.4864


In [31]:
[x for x in globals() if x.startswith("test_") or x.startswith("train_")]

['test_source1',
 'test_source2',
 'test_source3',
 'test_source1_keyed_v3',
 'test_source2_keyed_v3',
 'test_source3_keyed_v3',
 'train_source1',
 'train_source2',
 'train_source3',
 'train_ground_truth',
 'train_s1_b',
 'train_s2_b',
 'train_s3_b']

In [33]:
# Prepare only the columns required for K4 on TEST Source 1

import time

t0 = time.time()

test_s1_k4 = test_source1[['entity_id', 'country', 'business_name', 'business_address']].copy()

# K4 uses:
# country + normalized_name + address_number
test_s1_k4['norm_name'] = test_s1_k4['business_name'].apply(normalize_name)
test_s1_k4['address'] = test_s1_k4['business_address'].fillna('').astype(str)
test_s1_k4['address_number'] = test_s1_k4['address'].apply(address_number)

print("Prepared:", test_s1_k4.shape)
print(f"Runtime: {time.time() - t0:.2f}s")



Prepared: (1732544, 7)
Runtime: 20.70s


In [39]:
# ============================================================
# SUBMISSION #2 — K4-ONLY FULL TEST GENERATION
# ============================================================

import os
import csv
import time
import gc

os.makedirs("output", exist_ok=True)

matching_path = "output/matching_results.tsv"
candidate_path = "output/candidate_pairs.tsv"

# Remove previous files if they exist
for path in [matching_path, candidate_path]:
    if os.path.exists(path):
        os.remove(path)

total_entities = len(test_s1_k4)

processed = 0
total_candidates = 0

start_time = time.time()

with open(matching_path, "w", encoding="utf-8", newline="") as match_file, \
     open(candidate_path, "w", encoding="utf-8", newline="") as cand_file:

    match_writer = csv.writer(match_file, delimiter="\t")
    cand_writer = csv.writer(cand_file, delimiter="\t")

    match_writer.writerow([
        "source1_entity_id",
        "matched_entity_ids"
    ])

    cand_writer.writerow([
        "source1_entity_id",
        "candidate_entity_ids"
    ])

    for row in test_s1_k4.itertuples(index=False):

        s1_id = str(row.entity_id)

        # K4 = country + normalized name + address number
        key_k4 = (
            str(row.country)
            + "||"
            + str(row.norm_name)
            + "||"
            + str(row.address_number)
        )

        # Source 2 candidates
        ids2 = pruned_lookup(
            key_k4,
            k4_u2,
            k4_p2,
            k4_id2,
            2000
        )

        # Source 3 candidates
        ids3 = pruned_lookup(
            key_k4,
            k4_u3,
            k4_p3,
            k4_id3,
            2000
        )

        candidates = set(ids2.tolist())
        candidates.update(ids3.tolist())

        candidate_string = ",".join(
            sorted(str(x) for x in candidates)
        )

        # K4-only: every K4 candidate is a prediction
        match_string = candidate_string

        cand_writer.writerow([
            s1_id,
            candidate_string
        ])

        match_writer.writerow([
            s1_id,
            match_string
        ])

        total_candidates += len(candidates)
        processed += 1

        # Progress every 100k
        if processed % 100000 == 0:

            elapsed = time.time() - start_time
            rate = processed / elapsed
            remaining = total_entities - processed
            eta = remaining / rate if rate > 0 else 0

            print(
                f"{processed:,}/{total_entities:,} "
                f"({processed / total_entities:.1%}) | "
                f"{rate:,.0f} entities/sec | "
                f"ETA {eta / 60:.1f} min | "
                f"avg candidates {total_candidates / processed:.3f}"
            )

elapsed = time.time() - start_time

print("\n===== SUBMISSION #2 GENERATED =====")
print(f"Entities: {processed:,}")
print(f"Total candidate pairs: {total_candidates:,}")
print(f"Average candidates/entity: {total_candidates / processed:.3f}")
print(f"Runtime: {elapsed:.2f}s ({elapsed / 60:.2f} min)")
print(f"Matching file: {matching_path}")
print(f"Candidate file: {candidate_path}")

KeyboardInterrupt: 

In [35]:
import os

print("matching_results size:",
      os.path.getsize("output/matching_results.tsv") / (1024**2),
      "MB")

print("candidate_pairs size:",
      os.path.getsize("output/candidate_pairs.tsv") / (1024**2),
      "MB")

matching_results size: 29.622260093688965 MB
candidate_pairs size: 29.622262001037598 MB


## Final Production — Test-Time K4 Index

The validation experiments established K4 as the selected final blocking strategy:

`country + normalized business name + address number`

For final inference, the K4 indexes must be built from the TEST Source 2 and Source 3 records, because predictions must reference TEST S2/S3 entity IDs.

In [36]:
# ============================================================
# FINAL PRODUCTION — BUILD K4 INDEXES FROM TEST S2/S3
# ============================================================

import time
import gc
import numpy as np

# Reuse the already-prepared TEST V3 dataframes.
# They already contain normalized names.
# We only need to add the address number used by K4.

t0 = time.time()

test_source2_keyed_v3["address_number"] = (
    test_source2_keyed_v3["business_address"]
    .fillna("")
    .astype(str)
    .str.extract(r"\b(\d+[A-Za-z]?)\b", expand=False)
    .fillna("")
)

print("Building TEST Source 2 K4 index...")

test_k4_u2, test_k4_p2, test_k4_id2 = build_fast_index(
    test_source2_keyed_v3,
    ["country", "norm_name", "address_number"]
)

print(
    "TEST S2 K4 index:",
    len(test_k4_u2),
    "unique keys |",
    len(test_k4_id2),
    "entity IDs"
)

# Build S3 separately to keep memory usage controlled.
print("Building TEST Source 3 K4 index...")

test_source3_keyed_v3["address_number"] = (
    test_source3_keyed_v3["business_address"]
    .fillna("")
    .astype(str)
    .str.extract(r"\b(\d+[A-Za-z]?)\b", expand=False)
    .fillna("")
)

test_k4_u3, test_k4_p3, test_k4_id3 = build_fast_index(
    test_source3_keyed_v3,
    ["country", "norm_name", "address_number"]
)

print(
    "TEST S3 K4 index:",
    len(test_k4_u3),
    "unique keys |",
    len(test_k4_id3),
    "entity IDs"
)

print(f"Total index-build runtime: {time.time() - t0:.2f}s")

gc.collect()

Building TEST Source 2 K4 index...
TEST S2 K4 index: 4411170 unique keys | 4887273 entity IDs
Building TEST Source 3 K4 index...
TEST S3 K4 index: 4618594 unique keys | 5082316 entity IDs
Total index-build runtime: 633.46s


3692

In [37]:
# Confirm the K4 indexes contain TEST S2/S3 IDs, not training IDs

print("TEST S2 sample IDs:", test_k4_id2[:5])
print("TEST S3 sample IDs:", test_k4_id3[:5])

print("\nTEST S2 sample IDs:")
print(test_source2["entity_id"].head().tolist())

print("\nTEST S3 sample IDs:")
print(test_source3["entity_id"].head().tolist())

TEST S2 sample IDs: ['S2-437288209' 'S2-318506116' 'S2-392170207' 'S2-290552472'
 'S2-856926290']
TEST S3 sample IDs: ['S3-931055936' 'S3-933540912' 'S3-962770032' 'S3-731658603'
 'S3-686238937']

TEST S2 sample IDs:
['S2-192345572', 'S2-566025912', 'S2-158121477', 'S2-89663826', 'S2-884102769']

TEST S3 sample IDs:
['S3-462677478', 'S3-374810425', 'S3-198586129', 'S3-10300249', 'S3-604980231']


In [38]:
# Tiny production sanity check — first 100 test S1 records

for i in range(100):
    row = test_s1_k4.iloc[i]

    key = (
        str(row["country"])
        + "||"
        + str(row["norm_name"])
        + "||"
        + str(row["address_number"])
    )

    cand2 = lookup_fast_index(
        key,
        test_k4_u2,
        test_k4_p2,
        test_k4_id2
    )

    cand3 = lookup_fast_index(
        key,
        test_k4_u3,
        test_k4_p3,
        test_k4_id3
    )

    candidates = np.concatenate([cand2, cand3])

    if len(candidates) > 0:
        print("S1:", row["entity_id"])
        print("Candidates:", candidates[:10])
        print()

S1: S1-714132312
Candidates: ['S3-625880872']

S1: S1-106407869
Candidates: ['S2-705547832' 'S3-585937637']

S1: S1-921369899
Candidates: ['S2-808920974' 'S2-107533354']

S1: S1-909865979
Candidates: ['S2-528901108']

S1: S1-280204013
Candidates: ['S2-407834725' 'S3-257824113' 'S3-539891283' 'S3-789634527' 'S3-41229951'
 'S3-331318086' 'S3-921934808']

S1: S1-479574617
Candidates: ['S2-262582939' 'S2-985539429']

S1: S1-680637447
Candidates: ['S2-876151147' 'S3-261988229']

S1: S1-870190130
Candidates: ['S3-153733853']

S1: S1-365170694
Candidates: ['S2-487943573']

S1: S1-778640743
Candidates: ['S2-87750197' 'S3-698916948']

S1: S1-550154910
Candidates: ['S2-43467331']

S1: S1-704604312
Candidates: ['S2-497722920' 'S3-7278253']

S1: S1-628750886
Candidates: ['S2-696378102' 'S2-406813553' 'S3-734992856']

S1: S1-714038427
Candidates: ['S2-9389075' 'S3-556092751' 'S3-343322433']

S1: S1-426194890
Candidates: ['S2-189578510' 'S2-12575838']

S1: S1-123475953
Candidates: ['S2-596729763']



In [40]:
# ============================================================
# SUBMISSION #3 — K4-ONLY FULL TEST GENERATION
# ============================================================

import os
import csv
import time
import gc

os.makedirs("output", exist_ok=True)

matching_path = "output/matching_results.tsv"
candidate_path = "output/candidate_pairs.tsv"

# Remove previous files if they exist
for path in [matching_path, candidate_path]:
    if os.path.exists(path):
        os.remove(path)

total_entities = len(test_s1_k4)

processed = 0
total_candidates = 0

start_time = time.time()

with open(matching_path, "w", encoding="utf-8", newline="") as match_file, \
     open(candidate_path, "w", encoding="utf-8", newline="") as cand_file:

    match_writer = csv.writer(match_file, delimiter="\t")
    cand_writer = csv.writer(cand_file, delimiter="\t")

    match_writer.writerow([
        "source1_entity_id",
        "matched_entity_ids"
    ])

    cand_writer.writerow([
        "source1_entity_id",
        "candidate_entity_ids"
    ])

    for row in test_s1_k4.itertuples(index=False):

        s1_id = str(row.entity_id)

        # K4 = country + normalized name + address number
        key_k4 = (
            str(row.country)
            + "||"
            + str(row.norm_name)
            + "||"
            + str(row.address_number)
        )

        # Source 2 candidates
        ids2 = pruned_lookup(
             key_k4,
            test_k4_u2,
            test_k4_p2,
            test_k4_id2,
            2000
        )

        # Source 3 candidates
        ids3 = pruned_lookup(
            key_k4,
            test_k4_u3,
            test_k4_p3,
            test_k4_id3,
            2000
        )

        candidates = set(ids2.tolist())
        candidates.update(ids3.tolist())

        candidate_string = ",".join(
            sorted(str(x) for x in candidates)
        )

        # K4-only: every K4 candidate is a prediction
        match_string = candidate_string

        cand_writer.writerow([
            s1_id,
            candidate_string
        ])

        match_writer.writerow([
            s1_id,
            match_string
        ])

        total_candidates += len(candidates)
        processed += 1

        # Progress every 100k
        if processed % 100000 == 0:

            elapsed = time.time() - start_time
            rate = processed / elapsed
            remaining = total_entities - processed
            eta = remaining / rate if rate > 0 else 0

            print(
                f"{processed:,}/{total_entities:,} "
                f"({processed / total_entities:.1%}) | "
                f"{rate:,.0f} entities/sec | "
                f"ETA {eta / 60:.1f} min | "
                f"avg candidates {total_candidates / processed:.3f}"
            )

elapsed = time.time() - start_time

print("\n===== SUBMISSION #3 GENERATED =====")
print(f"Entities: {processed:,}")
print(f"Total candidate pairs: {total_candidates:,}")
print(f"Average candidates/entity: {total_candidates / processed:.3f}")
print(f"Runtime: {elapsed:.2f}s ({elapsed / 60:.2f} min)")
print(f"Matching file: {matching_path}")
print(f"Candidate file: {candidate_path}")

100,000/1,732,544 (5.8%) | 8,173 entities/sec | ETA 3.3 min | avg candidates 1.324
200,000/1,732,544 (11.5%) | 7,677 entities/sec | ETA 3.3 min | avg candidates 1.324
300,000/1,732,544 (17.3%) | 9,009 entities/sec | ETA 2.7 min | avg candidates 1.326
400,000/1,732,544 (23.1%) | 10,250 entities/sec | ETA 2.2 min | avg candidates 1.327
500,000/1,732,544 (28.9%) | 11,220 entities/sec | ETA 1.8 min | avg candidates 1.325
600,000/1,732,544 (34.6%) | 11,769 entities/sec | ETA 1.6 min | avg candidates 1.326
700,000/1,732,544 (40.4%) | 12,215 entities/sec | ETA 1.4 min | avg candidates 1.327
800,000/1,732,544 (46.2%) | 12,759 entities/sec | ETA 1.2 min | avg candidates 1.328
900,000/1,732,544 (51.9%) | 13,268 entities/sec | ETA 1.0 min | avg candidates 1.328
1,000,000/1,732,544 (57.7%) | 13,616 entities/sec | ETA 0.9 min | avg candidates 1.329
1,100,000/1,732,544 (63.5%) | 13,951 entities/sec | ETA 0.8 min | avg candidates 1.328
1,200,000/1,732,544 (69.3%) | 13,944 entities/sec | ETA 0.6 min |